# BEVFormer 代码实现全流程：从 nuScenes 到训练、BEV 和 3D 检测

本 notebook 是当前仓库 `端到端/BEVFormer` 的实现学习主线。它浓缩并串联以下文档：

- `nuscence_to_pkl.md`：原始 nuScenes 如何成为 PKL；
- `dataset.md`：Dataset、pipeline、DataContainer、sampler 和 batch；
- `config_and_registry.md`：Python config、plugin 和 Registry；
- `framework_training_pipeline.md`：MMCV/MMDetection3D 的装配、Runner、Hook；
- `model_architecture.md`：模型数据流、TSA、SCA、检测头；
- `training_overview.md`：匹配、loss、优化与验证。

目标不是背文件名，而是能从任意一个张量或配置字段追踪到：**它从哪里来、谁消费它、输出什么、如何参与训练。**


## 学习顺序和总图

```text
nuScenes JSON / 图像
    ↓  数据转换
PKL infos：标定、ego pose、GT、CAN bus、相机路径
    ↓  Dataset + pipeline
单样本：T 帧 × 6 相机 + 当前帧 GT + img_metas
    ↓  sampler / collate / DataContainer
训练 batch
    ↓  Config + Registry 构建对象，Runner 调度 train_step
BEVFormer.forward_train
    ↓
历史帧递推 prev_bev ；当前帧 CNN/FPN → TSA/SCA encoder → BEV
    ↓
Deformable-DETR 风格 3D 检测 decoder
    ↓
Hungarian matching + Focal/L1 loss → backward → optimizer
```

建议先完成第 1～6 节，再进入模型细节；否则容易在 SCA/TSA 张量变换中丢失整体上下文。


---

## 第一部分：数据是什么，怎样成为 batch

### 1. nuScenes 的四套坐标与当前帧 LiDAR 统一参考

nuScenes 原始信息涉及：global、ego、sensor/LiDAR、camera、image pixel。当前项目把**每个关键帧当前的 `LIDAR_TOP`**作为检测与 BEV 的统一三维参考：

```text
global  -- ego pose --> 当前 ego -- calibrated_sensor --> 当前 LiDAR_TOP
                                              ↑
历史 LiDAR / 6 cameras / GT 都最终与它对齐或建立投影关系
```

物理 LiDAR 坐标是右手系：`x` 前、`y` 左、`z` 上。相机像素不是 LiDAR 平面：`u` 向右、`v` 向下；SCA 通过 `lidar2img` 把 LiDAR 3D 点变成 `(u,v)`。


In [ ]:
# 最小齐次投影示例：真实工程中 lidar2img 已把外参和内参合并
import numpy as np

p_lidar = np.array([10.0, 2.0, 1.0, 1.0])  # 前方 10m、左方 2m、高 1m
lidar2img = np.array([[800., 0., 800., 0.],
                      [0., 800., 450., 0.],
                      [0.,   0.,   1., 0.],
                      [0.,   0.,   0., 1.]])
p_img_h = lidar2img @ p_lidar
u, v = p_img_h[0] / p_img_h[2], p_img_h[1] / p_img_h[2]
print('homogeneous:', p_img_h, 'pixel (u,v):', (u, v))


### 2. nuScenes 到 PKL：不是把图片打包，而是预计算“关联关系”

PKL 每条 info 的关键价值是让运行时不必反复查 nuScenes JSON：

| 内容 | 典型字段 | 后续消费者 |
|-|-|-|
| 当前帧路径 | `img_filename`、`lidar_path` | pipeline 读图；本模型不读 LiDAR 点云 |
| 6 相机标定 | `lidar2img`、`lidar2cam` | SCA 的 3D→像素投影 |
| 全局位姿 | `ego2global_translation/rotation` | 改写 CAN bus，时序对齐 |
| 3D 标注 | `gt_boxes`、`gt_names`、`gt_velocity` | 当前帧 matching 与 loss |
| scene / prev | `scene_token`、`prev` | 训练队列与推理缓存重置 |

一个非常重要的界限：PKL 有 LiDAR 路径和 LiDAR 坐标，不表示 BEVFormer 使用 LiDAR 点作为输入；这里使用它作为 3D 坐标和标注定义。


### 3. Dataset：单样本为何是一个时序队列

`CustomNuScenesDataset` 不是只返回时刻 t 的一帧。默认 `queue_length=4`，一个训练样本逻辑上是：

```text
[t-3, t-2, t-1, t]
  历史 3 帧          当前监督帧
```

每帧都有 6 张相机图。`union2one()` 会把入选帧的 CAN bus 改写为相对前一入选帧的位移与 yaw 差；当前帧 GT 保留作监督。模型随后把历史三帧无梯度递推成一张 `prev_bev`。

典型模型输入形状：

```text
img: [B,T,N,3,H_img,W_img] = [B,4,6,3,H_img,W_img]
img_metas: list[B]，每个元素保存 T 个 frame meta
gt_bboxes_3d / gt_labels_3d: 只对应 t
```


In [ ]:
# 最小形状演示：模型怎样分离历史帧和当前帧
import torch
B, T, N, H, W = 1, 4, 6, 928, 1600
img = torch.empty(B, T, N, 3, H, W)
prev_img = img[:, :-1]
cur_img = img[:, -1]
print('history:', tuple(prev_img.shape))  # [B,3,6,3,H,W]
print('current:', tuple(cur_img.shape))   # [B,6,3,H,W]


### 4. pipeline、DataContainer、Sampler、collate：为什么 batch 不只是 `torch.stack`

pipeline 读取并增强每帧 6 图，同时更新 `lidar2img`，保证“增强后的像素”仍与投影矩阵一致。随后 `Collect3D` 把不同类型字段放入 `DataContainer`：

| 字段 | 为什么不能简单堆叠 | DataContainer 意图 |
|-|-|-|
| 图像 | H/W 经 padding 后相同 | `stack=True`，形成 Tensor |
| 3D GT | 每样本框数不同 | 保持 list，不强行 padding |
| `img_metas` | Python dict，含 numpy/list | `cpu_only=True`，避免错误搬到 GPU |

训练 sampler 常用 `DistributedGroupSampler`：按宽高组分桶、每卡取相近尺寸样本，减少 padding；`BatchSampler` 决定“哪些 index 合成一个 batch”；`collate_fn` 再把这些样本递归合并为模型可消费的 DataContainer/Tensor。


In [ ]:
# 最小 collate 思想：图像可 stack，变长 3D box 应保留 list
samples = [
    {'img': torch.zeros(6, 3, 4, 4), 'boxes': torch.zeros(5, 9)},
    {'img': torch.ones (6, 3, 4, 4), 'boxes': torch.zeros(2, 9)},
]
batch_img = torch.stack([s['img'] for s in samples])
batch_boxes = [s['boxes'] for s in samples]
print(batch_img.shape, [x.shape for x in batch_boxes])


---

## 第二部分：配置、注册与 MMCV 训练框架

### 5. Config：Python 文件先执行、合并，随后才构建实例

启动命令把配置路径传给 `tools/train.py`：

```text
tools/dist_train.sh projects/configs/bevformer/bevformer_base.py 8
    → args.config
    → Config.fromfile(args.config)
    → 执行 Python 配置，递归合并 _base_
    → cfg（ConfigDict 风格的声明数据）
```

此时 `type='BEVFormer'` 仍只是字符串；配置不会自动创建任何网络。真正实例化发生在 `build_model(cfg.model)`、`build_dataset(cfg.data.train)` 等 build API 中。


In [ ]:
# 最小版 build_from_cfg：展示“配置字符串 → 类 → 实例”的核心思想
REGISTRY = {}
def register(cls):
    REGISTRY[cls.__name__] = cls  # 装饰器在类定义/模块 import 时执行
    return cls

@register
class ToyBackbone:
    def __init__(self, channels=256): self.channels = channels

cfg = dict(type='ToyBackbone', channels=128)
cls = REGISTRY[cfg['type']]
obj = cls(**{k:v for k,v in cfg.items() if k != 'type'})
print(type(obj).__name__, obj.channels)


### 6. Registry 与 plugin：为什么自定义类会被框架找到

不同组件注册到不同 Registry，例如：

```text
@DETECTORS.register_module()              → BEVFormer
@HEADS.register_module()                  → BEVFormerHead
@TRANSFORMER.register_module()            → PerceptionTransformer
@ATTENTION.register_module()              → TSA / SCA
@TRANSFORMER_LAYER_SEQUENCE.register...() → BEVFormerEncoder
@RUNNERS.register_module()                → EpochBasedRunner_video
```

装饰器在 Python import 时调用 Registry 的注册函数，保存“名字 → 类对象”；实例仍要等 `build_from_cfg`。本项目的 `plugin=True` 使 `tools/train.py` import `projects/mmdet3d_plugin`，其 `__init__.py` 再 import 子模块，于是装饰器真正执行。

选择哪个 Registry 的原则不是“随便选一个”：看调用方用什么 build API。例如 `build_attention` 查 `ATTENTION`，`build_model` 查 detector/head 等模型 Registry；类还必须满足调用方期望的 `__init__` 参数与 `forward` 接口。


### 7. Runner、Hook 与一次 iteration

`custom_train_model` 不是注册进 Runner 的模型组件，而是项目 API：它构建 DataLoader、optimizer、runner，注册 hook，最后调用 `runner.run(data_loaders, workflow)`。

```text
runner.run
  → before_run hooks
  → for epoch:
       before_train_epoch hooks
       for data_batch:
          before_train_iter hooks
          model.train_step(data_batch, optimizer)
            → model(..., return_loss=True)
            → loss dict → parse_losses
          OptimizerHook: zero_grad → backward → step
          after_train_iter hooks
       after_train_epoch hooks
  → after_run hooks
```

Hook 类可定义多个触发点，如 `before_train_iter(runner)`、`after_train_epoch(runner)`。参数只有 `runner`，但 runner 持有 `runner.model`、`runner.optimizer`、`runner.outputs`、`runner.data_loader`、`runner.epoch/iter` 等训练状态。


In [ ]:
# Hook 的最小模型：类定义触发点，Runner 在固定时机调用它
class PrintHook:
    def before_train_iter(self, runner):
        print('iter', runner['iter'], 'loss before update:', runner['loss'])
    def after_train_iter(self, runner):
        print('optimizer has updated')

runner = {'iter': 12, 'loss': 1.23}
hook = PrintHook()
hook.before_train_iter(runner)
hook.after_train_iter(runner)


---

## 第三部分：模型主链路——图像如何变成 BEV

### 8. 图像 backbone 与 FPN：相机维先并入 batch

当前帧图像为 `[B,6,3,H,W]`。共享 ResNet/FPN 不为每个相机复制参数，而是：

```text
[B,6,3,H,W] → [B×6,3,H,W] → ResNet/FPN
             → 每尺度 [B×6,256,H_l,W_l]
             → 还原 [B,6,256,H_l,W_l]
```

四尺度 feature 之后会展平空间维并加 camera embedding 与 level embedding，得到 SCA 使用的图像 token `[N,S,B,256]`。此时仍是各相机的二维图像信息，尚未融合成 BEV。


### 9. BEV query 与两类 reference point

初始 `bev_queries` 是 `[L_bev,256]` 的可学习表，而不是由图像直接生成。`L_bev=200×200=40000`。encoder 为同一网格构造两类参考点：

| 名称 | 形状 | 用途 |
|-|-|-|
| `ref_2d` | `[B,L_bev,1,2]` | TSA 在历史/当前二维 BEV 上采样 |
| `ref_3d` | `[B,D=4,L_bev,3]` | SCA 将 4 个高度锚点投影到相机 |

`ref_3d` 的 x 来自 W 列、y 来自 H 行；反归一化后是当前 LiDAR 坐标。它不是 object query 的 3D reference point，后者属于检测 decoder。


### 10. SCA：一个 BEV cell 如何从六相机读取证据

```text
ref_3d [B,4,L,3]
  → lidar2img 投影
  → ref_cam [6,B,L,4,2] + bev_mask [6,B,L,4]
  → 对每相机仅保留可见 cell（rebatch）
  → 图像 value [6,S,B,C] 变为 [B×6,S,C]
  → MSDeformableAttention3D：每相机在 4 FPN level 局部采样
  → scatter 回完整 [B,L,C]
  → 按可见相机数平均 + 残差
```

每个 BEV 柱有 4 个 z 锚点。当前 `num_points=8` 会拆成“每高度锚点 2 点 × 4 锚点”，不是每个高度锚点 8 点。z 通过投影改变 `(u,v)`，所以 z 在 SCA 的图像采样中真实参与几何。


In [ ]:
# SCA 的关键形状（不运行真实 CUDA attention）
B, N, L, D, C, S = 1, 6, 200*200, 4, 256, 30825
ref_cam_shape = (N, B, L, D, 2)
img_tokens_shape = (N, S, B, C)
rebatch_query_shape = (B, N, 'max_len', C)
print('ref_cam:', ref_cam_shape)
print('image tokens:', img_tokens_shape)
print('only visible BEV cells are rebatch as:', rebatch_query_shape)


### 11. TSA：历史状态怎样与当前 query 融合

历史 BEV 先处理 yaw，再处理平移：

1. `rotate(prev_bev, yaw_delta)` 对 `[C,H,W]` 的历史 feature map 做整图旋转；
2. `shift_ref_2d = ref_2d + shift` 移动历史槽的采样中心；
3. TSA 从历史槽和当前槽分别稀疏采样，最后等权平均。

![BEV 坐标和数组方向](../docs/bev_coordinate_grid.svg)

![历史旋转与逆采样](../docs/bev_rotate_inverse_sampling.svg)

`2B` 不是 batch 真翻倍，而是 `(sample, time-slot)` 合并到 batch 维。槽内先独立聚合，再沿时间槽做 `mean`：

```text
value / locations / weights: [2B,...]  # 历史槽与当前槽独立
       ↓ deformable attention
output: [2B,L,C]
       ↓ 恢复 [L,C,B,2]，mean(time-slot)
TSA output: [B,L,C]
```


In [ ]:
# 2-queue 的正确展平与恢复（B=2 时最直观）
B, L, C = 2, 3, 2
history = torch.full((B, L, C), 10.)
current = torch.full((B, L, C), 20.)
queue = torch.stack([history, current], dim=1)  # [B,2,L,C]
flat = queue.reshape(2*B, L, C)
print('flat slot labels:', flat[:, 0, 0].tolist())  # [hist0,curr0,hist1,curr1]
restored = flat.reshape(B, 2, L, C)
print('temporal mean:', restored.mean(dim=1)[..., 0])


#### 11.1 `shift` 的实现边界

物理上，当前 ego 的局部位移通常写成 `(Δx_E,Δy_E)`；而当前仓库的源码 `shift` 在加到 `(W,H)` 前得到：

$$
(ΔW_{code},ΔH_{code})=(-Δy_E,Δx_E).
$$

这不是“数组 h 向下显示”本身能够解释的等价变换，而是该实现必须保留并单独验证的时序约定。修改 sin/cos、交换 `shift_x/shift_y` 或改变 yaw 符号，会改变既有 checkpoint 行为。

![标准局部位移与源码 shift 的对比](../docs/bev_shift_comparison.svg)


### 12. 一个 encoder layer 的固定顺序

每层都执行：

```text
TemporalSelfAttention → LayerNorm
SpatialCrossAttention → LayerNorm
FFN                   → LayerNorm
```

TSA 管时间，SCA 管当前多视角空间证据，FFN 做逐 token 的通道非线性变换。6 层后输出 `bev_embed=[B,40000,256]`；完整 Transformer 再转为 decoder 使用的 `[40000,B,256]`。


---

## 第四部分：BEV 上的 3D 检测、监督与调试

### 13. 检测头：Deformable DETR 的 BEV 化版本

检测头的结构和 Deformable DETR decoder 相似：900 个 object query 先 self-attention，再对 memory 做 deformable cross-attention，再经 FFN 和分类/回归分支。

差异是 memory：Deformable DETR 通常从多尺度图像读特征；BEVFormer 从单层 `200×200` BEV memory 读特征。

| reference point | 在哪里使用 | z 是否影响采样位置 |
|-|-|-|
| SCA `ref_3d=[B,4,L,3]` | 图像 feature 采样 | 是，z 改变投影 `(u,v)` |
| decoder `[B,900,3]` | BEV memory 采样 | 否，源码只传 `[..., :2]` |

检测 decoder 的 z 仍被逐层回归和更新，最后与 x/y、尺寸、yaw、速度共同组成 3D box。


In [ ]:
# 检测 decoder 中 z 不进入 BEV cross-attention 的最小等价代码
B, Q = 1, 900
reference_points = torch.rand(B, Q, 3)       # x, y, z
reference_points_input = reference_points[..., :2].unsqueeze(2)
print(reference_points.shape, '->', reference_points_input.shape)
assert reference_points_input.shape[-1] == 2


### 14. 预测、Hungarian matching 与 loss

每个 decoder layer 都输出：

```text
all_cls_scores: [6,B,900,10]  # 10 类 logits
all_bbox_preds: [6,B,900,10]  # (cx,cy,log w,log l,cz,log h,sin yaw,cos yaw,vx,vy)
```

训练时，对每张图的 900 个预测与数量可变的 GT 做 Hungarian 一对一匹配。匹配成本主要由分类成本和 L1 box 成本组成；匹配后：

- 分类使用 sigmoid Focal Loss；
- 回归使用按 box code 权重加权的 L1；
- 6 个 decoder layer 都有辅助监督，最后一层不是唯一计算 loss 的层。

推理仅取最后一层，通过 `NMSFreeCoder` 对 query-class 分数做全局 top-k、范围过滤和 box 解码；这里不执行传统 NMS。


In [ ]:
# 最小 Focal Loss 形状示例；真实代码还会处理 matching、背景和归一化因子
logits = torch.randn(4, 10)             # 4 个 query，10 类
target = torch.zeros_like(logits)
target[0, 2] = 1                        # query 0 匹配到类别 2
p = logits.sigmoid()
alpha, gamma = 0.25, 2.0
pt = torch.where(target == 1, p, 1 - p)
alpha_t = torch.where(target == 1, alpha, 1 - alpha)
focal = -alpha_t * (1 - pt).pow(gamma) * torch.log(pt.clamp_min(1e-6))
print('mean focal loss:', focal.mean().item())


### 15. 从命令到参数更新：最终检查表

```text
dist_train.sh config.py 8
  → tools/train.py: Config.fromfile + plugin import
  → build_model / build_dataset / build_dataloader
  → custom_train_model
  → build_optimizer + build_runner + register_hook
  → runner.run
  → model.train_step
  → forward_train / loss / parse_losses
  → backward / optimizer.step
```

建议的源码断点顺序：

1. `tools/train.py`：配置、plugin、build_model；
2. `apis/train.py`：DataLoader、Runner、Hook；
3. `BEVFormer.forward_train`：历史/当前分离；
4. `BEVFormerHead.forward`：BEV query、object query；
5. `PerceptionTransformer.get_bev_features`：CAN bus、图像 token；
6. `BEVFormerEncoder.forward`：reference point、TSA、SCA；
7. `DetectionTransformerDecoder.forward`：xy/z reference 更新；
8. `BEVFormerHead.loss`：matching 和 loss。

最先打印的五个量：`img.shape`、`prev_bev.shape/None`、四层 FPN shape、`bev_embed.shape`、`all_cls_scores/all_bbox_preds.shape`。


### 16. 下一步练习

1. 用一个单点 BEV heatmap 验证 `rotate(prev_bev, Δψ)` 后该点在哪个数组位置；
2. 固定一个 BEV cell 和一个相机，打印其 4 个 z 锚点投影后的 `(u,v)`；
3. 在 TSA 中打印 `[B,2,L,C] → [2B,L,C]` 的前几项，确认时间槽的交替顺序；
4. 在 decoder 断点确认 `[B,Q,3] → [B,Q,1,2]`，理解 z 为何不参与 BEV cross-attention；
5. 修改配置中的 `bev_h/bev_w` 或 `pc_range` 前，列出需要同步检查的 embedding、位置编码、reference point 与 coder 范围。

完整细节请回到对应 Markdown；本 notebook 的职责是提供一条不会迷路的实现阅读路线。


## 详细实现手册：A. 原始 nuScenes、坐标系与 PKL 生成

以下单元是从 `nuscence_to_pkl.md` 整理进 notebook 的完整细节快照。前面的教程单元负责建立主线；本节保留推导、边界条件、张量形状与源码定位，便于离线顺序阅读。


## BEVFormer 中 nuScenes 到 PKL 的完整数据处理流程

本文档只讨论 BEVFormer 仓库中的数据准备阶段：

```text
nuScenes 原始数据 + nuScenes CAN bus 扩展数据
                         |
                         v
              temporal train/val/test PKL
```

不讨论网络结构、训练过程、损失函数和推理过程。仓库的数据脚本在生成 PKL 后还会继续生成单目检测使用的 COCO JSON，但那属于下一阶段，不在本文的主要范围内。

本文的比较基线是 BEVFormer 安装文档明确指定的 MMDetection3D v0.17.1，而不是当前较新的 MMDetection3D v1.4.0。官方基线源码位于：

```text
mmdetection3d-v0.17.1/
```

BEVFormer 数据转换代码位于：

```text
tools/create_data.py
tools/data_converter/nuscenes_converter.py
```


### 阅读导航

全文可以分成六部分阅读：

| 章节 | 内容 |
|-|-|
| 第 1～4 节 | 建立 PKL、nuScenes 表、sample、keyframe、sweep 的基本概念 |
| 第 5～9 节 | 从命令行入口走到逐关键帧 info 构建 |
| 第 10～13 节 | 坐标系、齐次矩阵和 sensor -> current LIDAR_TOP 的完整推导 |
| 第 14～22 节 | 相机、sweep、3D框、类别、速度、CAN bus 和数据集划分 |
| 第 23～26 节 | PKL 的完整结构、字段表和示例 |
| 第 27～31 节 | BEVFormer 相对 MMDetection3D v0.17.1 的逐脚本修改 |
| 第 32～34 节 | 检查方法、常见误解和完整流程总结 |

---


### 1. 最重要的结论

首先建立几个贯穿全文的认识。


#### 1.1 PKL 是索引和元数据，不是传感器数据本体

PKL 中主要保存：

- 图片路径
- 点云路径
- 样本 token
- 时间戳
- 传感器标定参数
- 车辆位姿
- 传感器之间的坐标变换
- 3D 标注框
- 类别
- 目标速度
- 时序关系
- CAN bus 信息

PKL 不会复制或内嵌：

- 图片像素矩阵
- 完整 LiDAR 点云数组
- 完整历史 sweep 点云数组
- 神经网络特征

图片和点云仍然位于 nuScenes 原始目录中的 `samples/` 和 `sweeps/`。PKL 只记录如何找到它们，以及如何解释它们。


#### 1.2 当前关键帧的 LIDAR_TOP 是统一的 3D 参考坐标系

对每一个关键帧，转换器把该时刻的 LIDAR_TOP 坐标系作为统一 3D 参考系：

```text
当前关键帧的 3D GT 框       -> 当前 LIDAR_TOP 坐标系
当前关键帧的 GT 速度分量     -> 当前 LIDAR_TOP 坐标轴
历史 LiDAR sweep 的变换参数  -> 历史 LiDAR 到当前 LIDAR_TOP
六个相机的变换参数           -> 相机到当前 LIDAR_TOP
```

这不代表相机图片被变换成了 LiDAR 数据，也不代表历史点云在生成 PKL 时已经被改写。相机和历史 sweep 保存的是路径与变换关系。


#### 1.3 BEVFormer 没有重写 MMDetection3D 的核心 nuScenes 转换

相对于 MMDetection3D v0.17.1，BEVFormer 没有修改：

- scene 划分
- sample 遍历
- 相机信息提取
- sweep 收集
- sensor -> current LIDAR_TOP 坐标变换
- 3D 框转换
- 目标速度转换
- 类别映射
- valid_flag 生成

BEVFormer 主要增加：

- CAN bus 数据
- prev / next
- scene_token
- frame_idx
- temporal PKL 文件名
- 独立 out_dir 输出位置

并关闭了纯视觉模型不需要的点云 GT database 生成。

---


### 2. 输入数据目录

典型目录如下：

```text
data/
├── can_bus/
│   ├── scene-0001_meta.json
│   ├── scene-0001_pose.json
│   └── ...
└── nuscenes/
    ├── maps/
    ├── samples/
    │   ├── CAM_FRONT/
    │   ├── CAM_FRONT_LEFT/
    │   ├── CAM_FRONT_RIGHT/
    │   ├── CAM_BACK/
    │   ├── CAM_BACK_LEFT/
    │   ├── CAM_BACK_RIGHT/
    │   ├── LIDAR_TOP/
    │   └── RADAR_*/
    ├── sweeps/
    │   ├── CAM_*/
    │   ├── LIDAR_TOP/
    │   └── RADAR_*/
    ├── v1.0-trainval/
    │   ├── scene.json
    │   ├── sample.json
    │   ├── sample_data.json
    │   ├── sample_annotation.json
    │   ├── calibrated_sensor.json
    │   ├── ego_pose.json
    │   ├── sensor.json
    │   ├── category.json
    │   ├── instance.json
    │   ├── attribute.json
    │   └── visibility.json
    └── v1.0-test/
        └── ...
```

其中：

```text
samples/                 关键帧附近的传感器文件
sweeps/                  关键帧之间的非关键帧传感器文件
v1.0-*/                  nuScenes 的关系表、标定、位姿和标注
can_bus/                 自车 CAN bus 扩展数据
```

`samples/LIDAR_TOP/*.pcd.bin` 和 `sweeps/LIDAR_TOP/*.pcd.bin` 才是真正的点云文件；`lidar_path` 和 `sweeps[*].data_path` 只是指向这些文件的路径。

---


### 3. nuScenes 的数据组织方式

nuScenes 的 JSON 不是一棵直接嵌套的大 JSON，而是一组通过 token 关联的表。可以把它理解成一个关系型数据库。


#### 3.1 scene

一个 scene 表示一段连续驾驶片段，包含：

- scene token
- scene 名称
- 第一帧 sample token
- 最后一帧 sample token
- sample 数量
- 对应 log

一个 scene 中有多个关键帧 sample。


#### 3.2 sample

一个 sample 表示一个带完整标注的关键时刻。nuScenes 的关键帧频率大约为 2 Hz。

重要字段可以抽象为：

```python
sample = {
    'token': '当前关键帧唯一ID',
    'timestamp': 1532402927647951,
    'scene_token': '所属scene',
    'prev': '上一个关键帧sample token',
    'next': '下一个关键帧sample token',
    'data': {
        'LIDAR_TOP': '对应sample_data token',
        'CAM_FRONT': '对应sample_data token',
        # 其他相机和雷达
    },
    'anns': [
        'sample_annotation token 1',
        'sample_annotation token 2',
        # 当前时刻的全部3D标注
    ],
}
```

sample 本身不保存图片或点云路径，它通过 sample_data token 指向具体传感器记录。


#### 3.3 sample_data

一条 sample_data 对应某个传感器在某个时间采集的一份数据：

- 一张相机图片
- 一帧 LiDAR 点云
- 一帧 RADAR 点云

它包含：

- 文件路径
- 采集时间戳
- 传感器标定 token
- 采集时刻 ego_pose token
- prev / next sample_data token
- 是否是关键帧

特别注意两组 prev/next：

```text
sample.prev / sample.next
    连接带完整标注的关键帧，约 2 Hz

sample_data.prev / sample_data.next
    连接某个具体传感器的连续采样，频率通常更高
```

转换器通过 `sample_data.prev` 向前查找历史 LiDAR sweep。


#### 3.4 calibrated_sensor

calibrated_sensor 描述传感器在车体上的安装外参：

```text
sensor -> ego
```

包含：

```python
{
    'translation': [tx, ty, tz],
    'rotation': [qw, qx, qy, qz],
    'camera_intrinsic': [[...], [...], [...]],
}
```

对于固定安装的传感器，这些参数主要描述安装位置和安装朝向。


#### 3.5 ego_pose

ego_pose 描述传感器采集时刻自车在全局坐标系中的位姿：

```text
ego -> global
```

包含：

```python
{
    'translation': [tx, ty, tz],
    'rotation': [qw, qx, qy, qz],
}
```

每条 sample_data 都有自己的 ego_pose_token。这是因为相机、LiDAR 和 RADAR 并不保证完全同时采集，车辆在不同传感器采样之间可能已经发生了运动。


#### 3.6 sample_annotation

sample_annotation 保存一个物体在某个关键帧时刻的标注：

- 全局坐标中的3D框中心
- 全局坐标中的朝向
- 框尺寸
- 类别/实例关系
- 前后标注token
- 框内LiDAR点数
- 框内RADAR点数
- 可见性和属性

原始标注框的中心和朝向以 global 坐标系表示。


#### 3.7 token 关联总图

```text
scene
  |
  +--> first_sample_token / last_sample_token
             |
             v
           sample <------ prev / next ------> sample
             |
             +--> data['LIDAR_TOP'] ----------> sample_data
             |                                     |
             |                                     +--> calibrated_sensor
             |                                     +--> ego_pose
             |                                     +--> filename
             |                                     +--> prev / next sweep
             |
             +--> data['CAM_FRONT'] ----------> sample_data
             |                                     |
             |                                     +--> calibrated_sensor
             |                                     +--> ego_pose
             |                                     +--> filename
             |
             +--> anns[] ---------------------> sample_annotation
                                                   |
                                                   +--> instance
                                                   +--> category
                                                   +--> attribute
```

---


### 4. sample、keyframe 和 sweep

这三个概念需要严格区分。


#### 4.1 sample/keyframe

一个 sample 是一个带完整 3D 标注的关键时刻：

```text
KEYFRAME -------- KEYFRAME -------- KEYFRAME
  完整标注           完整标注           完整标注
```

PKL 顶层 `infos` 中的一个元素对应一个 sample/keyframe。


#### 4.2 sweep

传感器频率高于关键帧标注频率，因此两个关键帧之间还有其他采样：

```text
时间 --->

KEYFRAME -- sweep -- sweep -- sweep -- KEYFRAME
```

当前转换器中的：

```python
info['sweeps']
```

专门收集当前关键帧之前的历史 LIDAR_TOP 数据，不是六个相机，也不是毫米波 RADAR。


#### 4.3 sweep 为什么需要坐标对齐

每个历史 sweep 点云中的点最初位于它自己采样时刻的 LiDAR 坐标系。车辆在历史时刻到当前关键帧之间发生了运动，所以不能直接把历史点与当前点拼接。

正确关系是：

```text
历史 sweep LiDAR
    -> 历史时刻 ego
    -> global
    -> 当前关键帧 ego
    -> 当前关键帧 LIDAR_TOP
```

转换器只把这个变换的旋转和平移写入 PKL。它不会在生成 PKL 时加载并改写全部历史点云。

---


### 5. 转换入口和执行命令

仓库文档提供的命令为：

```bash
python tools/create_data.py nuscenes \
    --root-path ./data/nuscenes \
    --out-dir ./data/nuscenes \
    --extra-tag nuscenes \
    --version v1.0 \
    --canbus ./data
```

参数含义：

| 参数 | 含义 |
|-|-|
| `nuscenes` | 选择 nuScenes 数据转换分支 |
| `--root-path` | nuScenes 原始数据根目录 |
| `--out-dir` | 生成的 PKL 输出目录 |
| `--extra-tag` | 输出文件名前缀 |
| `--version` | 数据版本前缀，v1.0 会组合出 trainval/test |
| `--canbus` | 包含 can_bus/ 的根目录 |
| `--max-sweeps` | 每个关键帧最多记录多少个历史 LiDAR sweep，默认 10 |

当 `--version v1.0` 时，入口会依次处理：

```text
v1.0-trainval
v1.0-test
```

当 `--version v1.0-mini` 时，只处理：

```text
v1.0-mini
```

如果只下载了 trainval 而没有 test，却执行完整 v1.0 分支，第二次 test 转换可能因缺少 v1.0-test 数据而失败。

---


### 6. Python 调用链

从命令行到 PKL 的调用关系如下：

```text
tools/create_data.py
  |
  +--> argparse 解析命令行
  |
  +--> nuscenes_data_prep(...)
          |
          +--> nuscenes_converter.create_nuscenes_infos(...)
                  |
                  +--> NuScenes(...)
                  +--> NuScenesCanBus(...)
                  +--> get_available_scenes(...)
                  +--> _fill_trainval_infos(...)
                  |       |
                  |       +--> _get_can_bus_info(...)
                  |       +--> obtain_sensor2top(...) for 6 cameras
                  |       +--> obtain_sensor2top(...) for LiDAR sweeps
                  |       +--> collect GT boxes/names/velocity
                  |
                  +--> mmcv.dump(train/val/test PKL)
```

PKL 生成之后，`nuscenes_data_prep()` 还会调用 `export_2d_annotation()`。那一步是 PKL 到单目 COCO JSON 的后续流程，不属于本文的 nuScenes 到 PKL 主链路。

---


### 7. 第一步：初始化 nuScenes SDK 和 CAN bus SDK

核心代码：

```python
nusc = NuScenes(
    version=version,
    dataroot=root_path,
    verbose=True,
)

nusc_can_bus = NuScenesCanBus(
    dataroot=can_bus_root_path,
)
```

NuScenes 负责：

- 加载 nuScenes JSON 表
- 根据 token 查询关联记录
- 得到图片/点云路径
- 读取并转换 3D Box
- 估算目标速度
- 提供官方数据划分

NuScenesCanBus 负责：

- 按 scene 读取 CAN bus 消息
- 提供车辆 pose、速度、加速度、角速度等信息

脚本不是自己手写 JSON 字符串解析，而是通过官方 SDK 查询：

```python
nusc.get('sample', token)
nusc.get('sample_data', token)
nusc.get('calibrated_sensor', token)
nusc.get('ego_pose', token)
nusc.get('sample_annotation', token)
nusc.get_sample_data(token)
nusc.box_velocity(token)
```

---


### 8. 第二步：确定 train/val/test scene

脚本使用 nuScenes 官方划分：

```python
if version == 'v1.0-trainval':
    train_scenes = splits.train
    val_scenes = splits.val
elif version == 'v1.0-test':
    train_scenes = splits.test
    val_scenes = []
elif version == 'v1.0-mini':
    train_scenes = splits.mini_train
    val_scenes = splits.mini_val
```

这里最初得到的是 scene 名称，例如：

```text
scene-0001
scene-0002
```

随后 `get_available_scenes()` 检查 scene 对应的 LiDAR 文件是否存在，再把 scene 名称转换为 scene token 集合：

```python
train_scenes: set[str]
val_scenes: set[str]
```

这样在遍历 sample 时，就可以通过：

```python
sample['scene_token'] in train_scenes
```

决定该 sample 写入 train 还是 val。

**实现细节**：当前 `get_available_scenes()` 实际上只检查每个 scene 第一条取得的 LiDAR 路径便退出循环，并没有遍历验证 scene 内所有文件。数据若只缺中间某些文件，仍可能在后续读取时才报错。

---


### 9. 第三步：逐关键帧构建 info

核心循环：

```python
for sample in nusc.sample:
    ...
```

每次循环生成一个 info 字典，对应一个关键帧。


#### 9.1 取得当前 LIDAR_TOP 记录

```python
lidar_token = sample['data']['LIDAR_TOP']
sd_rec = nusc.get('sample_data', lidar_token)
cs_record = nusc.get(
    'calibrated_sensor',
    sd_rec['calibrated_sensor_token'],
)
pose_record = nusc.get(
    'ego_pose',
    sd_rec['ego_pose_token'],
)
lidar_path, boxes, _ = nusc.get_sample_data(lidar_token)
```

这里得到：

| 变量 | 含义 |
|-|-|
| `lidar_token` | 当前关键帧 LIDAR_TOP 的 sample_data token |
| `sd_rec` | 当前 LiDAR 的 sample_data 记录 |
| `cs_record` | 当前 LiDAR 的 sensor -> ego 标定 |
| `pose_record` | 当前 LiDAR 采集时刻的 ego -> global 位姿 |
| `lidar_path` | 当前关键帧点云文件路径 |
| `boxes` | 已被 nuScenes SDK 转到当前 LIDAR_TOP 坐标系的 3D 框 |

最后一点尤其重要：

```text
nusc.get_box(annotation_token)
    返回 global 坐标中的框

nusc.get_sample_data(lidar_token)
    返回与这个传感器数据对应、并已转到该传感器坐标系中的框
```

因此后续 `box.center` 已经是当前 LIDAR_TOP 坐标，不需要脚本再手动执行 global -> LiDAR 平移旋转。


#### 9.2 建立基础 info 字典

```python
info = {
    'lidar_path': lidar_path,
    'token': sample['token'],
    'prev': sample['prev'],
    'next': sample['next'],
    'can_bus': can_bus,
    'frame_idx': frame_idx,
    'sweeps': [],
    'cams': {},
    'scene_token': sample['scene_token'],
    'lidar2ego_translation': cs_record['translation'],
    'lidar2ego_rotation': cs_record['rotation'],
    'ego2global_translation': pose_record['translation'],
    'ego2global_rotation': pose_record['rotation'],
    'timestamp': sample['timestamp'],
}
```

顶层的 `lidar2ego_*` 和 `ego2global_*` 都以当前关键帧 LIDAR_TOP 的采集时刻为准。

---


### 10. 坐标系定义


#### 10.1 global 坐标系

global 是一个 scene 内固定不动的世界坐标系：

```text
x、y：地图平面
z：向上
```

它不跟着车辆运动。原始 sample_annotation 的中心与朝向主要在 global 中表达。

global 的 x/y 轴方向由地图坐标定义，不能简单认为 global x 永远等于车辆前方。


#### 10.2 ego 坐标系

ego 是跟随自车运动的车体坐标系：

```text
x：车辆前方
y：车辆左方
z：车辆上方
```

ego_pose 给出采集时刻：

```text
ego -> global
```


#### 10.3 LIDAR_TOP 坐标系

LIDAR_TOP 原点位于车顶 LiDAR：

```text
x：LiDAR 前方
y：LiDAR 左方
z：LiDAR 上方
```

LiDAR 点云文件中的点首先位于采集该文件时的 LiDAR 坐标系。

calibrated_sensor 给出：

```text
LiDAR -> ego
```


#### 10.4 相机坐标系

相机通常使用：

```text
x：图像右方
y：图像下方
z：镜头前方
```

相机前方有效点满足：

```text
z_{cam} > 0
```

相机 calibrated_sensor 给出：

```text
camera -> ego
```


#### 10.5 图像像素坐标系

图像左上角为原点：

```text
u：向右
v：向下
```

它是二维坐标系，不是本文 PKL 统一采用的三维 LIDAR_TOP 坐标系。

---


### 11. 坐标变换的数学基础


#### 11.1 记号约定

本文使用列向量，定义：

$$T_{A \leftarrow B}$$

表示把 B 坐标系中的坐标转换到 A 坐标系。

三维刚体变换写为：

$$\mathbf{p}_A = R_{A \leftarrow B}\mathbf{p}_B + \mathbf{t}_{A \leftarrow B}$$

齐次矩阵写为：

$$T_{A \leftarrow B} = \begin{bmatrix} R_{A \leftarrow B} & \mathbf{t}_{A \leftarrow B} \\ \mathbf{0}^{T} & 1 \end{bmatrix}$$

齐次点：

$$\tilde{\mathbf{p}}= \begin{bmatrix} x & y & z & 1 \end{bmatrix}^{T}$$

于是：

$$\tilde{\mathbf{p}}_A = T_{A \leftarrow B} \tilde{\mathbf{p}}_B$$


#### 11.2 变换组合

如果已知：

```text
B -> A
C -> B
```

那么：

$$T_{A \leftarrow C} = T_{A \leftarrow B} T_{B \leftarrow C}$$

矩阵乘法从右向左执行。


#### 11.3 逆变换

如果：

$$\mathbf{p}_A=R\mathbf{p}_B+\mathbf{t}$$

则：

$$\mathbf{p}_B=R^T(\mathbf{p}_A-\mathbf{t})$$

因此：

$$T_{B \leftarrow A} = T_{A \leftarrow B}^{-1} = \begin{bmatrix} R^T & -R^T\mathbf{t} \\ \mathbf{0}^{T} & 1 \end{bmatrix}$$

旋转矩阵是正交矩阵，所以：

$$R^{-1}=R^T$$


#### 11.4 代码使用行向量

上面的公式使用列向量，便于推导。当前 converter 的 NumPy 代码大量采用行向量，应用形式为：

```python
points_target = points_source @ rotation.T + translation
```

这与列向量形式：

$$\mathbf{p}_{target}=R\mathbf{p}_{source}+\mathbf{t}$$

表达的是同一个变换。阅读源码时必须注意 `.T` 来自行向量/列向量写法差异，不代表多做了一次物理旋转。

---


### 12. 任意传感器到当前 LIDAR_TOP 的完整变换

设：

- $S_s$：源传感器在源采样时刻 s 的坐标系
- $E_s$：源采样时刻的 ego 坐标系
- $G$：global 坐标系
- $E_k$：当前关键帧时刻 k 的 ego 坐标系
- $L_k$：当前关键帧 LIDAR_TOP 坐标系

目标变换是：

```text
S_s -> E_s -> G -> E_k -> L_k
```

齐次矩阵为：

$$T_{L_k \leftarrow S_s} = T_{L_k \leftarrow E_k} T_{E_k \leftarrow G} T_{G \leftarrow E_s} T_{E_s \leftarrow S_s}$$

利用逆矩阵可写为：

$$T_{L_k \leftarrow S_s} = T_{E_k \leftarrow L_k}^{-1} T_{G \leftarrow E_k}^{-1} T_{G \leftarrow E_s} T_{E_s \leftarrow S_s}$$

也可以先定义：

$$T_{G \leftarrow S_s} = T_{G \leftarrow E_s} T_{E_s \leftarrow S_s}$$

以及：

$$T_{G \leftarrow L_k} = T_{G \leftarrow E_k} T_{E_k \leftarrow L_k}$$

那么最简洁的形式是：

$$T_{L_k \leftarrow S_s} = T_{G \leftarrow L_k}^{-1} T_{G \leftarrow S_s}$$

这就是“源传感器先到 global，再到当前 LIDAR_TOP”的数学表达。


#### 12.1 展开到点坐标

源传感器到源 ego：

$$\mathbf{p}_{E_s} = R_{E_s \leftarrow S_s}\mathbf{p}_{S_s} + \mathbf{t}_{E_s \leftarrow S_s}$$

源 ego 到 global：

$$\mathbf{p}_G = R_{G \leftarrow E_s}\mathbf{p}_{E_s} + \mathbf{t}_{G \leftarrow E_s}$$

global 到当前 ego：

$$\mathbf{p}_{E_k} = R_{G \leftarrow E_k}^{T} (\mathbf{p}_G-\mathbf{t}_{G \leftarrow E_k})$$

当前 ego 到当前 LiDAR：

$$\mathbf{p}_{L_k} = R_{E_k \leftarrow L_k}^{T} (\mathbf{p}_{E_k}-\mathbf{t}_{E_k \leftarrow L_k})$$

组合后就是：

$$\mathbf{p}_{L_k} = R_{L_k \leftarrow S_s}\mathbf{p}_{S_s} + \mathbf{t}_{L_k \leftarrow S_s}$$

转换器将最终的旋转和平移保存为：

```text
sensor2lidar_rotation
sensor2lidar_translation
```

按代码约定使用：

```python
points_lidar = (
    points_sensor @ sensor2lidar_rotation.T
    + sensor2lidar_translation
)
```

---


### 13. `obtain_sensor2top()` 的输入和输出

这个函数同时服务于：

- 六个相机 -> 当前关键帧 LIDAR_TOP
- 历史 LiDAR sweep -> 当前关键帧 LIDAR_TOP

输入：

```python
obtain_sensor2top(
    nusc,
    sensor_token,
    current_lidar2ego_translation,
    current_lidar2ego_rotation_matrix,
    current_ego2global_translation,
    current_ego2global_rotation_matrix,
    sensor_type,
)
```

函数先通过 sensor_token 查出源传感器自己的：

- data_path
- sensor2ego
- ego2global
- timestamp

然后计算源传感器到当前关键帧 LIDAR_TOP 的：

- sensor2lidar_rotation
- sensor2lidar_translation

最终输出：

```python
{
    'data_path': str,
    'type': str,
    'sample_data_token': str,
    'sensor2ego_translation': list,
    'sensor2ego_rotation': list,
    'ego2global_translation': list,
    'ego2global_rotation': list,
    'timestamp': int,
    'sensor2lidar_rotation': np.ndarray,      # 3x3
    'sensor2lidar_translation': np.ndarray,   # 3
}
```

为什么不能只用 sensor2ego：相机或历史 sweep 与当前 LiDAR 的采集时刻可能不同。只通过安装外参无法补偿车辆在两个采样时刻之间的运动，必须经过各自的 ego2global。

---


### 14. 六个相机信息的生成

转换器遍历：

```python
camera_types = [
    'CAM_FRONT',
    'CAM_FRONT_RIGHT',
    'CAM_FRONT_LEFT',
    'CAM_BACK',
    'CAM_BACK_LEFT',
    'CAM_BACK_RIGHT',
]
```

对每个相机执行：

```python
cam_token = sample['data'][cam]
cam_path, _, cam_intrinsic = nusc.get_sample_data(cam_token)

cam_info = obtain_sensor2top(...)
cam_info.update(cam_intrinsic=cam_intrinsic)
info['cams'][cam] = cam_info
```

每个相机最终结构为：

```python
{
    'data_path': str,
    'type': 'CAM_FRONT',
    'sample_data_token': str,

    'sensor2ego_translation': list[float],
    'sensor2ego_rotation': list[float],
    'ego2global_translation': list[float],
    'ego2global_rotation': list[float],
    'timestamp': int,

    'sensor2lidar_rotation': np.ndarray,      # (3, 3)
    'sensor2lidar_translation': np.ndarray,   # (3,)
    'cam_intrinsic': list | np.ndarray,       # (3, 3)
}
```

其中：

```text
sensor2ego_*       相机 -> 相机采集时刻 ego
ego2global_*       相机采集时刻 ego -> global
sensor2lidar_*     相机 -> 当前关键帧 LIDAR_TOP
cam_intrinsic      相机三维坐标 -> 像素坐标所需内参
```

生成 PKL 时不会把图片扭曲或重投影到 LiDAR 平面。`data_path` 仍指向原始 JPG。


#### 14.1 相机内参公式

相机三维点：

$$\mathbf{p}_C= \begin{bmatrix} x_C & y_C & z_C \end{bmatrix}^{T}$$

相机内参：

$$K= \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

投影满足：

$$\lambda \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K \begin{bmatrix} x_C \\ y_C \\ z_C \end{bmatrix}$$

其中 lambda = z_C，所以：

$$u=f_x\frac{x_C}{z_C}+c_x$$

$$v=f_y\frac{y_C}{z_C}+c_y$$

PKL 只保存 K 和相机外参，不在这一阶段完成整张图片的三维重建。

---


### 15. 历史 LiDAR sweeps 的生成

转换器从当前 LIDAR_TOP 的 sample_data 开始，沿 prev 向前查找：

```python
sd_rec = nusc.get(
    'sample_data',
    sample['data']['LIDAR_TOP'],
)

sweeps = []
while len(sweeps) < max_sweeps:
    if sd_rec['prev'] != '':
        sweep = obtain_sensor2top(
            nusc,
            sd_rec['prev'],
            ...,
            'lidar',
        )
        sweeps.append(sweep)
        sd_rec = nusc.get('sample_data', sd_rec['prev'])
    else:
        break
```

每个 sweep 保存：

```python
{
    'data_path': 'sweeps/LIDAR_TOP/xxx.pcd.bin',
    'type': 'lidar',
    'sample_data_token': str,
    'timestamp': int,

    'sensor2ego_translation': list,
    'sensor2ego_rotation': list,
    'ego2global_translation': list,
    'ego2global_rotation': list,

    'sensor2lidar_rotation': np.ndarray,
    'sensor2lidar_translation': np.ndarray,
}
```

这里的 sensor2lidar 表示：

```text
历史 sweep 的 LIDAR_TOP
    -> 历史时刻 ego
    -> global
    -> 当前关键帧 ego
    -> 当前关键帧 LIDAR_TOP
```

如果之后加载历史点云，才会执行：

```python
history_points_in_current_lidar = (
    history_points @ rotation.T + translation
)
```

因此 PKL 生成阶段做的是“预计算对齐关系”，不是“预先融合全部点云”。

---


### 16. 3D GT 框的生成与坐标转换


#### 16.1 原始标注在哪里

nuScenes 原始 sample_annotation 中的框中心和朝向位于 global 坐标系。

如果直接调用：

```python
box = nusc.get_box(annotation_token)
```

得到的是 global 中的 box。


#### 16.2 converter 为什么没有显式写 global -> LiDAR

转换器使用：

```python
lidar_path, boxes, _ = nusc.get_sample_data(lidar_token)
```

nuScenes SDK 已经在 `get_sample_data()` 内部把与当前 sample 关联的 global boxes 转换到了所请求的传感器坐标系。因为这里请求的是当前 LIDAR_TOP，所以 boxes 已经在当前 LIDAR_TOP 坐标系。

概念上的位置转换为：

```text
global box center
    -> 当前LiDAR采集时刻 ego
    -> 当前 LIDAR_TOP
```

列向量公式：

$$\mathbf{c}_{E_k} = R_{G \leftarrow E_k}^{T} (\mathbf{c}_G-\mathbf{t}_{G \leftarrow E_k})$$

$$\mathbf{c}_{L_k} = R_{E_k \leftarrow L_k}^{T} (\mathbf{c}_{E_k}-\mathbf{t}_{E_k \leftarrow L_k})$$

合并为：

$$\mathbf{c}_{L_k} = R_{E_k \leftarrow L_k}^{T} \left[ R_{G \leftarrow E_k}^{T} (\mathbf{c}_G-\mathbf{t}_{G \leftarrow E_k}) -\mathbf{t}_{E_k \leftarrow L_k} \right]$$


#### 16.3 朝向转换

设 box 在 global 中的旋转为 $R_{G \leftarrow B}$，那么 box 到当前 LiDAR 的朝向为：

$$R_{L_k \leftarrow B} = R_{E_k \leftarrow L_k}^{T} R_{G \leftarrow E_k}^{T} R_{G \leftarrow B}$$

这一步也由 nuScenes SDK 在 `get_sample_data()` 内完成。


#### 16.4 尺寸和 yaw 写入

代码提取：

```python
locs = np.array([b.center for b in boxes])
dims = np.array([b.wlh for b in boxes])
rots = np.array([
    b.orientation.yaw_pitch_roll[0]
    for b in boxes
])
```

其中：

```text
locs = [x, y, z]
dims = [w, l, h]
rots = nuScenes SDK 当前LiDAR框约定下的 yaw
```

刚体坐标变换不会改变物体真实尺寸，所以 w/l/h 的数值不因 global -> LiDAR 旋转和平移而改变。

随后为兼容 MMDetection3D v0.17.1/SECOND 的框角度约定：

```python
yaw_second = -yaw_nuscenes - np.pi / 2
```

最终：

```python
gt_boxes = np.concatenate(
    [locs, dims, -rots - np.pi / 2],
    axis=1,
)
```

每个框为：

```text
[x, y, z, w, l, h, yaw]
```

这里的 `-yaw-pi/2` 是不同框架对 yaw 零方向、正方向和长宽轴定义不同造成的参数约定转换，不是物体在现实中被额外旋转。

---


### 17. 类别映射

nuScenes 原始类别较细，例如：

```text
vehicle.car
vehicle.bus.rigid
vehicle.bus.bendy
human.pedestrian.adult
human.pedestrian.child
movable_object.trafficcone
```

转换器使用：

```python
NuScenesDataset.NameMapping
```

将它们映射为检测任务使用的 10 类：

```text
car
truck
trailer
bus
construction_vehicle
bicycle
motorcycle
pedestrian
traffic_cone
barrier
```

结果保存为：

```python
info['gt_names'] = np.ndarray(shape=(N,))
```

N 是当前关键帧中的标注数量。

---


### 18. GT 速度的计算与坐标转换


#### 18.1 原始速度怎么得到

sample_annotation 本身不直接存储完整速度字段。转换器调用：

```python
nusc.box_velocity(annotation_token)
```

nuScenes SDK 根据同一个 instance 相邻时刻的标注中心估算速度，本质是：

$$\mathbf{v}_G \approx \frac{ \mathbf{c}_G(t_2)-\mathbf{c}_G(t_1) }{t_2-t_1}$$

由于标注中心位于 global，所以得到的速度最初也是 global 坐标轴下的速度分量。

转换器取水平速度：

```python
velocity = nusc.box_velocity(token)[:2]
```

并补上垂直分量 0：

```python
velo = np.array([vx_global, vy_global, 0.0])
```


#### 18.2 为什么速度只旋转、不平移

位置需要旋转和平移：

$$\mathbf{p}'=R\mathbf{p}+\mathbf{t}$$

速度是位置差除以时间差：

$$\mathbf{v} = \frac{\mathbf{p}_2-\mathbf{p}_1}{\Delta t}$$

对两个位置施加同一个平移：

$$(\mathbf{p}_2+\mathbf{t}) -(\mathbf{p}_1+\mathbf{t}) = \mathbf{p}_2-\mathbf{p}_1$$

平移抵消，所以这里只需要把速度方向分量从 global 坐标轴旋转到当前 LiDAR 坐标轴：

$$\mathbf{v}_{L_k} = R_{E_k \leftarrow L_k}^{T} R_{G \leftarrow E_k}^{T} \mathbf{v}_G$$

对应行向量代码：

```python
velo = (
    velo
    @ np.linalg.inv(e2g_r_mat).T
    @ np.linalg.inv(l2e_r_mat).T
)
```

最后只保存水平分量：

```python
info['gt_velocity'] = velocity.reshape(-1, 2)
```


#### 18.3 速度大小不会改变

纯旋转满足：

$$\|R\mathbf{v}\|_2=\|\mathbf{v}\|_2$$

例如：

```text
global速度分量：[3, 4] m/s
速度大小：5 m/s

旋转后LiDAR分量：[4, -3] m/s
速度大小：仍然是5 m/s
```

改变的是坐标分量，不是物体真实速度。


#### 18.4 这里不是相对自车速度

当前代码计算：

$$\tilde{\mathbf{v}}_{object,L} = R_{L \leftarrow G}\mathbf{v}_{object,G}$$

它没有计算：

$$\mathbf{v}_{relative,L} = R_{L \leftarrow G} (\mathbf{v}_{object,G}-\mathbf{v}_{ego,G})$$

也没有加入旋转坐标系严格运动学中的角速度项。因此 `gt_velocity` 的语义是：

```text
目标对地绝对速度
以当前 LIDAR_TOP 坐标轴表达分量
```

静止路边车辆的 `gt_velocity` 仍为 [0, 0]，即使自车正在高速行驶。

---


### 19. 点数与 valid_flag

每个 sample_annotation 已包含：

```text
num_lidar_pts
num_radar_pts
```

转换器保存为：

```python
info['num_lidar_pts'] = np.ndarray(shape=(N,))
info['num_radar_pts'] = np.ndarray(shape=(N,))
```

并计算：

```python
valid_flag = (
    num_lidar_pts + num_radar_pts > 0
)
```

数学表达：

$$valid_i = \left(n^{lidar}_i+n^{radar}_i>0\right)$$

结果：

```python
info['valid_flag'] = np.ndarray(
    shape=(N,),
    dtype=bool,
)
```

这并不意味着 BEVFormer 使用 LiDAR/RADAR 点作为输入。它只是借助 nuScenes 已统计的点数，判断标注目标是否至少获得了一些传感器观测支持。

---


### 20. CAN bus 信息的生成

BEVFormer 新增 `_get_can_bus_info()`：

```python
scene_name = nusc.get(
    'scene', sample['scene_token']
)['name']

pose_list = nusc_can_bus.get_messages(
    scene_name,
    'pose',
)
```

然后遍历 CAN pose 消息，寻找：

- 时间戳不晚于当前sample
- 并且最接近当前sample的消息

即理想条件：

$$t_{can} = \max\{t_i\mid t_i\leq t_{sample}\}$$

随后构造 18 维数组：

- 前3维：pos
- 接着4维：orientation四元数
- 中间维度：pose消息中的其余车辆状态
- 最后2维：预留为0

保存为：

```python
info['can_bus'] = np.ndarray(shape=(18,))
```

如果某些 scene 没有 CAN bus 数据，代码返回：

```python
np.zeros(18)
```


#### 20.1 当前实现的注意事项

代码写成：

```python
for key in last_pose.keys():
    can_bus.extend(pose[key])
```

它遍历 `last_pose.keys()`，但取值使用 `pose[key]`。循环因遇到第一条未来消息而退出时，pose 可能是未来消息，而 last_pose 才是最近的历史消息。

从代码意图看，更一致的写法应当是：

```python
for key in last_pose.keys():
    can_bus.extend(last_pose[key])
```

此外，函数通过 `pop()` 修改了 last_pose 字典。更稳妥的实现通常会先复制：

```python
last_pose = last_pose.copy()
```

本文只记录现状，不在数据文档中直接修改原实现。

---


### 21. frame_idx 与场景时序字段

BEVFormer 在遍历 sample 时维护：

```python
frame_idx = 0
```

每个 info 保存：

```python
'frame_idx': frame_idx
```

然后：

```python
if sample['next'] == '':
    frame_idx = 0
else:
    frame_idx += 1
```

同时保存：

```python
'prev': sample['prev']
'next': sample['next']
'scene_token': sample['scene_token']
```

这些字段共同表达：

- 当前帧属于哪个scene
- 它在scene中的顺序
- 它前后连接哪些关键帧

这套逻辑依赖 `nusc.sample` 按 scene 的连续顺序组织。官方 nuScenes 表满足当前代码的这一使用方式。

---


### 22. train、val、test 的写入

每个 sample 完成 info 后：

```python
if sample['scene_token'] in train_scenes:
    train_nusc_infos.append(info)
else:
    val_nusc_infos.append(info)
```

非 test 版本写出：

```text
{prefix}_infos_temporal_train.pkl
{prefix}_infos_temporal_val.pkl
```

test 版本写出：

```text
{prefix}_infos_temporal_test.pkl
```

当 extra-tag=nuscenes 时为：

```text
nuscenes_infos_temporal_train.pkl
nuscenes_infos_temporal_val.pkl
nuscenes_infos_temporal_test.pkl
```

train/val 包含 GT 标注；nuScenes test 不公开 GT，所以 test PKL 不包含：

```text
gt_boxes
gt_names
gt_velocity
num_lidar_pts
num_radar_pts
valid_flag
```

---


### 23. PKL 顶层结构

train/val/test 的顶层结构一致：

```python
{
    'infos': list[dict],
    'metadata': {
        'version': str,
    },
}
```

例如：

```python
{
    'infos': [
        info_0,
        info_1,
        info_2,
        # ...
    ],
    'metadata': {
        'version': 'v1.0-trainval',
    },
}
```

每个 info 对应一个关键帧，不是对应一张相机图片。一个 info 内部同时包含六个相机。

---


### 24. 单帧 info 的完整字段表


#### 24.1 基础和时序字段

| 字段 | 类型/形状 | 坐标系 | 含义 |
|-|-|-|-|
| `token` | str | 无 | 当前关键帧 sample token |
| `prev` | str | 无 | 上一个关键帧 token，首帧为空字符串 |
| `next` | str | 无 | 下一个关键帧 token，末帧为空字符串 |
| `scene_token` | str | 无 | 所属 scene token |
| `frame_idx` | int | 无 | scene 内关键帧编号 |
| `timestamp` | int | 无 | sample 时间戳，单位通常为微秒 |
| `can_bus` | np.ndarray(18) | 主要含 global/车体状态 | 自车 CAN bus pose 和运动状态 |


#### 24.2 当前 LIDAR_TOP 字段

| 字段 | 类型/形状 | 坐标关系 | 含义 |
|-|-|-|-|
| `lidar_path` | str | 点在当前 LiDAR 坐标 | 当前关键帧点云文件路径 |
| `lidar2ego_translation` | 长度 3 | LiDAR -> ego | 当前 LiDAR 安装平移 |
| `lidar2ego_rotation` | 长度 4 四元数 | LiDAR -> ego | 当前 LiDAR 安装旋转 |
| `ego2global_translation` | 长度 3 | ego -> global | 当前 LiDAR 时刻自车全局位置 |
| `ego2global_rotation` | 长度 4 四元数 | ego -> global | 当前 LiDAR 时刻自车全局朝向 |


#### 24.3 相机字段

```python
info['cams'] = {
    'CAM_FRONT': cam_info,
    'CAM_FRONT_RIGHT': cam_info,
    'CAM_FRONT_LEFT': cam_info,
    'CAM_BACK': cam_info,
    'CAM_BACK_LEFT': cam_info,
    'CAM_BACK_RIGHT': cam_info,
}
```

每个 cam_info：

| 字段 | 类型/形状 | 含义 |
|-|-|-|
| `data_path` | str | 原始 JPG 路径 |
| `type` | str | 相机名称 |
| `sample_data_token` | str | 相机 sample_data token |
| `timestamp` | int | 相机实际采集时间 |
| `sensor2ego_translation` | 长度 3 | camera -> camera-time ego 平移 |
| `sensor2ego_rotation` | 长度 4 | camera -> camera-time ego 旋转 |
| `ego2global_translation` | 长度 3 | camera-time ego -> global 平移 |
| `ego2global_rotation` | 长度 4 | camera-time ego -> global 旋转 |
| `sensor2lidar_rotation` | (3,3) | camera -> current LIDAR_TOP 旋转 |
| `sensor2lidar_translation` | (3,) | camera -> current LIDAR_TOP 平移 |
| `cam_intrinsic` | (3,3) | 相机内参 K |


#### 24.4 sweeps 字段

```python
info['sweeps'] = list[sweep_info]
```

每个 sweep_info：

| 字段 | 类型/形状 | 含义 |
|-|-|-|
| `data_path` | str | 历史 LiDAR 点云路径 |
| `type` | 'lidar' | 传感器类型 |
| `sample_data_token` | str | 历史 LiDAR sample_data token |
| `timestamp` | int | 历史 LiDAR 采集时间 |
| `sensor2ego_translation` | 长度 3 | 历史 LiDAR -> 历史 ego |
| `sensor2ego_rotation` | 长度 4 | 历史 LiDAR -> 历史 ego |
| `ego2global_translation` | 长度 3 | 历史 ego -> global |
| `ego2global_rotation` | 长度 4 | 历史 ego -> global |
| `sensor2lidar_rotation` | (3,3) | 历史 LiDAR -> 当前 LiDAR 旋转 |
| `sensor2lidar_translation` | (3,) | 历史 LiDAR -> 当前 LiDAR 平移 |


#### 24.5 GT 字段，仅 train/val

| 字段 | 类型/形状 | 坐标系 | 含义 |
|-|-|-|-|
| `gt_boxes` | (N,7) | 当前 LIDAR_TOP | [x,y,z,w,l,h,yaw] |
| `gt_names` | (N,) | 无 | 映射后的类别名称 |
| `gt_velocity` | (N,2) | 当前 LiDAR 坐标轴 | 对地绝对速度 [vx,vy] |
| `num_lidar_pts` | (N,) | 无 | 每个框内 LiDAR 点数 |
| `num_radar_pts` | (N,) | 无 | 每个框内 RADAR 点数 |
| `valid_flag` | (N,) bool | 无 | LiDAR+RADAR 点数是否大于 0 |

---


### 25. 一个简化的 PKL 示例

```python
{
    'metadata': {
        'version': 'v1.0-trainval',
    },
    'infos': [
        {
            'token': 'sample_001',
            'prev': 'sample_000',
            'next': 'sample_002',
            'scene_token': 'scene_token_001',
            'frame_idx': 10,
            'timestamp': 1532402927647951,

            'lidar_path':
                'samples/LIDAR_TOP/frame_001.pcd.bin',
            'lidar2ego_translation': [0.94, 0.0, 1.84],
            'lidar2ego_rotation': [1.0, 0.0, 0.0, 0.0],
            'ego2global_translation': [100.0, 200.0, 0.0],
            'ego2global_rotation': [0.707, 0.0, 0.0, 0.707],

            'can_bus': np.array([
                # 18个自车状态量
            ]),

            'cams': {
                'CAM_FRONT': {
                    'data_path':
                        'samples/CAM_FRONT/frame_001.jpg',
                    'type': 'CAM_FRONT',
                    'sample_data_token': 'cam_front_001',
                    'timestamp': 1532402927612460,
                    'sensor2ego_translation': [...],
                    'sensor2ego_rotation': [...],
                    'ego2global_translation': [...],
                    'ego2global_rotation': [...],
                    'sensor2lidar_rotation': np.array([
                        [...], [...], [...],
                    ]),
                    'sensor2lidar_translation': np.array([...]),
                    'cam_intrinsic': np.array([
                        [1266.4, 0.0, 816.2],
                        [0.0, 1266.4, 491.5],
                        [0.0, 0.0, 1.0],
                    ]),
                },
                # 其余五个相机
            },

            'sweeps': [
                {
                    'data_path':
                        'sweeps/LIDAR_TOP/history_001.pcd.bin',
                    'type': 'lidar',
                    'sample_data_token': 'sweep_001',
                    'timestamp': 1532402927590000,
                    'sensor2ego_translation': [...],
                    'sensor2ego_rotation': [...],
                    'ego2global_translation': [...],
                    'ego2global_rotation': [...],
                    'sensor2lidar_rotation': np.array([...]),
                    'sensor2lidar_translation': np.array([...]),
                },
            ],

            'gt_boxes': np.array([
                [12.0, 3.0, 0.5, 1.8, 4.2, 1.6, 0.3],
                [20.0, -4.0, 0.8, 0.6, 0.8, 1.7, -1.2],
            ]),
            'gt_names': np.array([
                'car',
                'pedestrian',
            ]),
            'gt_velocity': np.array([
                [6.2, 0.1],
                [0.5, -0.2],
            ]),
            'num_lidar_pts': np.array([35, 4]),
            'num_radar_pts': np.array([2, 0]),
            'valid_flag': np.array([True, True]),
        },
    ],
}
```

示例数值仅用于说明结构，不对应某个真实 nuScenes sample。

---


### 26. 哪些内容已经转换，哪些只保存参数

| 数据 | PKL 中的状态 |
|-|-|
| 当前 LiDAR 点云 | 不存点本体；路径指向的原始点已经在当前 LiDAR 坐标系 |
| 历史 sweep 点云 | 不转换点本体；保存历史 LiDAR -> 当前 LiDAR 参数 |
| 相机图片 | 不转换像素；保存图片路径、内参和 camera -> current LiDAR 参数 |
| `gt_boxes` | 已转换到当前 LIDAR_TOP，并调整 yaw 约定 |
| `gt_velocity` | 已将 global 速度分量旋转到当前 LiDAR 坐标轴 |
| `lidar2ego_*` | 保存原始标定关系 |
| `ego2global_*` | 保存当前 LiDAR 时刻车辆位姿 |
| `can_bus` | 已匹配到当前 sample 附近的 CAN pose 消息 |

可以把 PKL 理解成两部分：

```text
已经规范化好的训练标注
    gt_boxes / gt_names / gt_velocity / valid_flag

加载其他原始数据所需的索引和几何关系
    paths / cams / sweeps / poses / calibration
```

---


### 27. 与 MMDetection3D v0.17.1 的精确对比

官方基线文件：

```text
mmdetection3d-v0.17.1/tools/create_data.py
mmdetection3d-v0.17.1/tools/data_converter/nuscenes_converter.py
```

BEVFormer 文件：

```text
tools/create_data.py
tools/data_converter/nuscenes_converter.py
```

对比命令：

```bash
diff -u \
  mmdetection3d-v0.17.1/tools/create_data.py \
  tools/create_data.py

diff -u \
  mmdetection3d-v0.17.1/tools/data_converter/nuscenes_converter.py \
  tools/data_converter/nuscenes_converter.py
```

同版本对比非常重要。若直接拿 MMDetection3D v1.4.0 对比，会混入 MMEngine、统一 v2 PKL schema、4x4 矩阵格式等大量框架升级差异，那些不是 BEVFormer 作者相对其依赖版本做的修改。

---


### 28. 修改脚本一：tools/create_data.py


#### 28.1 增加 CAN bus 参数

新增命令行参数：

```python
parser.add_argument(
    '--canbus',
    type=str,
    default='./data',
    help='specify the root path of nuScenes canbus',
)
```

`nuscenes_data_prep()` 新增：

```python
can_bus_root_path
```

调用时继续传给 converter：

```python
nuscenes_converter.create_nuscenes_infos(
    root_path,
    out_dir,
    can_bus_root_path,
    info_prefix,
    version=version,
    max_sweeps=max_sweeps,
)
```

修改原因：官方 converter 不读取 CAN bus，而 BEVFormer 的时序处理需要自车运动信息。


#### 28.2 输出文件名增加 temporal

官方：

```text
{prefix}_infos_train.pkl
{prefix}_infos_val.pkl
{prefix}_infos_test.pkl
```

BEVFormer：

```text
{prefix}_infos_temporal_train.pkl
{prefix}_infos_temporal_val.pkl
{prefix}_infos_temporal_test.pkl
```

修改原因：明确区分含时序/CAN bus 字段的自定义 PKL，避免与官方标准 PKL 混用或覆盖。


#### 28.3 使用 out_dir

官方 v0.17.1 的 nuScenes converter 主要把 PKL 写入 `root_path`。BEVFormer 显式把 `out_dir` 传入 converter 并以它作为输出目录。

修改原因：允许原始数据位置和生成索引位置分离，也使 `--out-dir` 的语义更直接。


#### 28.4 关闭 ground-truth database

官方代码：

```python
create_groundtruth_database(...)
```

BEVFormer 中：

```python
## create_groundtruth_database(...)
```

GT database 会裁剪每个 3D 框内部的点云，供点云模型执行 database sampling。BEVFormer 是纯视觉模型，不读取点云，也不使用点云对象粘贴增强，因此关闭这一流程可以避免不必要的时间和磁盘开销。


#### 28.5 调整 import 路径

官方：

```python
from tools.data_converter import nuscenes_converter
```

BEVFormer：

```python
from data_converter import nuscenes_converter
import sys
sys.path.append('.')
```

这是脚本启动路径和项目组织方式的调整，不改变数据内容或数学逻辑。

---


### 29. 修改脚本二：tools/data_converter/nuscenes_converter.py


#### 29.1 扩展函数签名

官方：

```python
create_nuscenes_infos(
    root_path,
    info_prefix,
    version='v1.0-trainval',
    max_sweeps=10,
)
```

BEVFormer：

```python
create_nuscenes_infos(
    root_path,
    out_path,
    can_bus_root_path,
    info_prefix,
    version='v1.0-trainval',
    max_sweeps=10,
)
```

新增：

```text
out_path
can_bus_root_path
```


#### 29.2 初始化 NuScenesCanBus

新增：

```python
from nuscenes.can_bus.can_bus_api import NuScenesCanBus
nusc_can_bus = NuScenesCanBus(dataroot=can_bus_root_path)
```

并将 `nusc_can_bus` 传给 `_fill_trainval_infos()`。


#### 29.3 新增 `_get_can_bus_info()`

官方 v0.17.1 没有这个函数。BEVFormer 新增它来完成：

```text
sample -> scene name
scene name -> CAN pose消息列表
sample timestamp -> 最近历史CAN消息
CAN消息 -> 18维数组
```


#### 29.4 每个 info 增加五个字段

新增：

```python
'prev': sample['prev'],
'next': sample['next'],
'can_bus': can_bus,
'frame_idx': frame_idx,
'scene_token': sample['scene_token'],
```

作用：

| 字段 | 修改目的 |
|-|-|
| `prev` | 保留关键帧前向关系 |
| `next` | 保留关键帧后向关系 |
| `can_bus` | 提供自车运动状态 |
| `frame_idx` | 提供 scene 内顺序 |
| `scene_token` | 防止跨 scene 组成时序序列 |


#### 29.5 增加 frame_idx 维护

新增：

```python
frame_idx = 0
```

以及：

```python
if sample['next'] == '':
    frame_idx = 0
else:
    frame_idx += 1
```


#### 29.6 修改输出目录和文件名

从：

```python
osp.join(root_path, f'{prefix}_infos_train.pkl')
```

改为：

```python
osp.join(
    out_path,
    f'{prefix}_infos_temporal_train.pkl',
)
```

val/test 同理。

---


### 30. 明确没有修改的代码

相对于官方 MMDetection3D v0.17.1，以下逻辑保持一致：


#### 30.1 没有修改相机和 sweep 坐标变换

```python
obtain_sensor2top(...)
```

核心 R/T 计算没有变化。


#### 30.2 没有修改 GT 框格式

以下代码与官方 v0.17.1 一致：

```python
gt_boxes = np.concatenate(
    [locs, dims, -rots - np.pi / 2],
    axis=1,
)
```

因此 [w,l,h] 和 -yaw-pi/2 不是 BEVFormer 额外发明的格式，而是它依赖的 MMDetection3D 版本本身采用的约定。


#### 30.3 没有修改速度变换

以下 global -> current LiDAR 速度旋转与官方一致：

```python
velo = (
    velo
    @ np.linalg.inv(e2g_r_mat).T
    @ np.linalg.inv(l2e_r_mat).T
)
```


#### 30.4 没有修改 sweeps

历史 LIDAR_TOP 的查找、数量限制和 sensor2lidar 生成与官方一致。


#### 30.5 没有修改类别和有效性判断

```python
NuScenesDataset.NameMapping
```

以及：

```python
num_lidar_pts + num_radar_pts > 0
```

都来自官方数据转换逻辑。


#### 30.6 没有新增 JSON 投影算法

`export_2d_annotation()`、`get_2d_boxes()` 和 `generate_record()` 在官方 MMDetection3D v0.17.1 已经存在。BEVFormer 只是让它们读取带 temporal 文件名的 PKL。

---


### 31. 修改内容总表

| 脚本 | 修改内容 | 原因 |
|-|-|-|
| tools/create_data.py | 增加 --canbus | 指定 CAN bus 扩展数据路径 |
| tools/create_data.py | 传递 can_bus_root_path | 让 converter 读取 CAN bus |
| tools/create_data.py | 传递 out_dir | 控制 PKL 输出位置 |
| tools/create_data.py | 文件名增加 temporal | 区分自定义时序 PKL |
| tools/create_data.py | 注释 GT database | 纯视觉模型不做点云 database sampling |
| tools/create_data.py | 调整 import | 适配项目脚本组织方式 |
| nuscenes_converter.py | 初始化 NuScenesCanBus | 读取自车运动状态 |
| nuscenes_converter.py | 新增 _get_can_bus_info() | 对齐 sample 与 CAN pose 消息 |
| nuscenes_converter.py | 新增 can_bus | 保存自车状态 |
| nuscenes_converter.py | 新增 prev/next | 保存关键帧连接关系 |
| nuscenes_converter.py | 新增 scene_token | 标识场景边界 |
| nuscenes_converter.py | 新增 frame_idx | 保存场景内帧顺序 |
| nuscenes_converter.py | 修改输出目录和名称 | 生成 temporal PKL |

---


### 32. 如何检查生成的 PKL

可以使用下面的只读脚本检查结构：

```python
import mmcv

pkl_path = (
    'data/nuscenes/'
    'nuscenes_infos_temporal_train.pkl'
)

data = mmcv.load(pkl_path)

print(data.keys())
print(data['metadata'])
print('sample count:', len(data['infos']))

info = data['infos'][0]
print('info keys:', info.keys())
print('token:', info['token'])
print('scene_token:', info['scene_token'])
print('frame_idx:', info['frame_idx'])
print('camera names:', info['cams'].keys())
print('sweep count:', len(info['sweeps']))
print('gt_boxes shape:', info['gt_boxes'].shape)
print('gt_velocity shape:', info['gt_velocity'].shape)

front = info['cams']['CAM_FRONT']
print('front image:', front['data_path'])
print('cam intrinsic:', front['cam_intrinsic'])
print('camera to lidar R:', front['sensor2lidar_rotation'])
print('camera to lidar t:', front['sensor2lidar_translation'])
```

还可以检查时序连续性：

```python
infos = data['infos']

for previous, current in zip(infos, infos[1:]):
    if previous['scene_token'] == current['scene_token']:
        assert previous['next'] == current['token']
        assert current['prev'] == previous['token']
        assert current['frame_idx'] == previous['frame_idx'] + 1
```

需要注意：最后这段检查依赖 PKL 中相邻元素确实按 scene 和时间排序，与当前 converter 的生成顺序一致。

---


### 33. 常见误解


#### 33.1 “PKL 中保存了点云”

错误。PKL 只保存：

```python
lidar_path
sweeps[*]['data_path']
```

真正点云在 `.pcd.bin` 文件中。


#### 33.2 “相机图片已经转换到 LIDAR_TOP”

错误。图片仍是原始二维像素。PKL 保存的是相机与当前 LiDAR 的几何关系。


#### 33.3 “所有字段最后都处于 LIDAR_TOP”

错误。更准确地说，当前 LIDAR_TOP 是统一三维参考系：

```text
gt_boxes 已在 LIDAR_TOP
gt_velocity 使用 LIDAR_TOP 轴表达
sensor2lidar 描述如何到 LIDAR_TOP
图片仍在像素坐标系
global/ego 位姿关系仍被保留
```


#### 33.4 “sweeps 是毫米波雷达”

在当前 `info['sweeps']` 中不是。这里沿 `LIDAR_TOP.sample_data.prev` 收集的是历史 LiDAR 点云。nuScenes 本身有 RADAR sweep，但当前列表没有收集它们。


#### 33.5 “gt_velocity 是相对自车速度”

错误。它是目标对地绝对速度，只是用当前 LiDAR 坐标轴表达分量，没有减去自车速度。


#### 33.6 “BEVFormer 自己重写了坐标转换”

错误。与官方 MMDetection3D v0.17.1 对比，核心坐标转换完全一致。BEVFormer 增加的是 CAN bus 和时序字段。


#### 33.7 “可以直接使用 MMDetection3D 1.4.0 的 PKL”

不可以直接混用。MMDetection3D 1.4.0 使用新版统一 PKL schema，例如：

```text
metainfo / data_list / images / lidar_points / instances
```

当前 BEVFormer 按 v0.17.1 的：

```text
metadata / infos / cams / gt_boxes
```

读取。两者字段结构和 Dataset API 不兼容。

---


### 34. 完整流程总结

1. 读取命令行参数
   `root_path / out_dir / version / max_sweeps / canbus`
2. 初始化官方SDK
   `NuScenes + NuScenesCanBus`
3. 获取官方scene划分
   `train / val / test`
4. 检查scene的基础LiDAR文件是否存在
5. 遍历每个关键帧sample
6. 找到当前LIDAR_TOP
   `lidar_path`
   `lidar2ego`
   `ego2global`
7. 读取当前sample的时序关系
   `token / prev / next / scene_token / frame_idx / timestamp`
8. 按sample时间匹配CAN pose消息
   `can_bus[18]`
9. 遍历六个相机
   图片路径
   相机内参
   camera -> ego -> global -> current ego -> current LIDAR
10. 沿LIDAR_TOP的sample_data.prev收集历史sweeps
    历史点云路径
    历史LiDAR -> current LIDAR变换
11. 读取当前帧3D标注
    SDK把global box转到current LIDAR
    调整yaw为SECOND/MMDet3D v0.17.1约定
12. 映射类别名称
13. 估算目标global绝对速度
    只旋转速度分量到current LIDAR坐标轴
14. 保存LiDAR/RADAR点数和valid_flag
15. 按scene_token放入train或val列表
16. 使用mmcv.dump写出PKL
    metadata + infos

最终得到的 PKL 是一份以关键帧为单位、以当前 LIDAR_TOP 为统一三维参考系、同时包含六相机几何关系和时序自车信息的数据索引。


## 详细实现手册：B. Dataset、时序队列、pipeline、Sampler 与 DataLoader

以下单元是从 `dataset.md` 整理进 notebook 的完整细节快照。前面的教程单元负责建立主线；本节保留推导、边界条件、张量形状与源码定位，便于离线顺序阅读。


## BEVFormer Dataset 完整解析：从 PKL 到模型输入

本文只讨论当前仓库中经典 BEVFormer 配置使用的数据集链路，重点是：

- 配置文件如何构造 CustomNuScenesDataset；
- 它从 PyTorch、MMDetection3D 父类继承了什么；
- 一个 Dataset 的标准职责是什么，哪些部分可定制；
- nuScenes PKL 如何被加载、解析并经过 pipeline；
- BEVFormer 如何把单帧样本组织成时序队列；
- `__getitem__()` 和 DataLoader 最终分别返回什么；
- 每一项数据大致供模型的哪个部分使用。

本文对应的主要源码版本和文件如下：

- BEVFormer Dataset：`projects/mmdet3d_plugin/datasets/nuscenes_dataset.py`
- BEVFormer 配置：`projects/configs/bevformer/bevformer_base.py`
- BEVFormer 自定义 pipeline：`projects/mmdet3d_plugin/datasets/pipelines/transform_3d.py`
- MMDetection3D v0.17.1 父类：`mmdetection3d-v0.17.1/mmdet3d/datasets/nuscenes_dataset.py`
- MMDetection3D v0.17.1 祖先类：`mmdetection3d-v0.17.1/mmdet3d/datasets/custom_3d.py`

本文描述的是 `bevformer_base.py` 当前配置的实际行为。其他 BEVFormerV2、单目、地图分割或自行修改过的配置，返回字段可能不同。

---


### 1. 先建立整体认识


#### 1.1 Dataset 的位置

数据转换脚本已经把 nuScenes 原始数据库整理成了 PKL，但 PKL 还不是神经网络可以直接计算的 Tensor。Dataset 位于两者中间：

```text
nuScenes 原始数据
  │
  │ 数据转换脚本，只需离线执行一次
  ▼
nuscenes_infos_temporal_train.pkl
  │
  │ Dataset 按 index 查一条 info
  ▼
文件路径、标注、相机矩阵、时序信息组成的 Python dict
  │
  │ pipeline 读取图片、增强、过滤、归一化、格式化
  ▼
Dataset 的一个训练样本：4 帧 × 6 相机 + 当前帧 GT + 元信息
  │
  │ DataLoader 采样、并行读取、collate
  ▼
一个 batch
  │
  ▼
BEVFormer.forward_train(...)
```

Dataset 的本质不是保存数据，而是定义：给定一个整数索引 idx，怎样返回模型训练或测试需要的一个样本。


#### 1.2 Dataset、pipeline、DataLoader 的边界

三个概念不要混在一起：

| 组件 | 主要职责 | 当前项目中的例子 |
|-|-|-|
| Dataset | 管理样本列表，解释一条 PKL info，决定训练/测试取样逻辑 | `CustomNuScenesDataset` |
| pipeline | 对一条帧数据顺序执行读取、增强、过滤、格式化 | `LoadMultiViewImageFromFiles` 等 |
| DataLoader | 决定索引顺序、多进程读取、组成 batch | `build_dataloader()` + sampler + mmcv.collate |

Dataset 的 `__getitem__()` 返回“一个样本”；DataLoader 返回“一个 batch”。因此两处看到的维度不同。

---


### 2. 继承链：哪些代码不是写在当前类里的

当前类的完整继承链是：

```text
torch.utils.data.Dataset
  └── mmdet3d.datasets.Custom3DDataset
        └── mmdet3d.datasets.NuScenesDataset
              └── projects...CustomNuScenesDataset
```


#### 2.1 torch.utils.data.Dataset

PyTorch 最基础的 map-style Dataset 约定是实现：

```python
class MyDataset(torch.utils.data.Dataset):
    def __len__(self):
        return 样本数量

    def __getitem__(self, idx):
        return 第_idx_个样本
```

PyTorch 不关心样本来自图片、数据库还是 PKL，也不规定返回字典中的键。它只要求 DataLoader 能用整数索引取样。


#### 2.2 Custom3DDataset 提供的通用 3D Dataset 骨架

Custom3DDataset 是 MMDetection3D 的通用父类，它提供了：

- 保存 `data_root`、`ann_file`、`test_mode` 等公共属性；
- 根据 `box_type_3d` 建立三维框类型和坐标模式；
- 解析类别列表，生成 `cat2id`；
- 调用 `load_annotations()` 建立 `self.data_infos`；
- 用 `Compose(pipeline)` 把配置中的变换列表构造成可调用流水线；
- 实现 `__len__()`；
- 实现默认的训练、测试取样流程；
- 实现空标注样本的随机重取；
- 建立供 GroupSampler 使用的 flag；
- 提供评测、可视化所需的一些通用辅助函数。

因此，当前 BEVFormer 类中看不到 `__len__()` 并不代表没有。继承得到的实现是：

```python
def __len__(self):
    return len(self.data_infos)
```


#### 2.3 NuScenesDataset 提供的 nuScenes 语义

NuScenesDataset 在通用骨架上补充：

- nuScenes 的 10 个检测类别；
- nuScenes 原类别名与检测类别的映射；
- PKL 顶层 `infos`、`metadata` 的加载规则；
- 按时间戳排序和 `load_interval` 抽帧；
- 从 `gt_boxes`、`gt_names`、`gt_velocity` 构造训练标注；
- `LiDARInstance3DBoxes` 封装；
- 预测结果到 nuScenes JSON 的格式化与官方指标评测。


#### 2.4 CustomNuScenesDataset 增加的 BEVFormer 语义

当前项目主要覆盖或新增了：

| 方法/参数 | 作用 |
|-|-|
| `queue_length` | 训练时一个样本包含多少帧 |
| `prepare_train_data()` | 从单帧训练改为历史帧加当前帧的队列 |
| `union2one()` | 堆叠多帧图片，组织多帧 meta，计算相邻帧位姿增量 |
| `get_data_info()` | 增加场景、前后帧、CAN bus、相机内外参等字段 |
| `__getitem__()` | 与父类逻辑接近，为时序训练返回合法队列 |
| `_evaluate_single()` | 使用项目自定义 nuScenes evaluator，支持 overlap test |

这里采用的是“继承父类并覆盖少量差异”的方式，不需要复制完整的 MMDetection3D Dataset。

---


### 3. 配置如何变成 Dataset 对象

配置中的关键部分可以简化为：

```python
dataset_type = 'CustomNuScenesDataset'
data_root = 'data/nuscenes/'

data = dict(
    samples_per_gpu=1,
    workers_per_gpu=4,
    train=dict(
        type=dataset_type,
        data_root=data_root,
        ann_file=data_root + 'nuscenes_infos_temporal_train.pkl',
        pipeline=train_pipeline,
        classes=class_names,
        modality=input_modality,
        test_mode=False,
        use_valid_flag=True,
        bev_size=(200, 200),
        queue_length=4,
        box_type_3d='LiDAR'))
```


#### 3.1 Registry 构造

CustomNuScenesDataset 上有装饰器：

```python
@DATASETS.register_module()
class CustomNuScenesDataset(NuScenesDataset):
    ...
```

导入 `projects.mmdet3d_plugin` 后，这个类被登记进 MMDetection 的 DATASETS Registry。构建器看到：

```python
dict(type='CustomNuScenesDataset', ...)
```

会完成近似如下操作：

```python
dataset_cls = DATASETS.get('CustomNuScenesDataset')
dataset = dataset_cls(**其余配置参数)
```

所以配置里的 `type` 是注册名，不是 Python 文件路径。


#### 3.2 构造器的真实调用顺序

调用链为：

```text
CustomNuScenesDataset.__init__
  └── NuScenesDataset.__init__
        └── Custom3DDataset.__init__
              ├── get_box_type()
              ├── get_classes()
              ├── self.load_annotations(...)
              ├── Compose(pipeline)
              └── _set_group_flag()
```

这里有一个重要的 Python 多态行为：虽然调用代码写在 `Custom3DDataset.__init__()` 中，但 `self` 实际是 `CustomNuScenesDataset` 对象，因此：

```python
self.load_annotations(self.ann_file)
```

最终调用的是继承链中更具体的 `NuScenesDataset.load_annotations()`，不是 Custom3DDataset 的简单实现。

---


### 4. 所有构造参数详解


#### 4.1 BEVFormer 子类直接声明的参数

##### `queue_length=4`

训练时一个 Dataset 样本最终包含的帧数。当前值为 4，即 3 个历史帧和 1 个当前帧。

**注意**：它不是严格取 index-3, index-2, index-1, index。当前实现会在当前帧之前的 4 个索引中随机丢弃 1 个，再保留 3 个。

##### `bev_size=(200, 200)`

保存为 `self.bev_size`。它表达 BEV 网格高宽，但当前 CustomNuScenesDataset 文件没有继续使用这个属性。真正决定模型 BEV query 数量的是模型配置中的 `bev_h`、`bev_w`。

它在这里更像为其他代码版本或扩展预留的 Dataset 属性，不能仅靠修改它来改变网络 BEV 尺寸。

##### `overlap_test=False`

保存为 `self.overlap_test`，只在 `_evaluate_single()` 中传给自定义 `NuScenesEval_custom`。它不改变图片加载、训练样本或前向输入。

##### `*args, **kwargs`

接收其余参数并原样交给 `NuScenesDataset.__init__()`。这使子类不必重复声明父类全部参数。


#### 4.2 NuScenesDataset 参数

##### `ann_file`

annotation file，即“标注/索引描述文件”的路径。当前是：

```text
data/nuscenes/nuscenes_infos_temporal_train.pkl
```

它不只是 GT 标签。它同时保存每个关键帧的：

- token 和时间戳；
- 六相机图片路径；
- LiDAR 路径和 sweeps；
- 相机标定；
- 自车位姿和 CAN bus；
- 三维框、类别、速度、有效性等训练标注。

所以更准确的理解是“Dataset 的总索引和元数据文件”。

##### `pipeline`

一个由若干配置字典组成的有序列表。初始化时被转换为 Compose：

```python
self.pipeline = Compose(pipeline)
```

运行时近似执行：

```python
results = transform_1(results)
results = transform_2(results)
...
```

任何一步返回 None，整条 pipeline 就会停止并返回 None，训练 Dataset 随后会重新取样。

##### `data_root`

数据集根目录，当前为 `data/nuscenes/`。部分数据集会用它和相对路径拼接；当前 BEVFormer `get_data_info()` 直接使用 PKL 中记录的 `lidar_path` 和 `cams[*].data_path`，因此这些路径本身必须在当前运行环境中可解析。

##### `classes`

最终参与训练和评测的类别集合。可以是：

- None：使用 `NuScenesDataset.CLASSES`；
- list/tuple：直接覆盖默认类别；
- 字符串：把它当作类别文本文件路径，每行一个类别。

当前配置的顺序是：

```text
0 car
1 truck
2 construction_vehicle
3 bus
4 trailer
5 barrier
6 motorcycle
7 bicycle
8 pedestrian
9 traffic_cone
```

类别顺序决定 `gt_labels_3d` 的整数编号，必须与模型 `num_classes=10` 的语义保持一致。

##### `load_interval=1`

加载 PKL 后按固定间隔下采样：

```python
data_infos = data_infos[::load_interval]
```

1 表示保留全部关键帧，2 表示每隔一条保留一条。时序模型一般不要随意修改，因为它会改变相邻索引的时间间隔，也会与 `queue_length` 共同影响历史序列。

##### `with_velocity=True`

是否把 `gt_velocity=[vx, vy]` 拼接到 3D GT 框后面。开启时每个框的底层 tensor 通常由 7 维变成 9 维：

```text
[x, y, z, w, l, h, yaw, vx, vy]
```

这里的速度分量已经由数据转换阶段表达在当前关键帧 LIDAR_TOP 坐标轴方向上。

##### `modality`

描述启用哪些传感器。当前配置：

```python
dict(
    use_lidar=False,
    use_camera=True,
    use_radar=False,
    use_map=False,
    use_external=True)
```

对本 Dataset 最关键的是 `use_camera=True`，它使 `get_data_info()` 构造六相机图片路径和 `lidar2img`。`use_lidar=False` 加上 pipeline 中没有点云 loader，意味着模型不会读取点云。

`use_external=True` 是模态声明；当前 Dataset 仍直接构造 CAN bus 字段。它本身不是“自动加载一份外部 Tensor”的开关。

##### `box_type_3d='LiDAR'`

指定三维框采用 LiDAR 坐标模式。当前 GT 和预测目标均以当前关键帧 LIDAR_TOP 为基准，而不是某一相机坐标系。

父类通过 `get_box_type()` 得到：

```text
self.box_type_3d  -> LiDARInstance3DBoxes 类
self.box_mode_3d  -> Box3DMode.LIDAR
```

这些对象也会进入 pipeline 的 meta，便于后续变换正确处理框。

##### `filter_empty_gt=True`

训练时是否过滤没有有效类别 GT 的样本。如果 pipeline 后不存在任何 `gt_labels_3d != -1`，返回 None，`__getitem__()` 会随机选择另一个索引重试。

在 BEVFormer 的时序实现中，这个检查对队列中的每一帧都执行，而不仅是当前帧。因此某个历史帧没有有效 GT，也会导致整个队列重取。这是当前代码行为，不是所有时序 Dataset 的必然要求。

##### `test_mode=False`

控制训练模式和测试模式：

| 值 | `__getitem__()` 路径 | 是否构造 ann_info | 是否构造训练时序 queue |
|-|-|-|-|
| False | `prepare_train_data()` | 是 | 是 |
| True | `prepare_test_data()` | 否 | 否，Dataset 每次只返回当前帧 |

验证时，训练 API 会用 `default_args=dict(test_mode=True)` 构造验证集。配置中的 `samples_per_gpu` 是 DataLoader 参数，会在构建验证 Dataset 前被 pop 掉。

##### `eval_version='detection_cvpr_2019'`

nuScenes 检测评测规则版本，用于创建距离阈值、类别范围等官方 evaluation config。不影响训练样本内容。

##### `use_valid_flag=False`

决定最初如何过滤 PKL 中的 GT：

```python
if use_valid_flag:
    mask = info['valid_flag']
else:
    mask = info['num_lidar_pts'] > 0
```

当前配置设为 True。`valid_flag` 是转换脚本生成的布尔数组，通常表示该框内存在 LiDAR 点或 radar 点。即便 BEVFormer 只使用图片，也可用点云/radar 的可观测性来筛掉缺乏有效传感器证据的框。

---


### 5. 初始化时如何读取 PKL


#### 5.1 PKL 顶层结构

`NuScenesDataset.load_annotations()` 使用：

```python
data = mmcv.load(ann_file)
```

当前信息文件的逻辑结构是：

```python
{
    'infos': [
        frame_info_0,
        frame_info_1,
        ...
    ],
    'metadata': {
        'version': 'v1.0-trainval'
    }
}
```

加载后执行：

```python
data_infos = sorted(data['infos'], key=lambda e: e['timestamp'])
data_infos = data_infos[::self.load_interval]
self.metadata = data['metadata']
self.version = self.metadata['version']
```

最终：

```python
self.data_infos[index] == 按时间排序后的第 index 个关键帧 info
len(dataset)           == len(self.data_infos)
```

PKL 在初始化时整体反序列化到内存，但图片没有被提前读入。六张图片只会在 `__getitem__()` 被 worker 请求某个样本时从磁盘读取。


#### 5.2 排序不等于按 scene 分组采样

父类只是全局按 timestamp 排序。nuScenes 不同 scene 的时间戳通常能形成稳定顺序，但 `prepare_train_data()` 仍然只是按列表索引向前取历史帧，没有沿 prev token 逐级查找。

当索引跨越 scene 边界时，`union2one()` 会通过 `scene_token` 把新场景第一帧标记为 `prev_bev_exists=False`，模型据此丢弃之前场景的 BEV。也就是说，场景连续性主要由 scene_token 检查保护。


#### 5.3 `__len__()`、flag 和 sampler

PKL 加载完毕后，继承的 `__len__()` 直接返回：

```python
len(self.data_infos)
```

训练模式下，父类还调用 `_set_group_flag()`：

```python
self.flag = np.zeros(len(self), dtype=np.uint8)
```

2D 检测 Dataset 常按横图/竖图设置不同 flag，让同 batch 图片形状接近，减少 padding。MMDetection3D 的这个通用父类把所有样本设为同一组 0。因此当前 DistributedGroupSampler 虽然仍读取 `dataset.flag`，实际上整个训练集只有一个组。

空标注重采样使用：

```python
pool = np.where(self.flag == self.flag[idx])[0]
return np.random.choice(pool)
```

因为 flag 全部为 0，“从相同组随机取另一条”在当前项目中基本等价于从整个训练 Dataset 随机取一个索引。它可能再次抽到原索引，`while True` 会继续检查，直到拿到合法队列。

---


### 6. 一次训练取样的完整调用链

DataLoader 请求 `dataset[index]` 后执行：

```text
CustomNuScenesDataset.__getitem__(index)
  └── prepare_train_data(index)
        ├── 选择 3 个历史索引 + 当前索引
        ├── 对每帧执行：
        │     ├── get_data_info(i)
        │     ├── pre_pipeline(input_dict)
        │     └── pipeline(input_dict)
        ├── 检查每帧是否有有效 GT
        └── union2one(queue)
              ├── stack 四帧图像
              ├── 组织四帧 img_metas
              ├── 设置 prev_bev_exists
              └── 把绝对位姿改成帧间增量
```

如果任一步返回 None：

```python
idx = self._rand_another(idx)
```

然后继续循环，直到得到合法样本。

---


### 7. 时序索引是怎样选择的

当前实现：

```python
index_list = list(range(index - self.queue_length, index))
random.shuffle(index_list)
index_list = sorted(index_list[1:])
index_list.append(index)
```

假设：

```python
index = 100
queue_length = 4
```

先产生：

```text
[96, 97, 98, 99]
```

随机打乱后丢掉第一个元素，也就是随机丢掉其中一帧；剩余三帧重新按时间排序，再加当前帧 100。例如：

```text
[96, 98, 99, 100]
```

这种随机间隔是一种简单的时序采样扰动，使模型不只适应完全固定的帧间隔。


#### 7.1 数据集开头的处理

每个历史索引都会执行：

```python
i = max(0, i)
```

因此靠近整个数据集开头时，负索引不会像 Python list 那样从末尾取值，而是统一变成第 0 帧。一个队列中可能重复出现第 0 帧。


#### 7.2 场景边界的处理

索引选择本身不会禁止跨 scene。真正合并时：

- 如果当前 meta 的 `scene_token` 与上一条不同：`prev_bev_exists=False`；
- 如果相同：`prev_bev_exists=True`。

模型处理到 False 时会重置历史 BEV，因此不会把前一个场景的内容真正融合进新场景。

---


### 8. `get_data_info()`：把一条 PKL info 解释成 pipeline 输入


#### 8.1 首先复制/引用的基础字段

```python
input_dict = dict(
    sample_idx=info['token'],
    pts_filename=info['lidar_path'],
    sweeps=info['sweeps'],
    ego2global_translation=info['ego2global_translation'],
    ego2global_rotation=info['ego2global_rotation'],
    prev_idx=info['prev'],
    next_idx=info['next'],
    scene_token=info['scene_token'],
    can_bus=info['can_bus'],
    frame_idx=info['frame_idx'],
    timestamp=info['timestamp'] / 1e6,
)
```

其中时间戳从微秒转换为秒。

`pts_filename` 和 `sweeps` 被保留是因为类继承自通用 nuScenes 3D Dataset，也方便切换到点云 pipeline。但当前纯视觉 pipeline 没有 `LoadPointsFromFile` 或 `LoadPointsFromMultiSweeps`，所以点云文件不会被读取，也不会形成模型输入 Tensor。


#### 8.2 六相机信息

当 `modality['use_camera']` 为真时，遍历：

```python
for cam_type, cam_info in info['cams'].items():
```

对每个相机收集：

- `data_path` → 图片路径；
- `sensor2lidar_rotation`、`sensor2lidar_translation` → camera 到当前 LIDAR_TOP 的外参；
- `cam_intrinsic` → camera 到像素平面的内参。

最终增加：

```python
input_dict.update(dict(
    img_filename=image_paths,
    lidar2img=lidar2img_rts,
    cam_intrinsic=cam_intrinsics,
    lidar2cam=lidar2cam_rts,
))
```

六相机的顺序取决于 `info['cams']` 字典的插入顺序。数据转换与后续处理必须始终保持同一顺序，模型并不会从图片内容自动识别“这是前相机还是后相机”。


#### 8.3 sensor2lidar 为什么要取逆

PKL 保存的是：

```text
camera sensor 坐标 → 当前关键帧 LIDAR_TOP 坐标
```

但 BEVFormer 从一个 LiDAR/BEV 三维参考点出发，需要知道它投影到哪个相机像素，因此需要反方向：

```text
当前关键帧 LIDAR_TOP 坐标 → camera 坐标 → image 像素
```

对列向量写法，设 camera 到 LiDAR 的刚体变换为：

$$p_L = R_{CL}\, p_C + t_{CL}$$

逆变换为：

$$p_C = R_{CL}^{T}\,(p_L - t_{CL}) = R_{LC}\, p_L + t_{LC}$$

$$R_{LC} = R_{CL}^{T}, \qquad t_{LC} = -R_{CL}^{T}\, t_{CL}$$

再通过相机内参 K 投影：

$$\begin{bmatrix} \tilde u \\ \tilde v \\ d \end{bmatrix} = K \begin{bmatrix} x_C \\ y_C \\ z_C \end{bmatrix}, \qquad u = \frac{\tilde u}{d}, \quad v = \frac{\tilde v}{d}$$

齐次形式可写作：

$$\tilde p_{img,h} = K_{pad} \cdot T_{lidar \to camera} \cdot \tilde p_{lidar,h}$$

代码中的 `lidar2img` 就是这个组合矩阵。刚体外参本身可逆；相机的透视投影从 3D 压到 2D 后丢失深度，单凭 (u,v) 一般不可逆。BEVFormer 投影的是自己建立的 BEV 三维参考点，不是为了把 GT 框画到图片上。

##### 代码逐行对应（converter 内部使用行向量写法）

上面公式用列向量便于推导，但 `get_data_info()` 的真实代码按 nuScenes 的行向量约定组装矩阵，实际用法是：

$$p_L^{T} = p_C^{T} R_{CL}^{T} + t_{CL}^{T}$$

等价于代码中的 `points_lidar = points_cam @ R.T + t`。

逐行对应关系：

| 代码 | 数学含义 |
|---|---|
| `lidar2cam_r = np.linalg.inv(R_CL)` | $R_{LC} = R_{CL}^{-1} = R_{CL}^T$，形状 (3,3) |
| `lidar2cam_t = t_CL @ lidar2cam_r.T` | 算出的是 $-t_{LC}$（行向量约定下），还差一个负号 |
| `lidar2cam_rt[:3,:3] = lidar2cam_r.T` | 4×4 左上块放 $R_{LC}^T$（行向量必须右乘 $R^T$） |
| `lidar2cam_rt[3,:3] = -lidar2cam_t` | 第 4 行放真正的平移 $t_{LC}$（把负号补回来） |

此时 `lidar2cam_rt` 是行向量约定的齐次矩阵（记 $\tilde p = [x,y,z,1]$，旋转在左上、平移在第 4 行）：

$$\tilde p_C^{\,T} = \tilde p_L^{\,T} \begin{bmatrix} R_{LC}^{T} & \mathbf{0} \\ t_{LC} & 1 \end{bmatrix}$$

接下来两次转置各有一次明确用途：

- `lidar2cam_rt.T` 把平移从第 4 行“翻”回第 4 列，得到标准列向量齐次矩阵 $T_{lidar \to camera}$（即最终 meta 里的 `lidar2cam`，6 个 4×4）；
- 相机内参 `cam_intrinsic` (3,3) 填入 4×4 单位阵左上角得到 `viewpad`（即 $K_{pad}$）。

最终组合：

$$\text{lidar2img} = K_{pad} \cdot T_{lidar \to camera} \in \mathbb{R}^{4 \times 4}$$

每帧共 6 个。一步完成整条链：

$$ (x,y,z)_{lidar} \;\xrightarrow[\text{刚体，可逆}]{\;R_{LC},\,t_{LC}\;}\; (x,y,z)_{cam} \;\xrightarrow[\text{透视，丢深度}]{\;K\;}\; (u,v)$$

##### 往返验证（数值实测）

对任一 LiDAR 系点（如 `[10, 3, -1]`）：

- 先用 `lidar2cam_rt`（行向量用法）变到相机系，再用 PKL 原始的 sensor2lidar 变换变回来，往返误差为 0 —— 证明取逆组装正确；
- `lidar2img @ [x,y,z,1]^T` 齐次结果除以第三行得到的 (u,v)，与“先手动求 $p_C$ 再算 $K p_C$ 归一化”的结果完全一致；
- 齐次结果的第三行恰好等于相机系深度 $d = z_{cam}$ —— 后面所有可见性判断都靠它。

##### 模型侧怎样消费这个 4×4

`BEVFormerEncoder.point_sampling` 先反归一化（把 [0,1] 的 BEV 参考点乘回 `pc_range` 的 ±51.2m 真实坐标），再按列向量约定矩阵乘：

```python
reference_points_cam = torch.matmul(lidar2img, reference_points)   # (4,4) @ (4,1)
```

齐次结果前三行为 $(u \cdot d,\ v \cdot d,\ d)$，随后两步生成 `bev_mask`：

- $d > \epsilon$：点是否在相机**前方**；
- 除以 `d` 得 (u,v) 后判断是否落在 $[0,W) \times [0,H)$：是否在**画面内**。

两者共同决定每个 BEV query 在哪些相机、哪些像素附近做 SCA 采样。

##### 为什么“可逆”与“不可逆”同时成立

刚体部分（lidar 与 cam 之间）是严格可逆的欧氏变换；投影部分（cam 到像素）是 3D 压 2D 的降维，单凭像素无法恢复深度，不可逆。BEVFormer 在这条链上只使用 3D→2D 方向——已知自建的 BEV 三维参考点，去图像特征上找采样位置——所以信息损失不构成问题。


#### 8.4 `cam_intrinsic` 为什么最后看不到

`get_data_info()` 的确产生了 `cam_intrinsic`，但当前 `CustomCollect3D.meta_keys` 不包含它，所以单帧 pipeline 最终收集时它会被丢弃。

这不影响当前模型，因为内参已经乘进 `lidar2img`。若自定义模型需要单独使用内参，必须把 `cam_intrinsic` 加入 `CustomCollect3D` 的 meta_keys。


#### 8.5 CAN bus 的整理

接下来用 PKL 的 ego pose 覆盖 CAN bus 中与位姿相关的部分：

```python
rotation = Quaternion(input_dict['ego2global_rotation'])
translation = input_dict['ego2global_translation']
can_bus[:3] = translation
can_bus[3:7] = rotation
```

并计算自车 yaw：

```python
patch_angle = quaternion_yaw(rotation) / np.pi * 180
if patch_angle < 0:
    patch_angle += 360
can_bus[-2] = patch_angle / 180 * np.pi  # 弧度
can_bus[-1] = patch_angle                # 角度
```

此时仍是当前帧的全局绝对位置和绝对 yaw。到 `union2one()` 时才改成相邻入选帧之间的差值。

---


### 9. `get_ann_info()`：从 PKL 构造监督信号

这个方法没有在 BEVFormer 子类中重写，而是继承 NuScenesDataset。


#### 9.1 初始过滤

当前 `use_valid_flag=True`：

```python
mask = info['valid_flag']
gt_bboxes_3d = info['gt_boxes'][mask]
gt_names_3d = info['gt_names'][mask]
```

若设为 False，则使用：

```python
mask = info['num_lidar_pts'] > 0
```


#### 9.2 类别名转整数标签

```python
for category in gt_names_3d:
    if category in self.CLASSES:
        label = self.CLASSES.index(category)
    else:
        label = -1
```

-1 表示不属于当前训练类别，后面的 ObjectNameFilter 会移除它。


#### 9.3 拼接速度

当 `with_velocity=True`：

```python
gt_velocity = info['gt_velocity'][mask]
gt_velocity[np.isnan(gt_velocity[:, 0])] = [0.0, 0.0]
gt_bboxes_3d = np.concatenate([gt_bboxes_3d, gt_velocity], axis=-1)
```

因此每个框通常为 9 维。无法估计的 NaN 速度会被置为零。


#### 9.4 封装为三维框对象

```python
gt_bboxes_3d = LiDARInstance3DBoxes(
    gt_bboxes_3d,
    box_dim=gt_bboxes_3d.shape[-1],
    origin=(0.5, 0.5, 0.5)
).convert_to(self.box_mode_3d)
```

`LiDARInstance3DBoxes` 不只是一个 ndarray。它记录框的坐标语义，并提供：

- 按 mask 索引框；
- 判断中心是否位于 BEV 范围；
- 规范化 yaw；
- 坐标模式转换；
- 访问中心、尺寸、角点等。

`origin=(0.5, 0.5, 0.5)` 告诉框类：传入的 nuScenes 框坐标以几何中心为原点描述。类内部可以按 MMDetection3D 的统一约定处理。

最终返回：

```python
ann_info = {
    'gt_bboxes_3d': LiDARInstance3DBoxes,  # M 个 9 维框
    'gt_labels_3d': np.ndarray,            # shape [M]
    'gt_names': np.ndarray/list,           # M 个类别名
}
```

---


### 10. `pre_pipeline()` 做什么

在正式 pipeline 前，继承的 `pre_pipeline()` 增加一些约定字段：

```python
results['img_fields'] = []
results['bbox3d_fields'] = []
results['pts_mask_fields'] = []
results['pts_seg_fields'] = []
results['bbox_fields'] = []
results['mask_fields'] = []
results['seg_fields'] = []
results['box_type_3d'] = self.box_type_3d
results['box_mode_3d'] = self.box_mode_3d
```

这些空列表不是模型输入，而是 pipeline 各变换之间的协议。例如 LoadAnnotations3D 会把 `gt_bboxes_3d` 登记到 `bbox3d_fields`，后续几何增强就知道哪些三维框必须与数据同步变换。

可以把 `pre_pipeline()` 理解为给流水线建立工作区和类型上下文。

---


### 11. 当前训练 pipeline 逐步详解

当前顺序是：

1. LoadMultiViewImageFromFiles
2. PhotoMetricDistortionMultiViewImage
3. LoadAnnotations3D
4. ObjectRangeFilter
5. ObjectNameFilter
6. NormalizeMultiviewImage
7. PadMultiViewImage
8. DefaultFormatBundle3D
9. CustomCollect3D

顺序很重要。比如必须先读取图片才能做颜色增强，必须先加载标注才能过滤框，必须在最终收集前完成 Tensor 格式化。


#### 11.1 LoadMultiViewImageFromFiles(to_float32=True)

输入核心字段：

```python
img_filename = [六个图片路径]
```

内部先把六张图读成：

```text
[H, W, C, N]
```

再拆成 list：

```text
img = [image_0, ..., image_5]
每张为 [H, W, C]
```

`to_float32=True` 是后续光度增强的要求。它还建立 `filename`、`img_shape`、`ori_shape`、`pad_shape`、`scale_factor` 和初始 `img_norm_cfg`。


#### 11.2 PhotoMetricDistortionMultiViewImage

对每个相机图像独立进行随机光度增强，候选操作包括：

- 亮度扰动；
- 对比度扰动；
- 饱和度扰动；
- 色相扰动；
- 通道随机交换。

它只改像素值，不改图像尺寸和几何位置，所以无需修改 `lidar2img` 或 GT 3D 框。

值得注意的是，各相机使用独立随机抽样，因此同一时刻六张图的颜色扰动不一定相同。


#### 11.3 LoadAnnotations3D

配置为：

```python
dict(
    type='LoadAnnotations3D',
    with_bbox_3d=True,
    with_label_3d=True,
    with_attr_label=False)
```

它主要把嵌套在 `ann_info` 中的字段提升到 pipeline 顶层：

```text
ann_info['gt_bboxes_3d'] → results['gt_bboxes_3d']
ann_info['gt_labels_3d'] → results['gt_labels_3d']
```

并把 `gt_bboxes_3d` 登记进 `bbox3d_fields`。它没有重新从 JSON 读取一次标注。


#### 11.4 ObjectRangeFilter

当前检测范围：

```text
x ∈ [-51.2, 51.2]
y ∈ [-51.2, 51.2]
z 配置范围为 [-5.0, 3.0]
```

对 `LiDARInstance3DBoxes`，这一步实际用 BEV 的 `[x_min, y_min, x_max, y_max]` 判断框中心，移除平面范围外的框，并同步过滤标签。之后把 yaw 规范到 [-π, π] 附近的等价周期。

注意它不是裁剪六张图片，也不是把框投影到图片后判断可见性。


#### 11.5 ObjectNameFilter

只保留类别编号位于当前 classes 范围内的框。此前映射成 -1 的未知类别会在这里移除。


#### 11.6 NormalizeMultiviewImage

当前参数：

```text
mean = [103.530, 116.280, 123.675]
std = [1.0, 1.0, 1.0]
to_rgb = False
```

对每张图执行：

```python
normalized = (image - mean) / std
```

`to_rgb=False` 表示维持 OpenCV/MMCV 读取的 BGR 通道顺序，这与使用 Caffe 风格预训练的 ResNet 配置相匹配。


#### 11.7 PadMultiViewImage(size_divisor=32)

把每张图右侧/底部补零，使高宽都能被 32 整除。nuScenes 常见原图约为：

```text
[900, 1600, 3]
```

padding 后通常为：

```text
[928, 1600, 3]
```

因为 928 是不小于 900 的最近 32 倍数，而 1600 本身可被 32 整除。固定整除尺寸便于 CNN 多次下采样以及同 batch 堆叠。

这里没有缩放图像，因此 `lidar2img` 不需要乘 resize 矩阵。padding 只发生在右侧和底部，原图像素坐标原点也不移动。


#### 11.8 DefaultFormatBundle3D

这是 MMDetection3D v0.17.1 的多视图格式化器。对图片执行：

```text
list 中每张 [H, W, C]
→ transpose
每张 [C, H, W]
→ stack 六张
[N, C, H, W]
→ torch.Tensor
→ DataContainer(stack=True)
```

同时：

- `gt_labels_3d`：NumPy 数组转 Tensor，再包为 DataContainer；
- `gt_bboxes_3d`：保留 `LiDARInstance3DBoxes` 对象，包为 `DataContainer(cpu_only=True)`。

框的数量 M 每帧不同，不能像固定尺寸图片一样简单堆成一个规则 Tensor。


#### 11.9 CustomCollect3D

训练配置只明确收集：

```python
keys=['gt_bboxes_3d', 'gt_labels_3d', 'img']
```

此外，它总会创建：

```python
data['img_metas'] = DataContainer(img_metas, cpu_only=True)
```

默认允许进入 `img_metas` 的字段包括：

```text
filename, ori_shape, img_shape, lidar2img, lidar2cam,
depth2img, cam2img, pad_shape, scale_factor, flip,
pcd_horizontal_flip, pcd_vertical_flip,
box_mode_3d, box_type_3d, img_norm_cfg, pcd_trans,
sample_idx, prev_idx, next_idx, pcd_scale_factor, pcd_rotation,
pts_filename, transformation_3d_flow, scene_token, can_bus
```

只有同时存在于 results 的键才会真正加入。

一个容易忽略的结论是：`frame_idx`、`timestamp`、`ego2global_translation`、`ego2global_rotation`、`sweeps`、`cam_intrinsic` 不在默认 meta_keys 中，因此虽然 `get_data_info()` 曾经构造它们，最终返回模型前会被丢弃。需要它们时应定制 meta_keys。

---


### 12. `union2one()`：四个单帧结果如何合成一个样本

单帧 pipeline 结束后，queue 是长度为 4 的 list：

```python
queue = [frame_0, frame_1, frame_2, current_frame]
```

每项大致为：

```python
{
    'img': DataContainer(Tensor[N, C, H, W], stack=True),
    'img_metas': DataContainer(dict, cpu_only=True),
    'gt_bboxes_3d': DataContainer(LiDARInstance3DBoxes, cpu_only=True),
    'gt_labels_3d': DataContainer(Tensor[M]),
}
```


#### 12.1 堆叠图片

```python
imgs_list = [each['img'].data for each in queue]
torch.stack(imgs_list)
```

形状变化：

```text
4 个 [N, C, H, W]
→ [T, N, C, H, W]
→ 当前配置 [4, 6, 3, 928, 1600]
```

然后重新包装为：

```python
DataContainer(stacked_images, stack=True, cpu_only=False)
```


#### 12.2 组织多帧元信息

每帧的 meta 被组织成：

```python
metas_map = {
    0: frame_0_meta,
    1: frame_1_meta,
    2: frame_2_meta,
    3: current_frame_meta,
}
```

注意这里是整数键字典，不是 list。模型随后用 `each[i]` 访问第 i 帧。


#### 12.3 prev_bev_exists

逐帧比较 `scene_token`：

```text
新 scene 的第一条入选帧 → False
与上一条属于同一 scene   → True
```

它告诉模型当前帧能否使用前一帧累计得到的 BEV。这个标志不是说“PKL 中是否有 prev token”，而是说“当前组成的训练 queue 中，前一个 BEV 是否属于同一 scene”。


#### 12.4 CAN bus 从绝对量改为相对量

对新 scene 的第一帧：

```text
can_bus[:3] = [0, 0, 0]
can_bus[-1] = 0°
```

同一 scene 的后续帧：

```text
Δposition_i = global_position_i - global_position_previous_selected
Δyaw_i      = global_yaw_i - global_yaw_previous_selected
```

这里的 previous 指“queue 中上一条被选中的帧”，不一定是数据集严格相邻关键帧，因为历史选择随机丢掉了一帧。

这些差值供模型：

- 根据自车平移移动历史 BEV；
- 根据 yaw 旋转历史 BEV；
- 通过 CAN bus MLP 向 BEV query 注入运动信息。

当前实现只对 `can_bus[:3]` 和 `can_bus[-1]` 做帧间差分；`can_bus[-2]` 仍保留该帧的绝对 yaw 弧度。模型的平移计算使用 `[:2]` 与 `[-2]`，BEV 旋转使用 `[-1]`，这是代码中两种 yaw 表达的具体分工。


#### 12.5 为什么只返回 `queue[-1]`

合并后以当前帧字典为容器：

```python
queue[-1]['img'] = 四帧图片
queue[-1]['img_metas'] = 四帧元信息
return queue[-1]
```

结果是：

- 历史帧只保留图片和 meta；
- 当前帧保留图片、meta、GT 框和 GT 标签；
- 历史帧 GT 虽执行过 pipeline 和有效性检查，最终不会交给 loss。

这是合理的监督方式：前三帧用于无梯度地建立历史 BEV，最后一帧才计算检测损失。

> **说明**：以下第 13～16 节的原稿在拷贝过程中丢失，现依据当前仓库代码补写，其余章节均为原文。

---


### 13. DataLoader 侧：从一个样本到一个 batch

前面 6～12 节描述的是 `Dataset.__getitem__(idx)` 如何产生一个时序样本。本节继续追踪这个样本经过 **sampler → BatchSampler → DataLoader worker → collate → 并行封装 → model.train_step** 的全过程。

这部分容易混淆，是因为“一个 batch”在不同层有不同表示：

```text
Dataset 单样本
  dict，img 为 DC(Tensor[T, N, C, H, W])
        │
        │ sampler 只产出整数 index
        ▼
DataLoader worker 对 batch 内每个 index 调 Dataset[index]
  list[dict]，长度 = 本进程 batch_size
        │
        │ mmcv.parallel.collate
        ▼
Runner 接到的 data_batch
  dict，值仍是 DataContainer；其 data 按 GPU 分块
        │
        │ MMDataParallel / MMDistributedDataParallel 的 scatter
        ▼
model.train_step / BEVFormer.forward_train
  img 是 GPU Tensor，meta 和 3D boxes 是 Python list
```

下面以经典 `bevformer_base.py` 的分布式训练设置为主（通常 `samples_per_gpu=1`），同时说明多卡和 `samples_per_gpu>1` 时的行为。


#### 13.1 调用入口：`build_dataloader` 实际传入了什么

`custom_train_detector()` 中的代码是：

```python
data_loaders = [
    build_dataloader(
        ds,
        cfg.data.samples_per_gpu,
        cfg.data.workers_per_gpu,
        len(cfg.gpu_ids),
        dist=distributed,
        seed=cfg.seed,
        shuffler_sampler=cfg.data.shuffler_sampler,
        nonshuffler_sampler=cfg.data.nonshuffler_sampler)
    for ds in dataset
]
```

实际实现位于 `projects/mmdet3d_plugin/datasets/builder.py`。当 `dist=True` 时，每个 GPU 对应一个独立进程，也对应一个独立 DataLoader：

```python
DataLoader(
    dataset,
    batch_size=samples_per_gpu,
    sampler=sampler,
    num_workers=workers_per_gpu,
    collate_fn=partial(collate, samples_per_gpu=samples_per_gpu),
    pin_memory=False,
    worker_init_fn=init_fn,
    persistent_workers=(num_workers > 0))
```

因此几个配置项的准确含义是：

| 配置/参数 | 分布式训练 `dist=True` | 含义 |
|-|-|-|
| `samples_per_gpu` | 当前 rank 的 `batch_size` | 一张 GPU 一次前向处理多少个 Dataset 样本 |
| `workers_per_gpu` | 当前 rank 的 `num_workers` | 该 GPU 对应 DataLoader 启动多少个子进程读图、做 pipeline |
| `len(cfg.gpu_ids)` | 在分布式分支中忽略 | world size 来自 `torch.distributed` |
| `seed` | 传给 sampler 和 worker 初始化函数 | 控制 index shuffle 与 NumPy/Python 随机增强 |
| `shuffler_sampler` | 默认 `DistributedGroupSampler` | 训练的 index 产生器 |
| `nonshuffler_sampler` | 默认项目 `DistributedSampler` | 验证/测试的 index 产生器 |

在非分布式分支中，代码将 `batch_size` 设为 `num_gpus * samples_per_gpu`，并使用 MMDetection 的 `GroupSampler` 或不传 sampler。源码明确打印“Only can be used for obtain inference speed”，所以这不是项目主要支持的训练路径。


#### 13.2 四个角色的边界：Dataset、Sampler、BatchSampler、DataLoader

把它们严格区分开：

| 组件 | 输入 | 输出 | 是否读取图片/PKL |
|-|-|-|-
| Dataset | 一个整数 `idx` | 一个样本 dict | 是；调用 `prepare_train_data` 和 pipeline |
| Sampler | Dataset 的长度/flag | 一串整数 index | 否 |
| BatchSampler | 一串 index + batch size | 一串 `list[index]` | 否 |
| DataLoader | 每个 batch 的 index list | collate 后的 batch dict | 是；调度 worker 调用 Dataset |

Sampler 不调用 `Dataset.__getitem__()`，也不产生 Tensor；它只决定“哪个样本先取、哪个 rank 取”。图像读取、随机光度增强、时序 queue 合并仍由 worker 调用 Dataset 完成。


#### 13.3 Sampler：训练时怎样划分和随机化索引

训练 DataLoader 默认使用项目的 `DistributedGroupSampler`。Dataset 初始化时，MMDetection3D 的 `Custom3DDataset._set_group_flag()` 创建：

```python
dataset.flag = np.zeros(len(dataset), dtype=np.uint8)
```

对于常规二维检测，group flag 通常按图片横纵比把样本分组，让同一 batch 的图像尺寸相近，减少 padding。当前 3D Dataset 的实现明确把全部样本设为组 `0`；而且 BEVFormer pipeline 已经把六张图 pad 到 32 的倍数。因此在本项目中，GroupSampler 的“按形状分组”功能没有实际区分样本，主要留下了分布式 shuffle、补齐和按卡切分功能。

令：

```text
L = len(dataset)
R = world_size（进程/GPU 数）
S = samples_per_gpu
```

`DistributedGroupSampler.__iter__()` 的行为为：

1. 用随机数发生器种子 `seed + epoch` 打乱组 0 的全部 index；
2. 把 index 尾部重复若干项，补齐到 `S × R` 的整数倍；
3. 将补齐后的 index 列表按每 `S` 个组成一个小 batch；
4. 打乱这些“小 batch”的先后顺序，但不打散 batch 内的 `S` 个 index；
5. rank `r` 取连续片段 `indices[r * num_samples : (r + 1) * num_samples]`。

其中每个 rank 的样本数是：

```text
num_samples = ceil(L / (S × R)) × S
total_size  = num_samples × R
```

所以每个 rank 恰好取得 `num_samples / S` 个完整 batch；没有训练末尾的“较小 batch”。当 `L` 不能整除 `S × R` 时，少量样本在同一 epoch 中会重复一次。这是为了使所有 DDP 进程执行相同数量的 iteration，避免某个 rank 提前结束而同步死锁。

每个 epoch 必须改变 sampler 的 `epoch`，否则随机顺序永远相同。训练代码在分布式 `EpochBasedRunner` 上注册了 MMCV 的 `DistSamplerSeedHook`，它会调用 sampler 的 `set_epoch(epoch)`；项目的 `DistributedGroupSampler.set_epoch()` 随后使第 1 步使用新的 `seed + epoch`。


#### 13.4 验证/测试 sampler：为什么 `shuffle=False`，以及它的边界

验证时，`custom_train_detector` 调用同一个 builder，但传入 `shuffle=False`。因此使用项目的 `DistributedSampler`，其逻辑是：

1. 生成升序 index `[0, 1, ..., L-1]`；
2. 为让每个 rank 样本数相等，在尾部重复开头的少量 index，直到总数可被 `R` 整除；
3. 每个 rank 取得一个**连续区间**，不是 PyTorch 默认 `DistributedSampler` 常见的隔行切片。

例如 `L=10, R=3` 时，补齐后总长度为 12：

```text
完整 index：  [0,1,2,3,4,5,6,7,8,9,0,1]
rank 0：      [0,1,2,3]
rank 1：      [4,5,6,7]
rank 2：      [8,9,0,1]
```

这保证了每个 rank 内部的原始时间顺序不被打乱。`CustomNuScenesDataset.data_infos` 已按 timestamp 排序，因此同一 rank 内 `BEVFormer.forward_test` 的 `prev_bev` 缓存可以按顺序推进。

但应准确理解这个保证：多卡的区间边界仍可能落在一个 scene 中间。一个 rank 的第一帧没有前一 rank 的 `prev_bev`，因此该边界帧会以空历史 BEV 开始；另外尾部补齐的重复样本可能在结果收集时需要去重。这是分布式时序推理的固有限制，不应表述为“全数据集的时序缓存绝不会断”。项目选择连续区间，而非交错 index，已经尽可能保留每个 rank 内的局部时序。


#### 13.5 BatchSampler：虽然代码没有写，它仍然存在

当前 `DataLoader(...)` 没有显式传入 `batch_sampler`，只传了：

```python
batch_size=batch_size
sampler=sampler
```

PyTorch DataLoader 会在内部创建等价于下面的对象：

```python
batch_sampler = BatchSampler(
    sampler=sampler,
    batch_size=batch_size,
    drop_last=False)  # DataLoader 默认值
```

其职责非常简单：从 sampler 连续取 `batch_size` 个 index，组合为一组 list。例如 sampler 给出：

```text
[7, 3, 9, 2, 6, 1]
```

且 `batch_size=2`，则 BatchSampler 依次给 DataLoader：

```text
[7, 3]  →  [9, 2]  →  [6, 1]
```

DataLoader 再让 worker 分别执行 `dataset[7]`、`dataset[3]` 等。它**不关心**返回的是图像、3D box 还是 `DataContainer`。

训练 sampler 已保证每个 rank 的 index 数量是 `S` 的倍数，因此 `drop_last=False` 在训练中不会产生不完整 batch。验证 sampler 只保证长度能被 rank 数整除，不保证能被 `S` 整除；当验证 `samples_per_gpu > 1` 时，最后一个 local batch 可能较小。当前 BEVFormer val 配置通常为每卡 1 个样本，因此不存在这个差别。


#### 13.6 worker：谁真正执行 `Dataset.__getitem__`

当 `num_workers=4`，主进程不亲自读取全部样本，而是维护 4 个 worker 子进程。它们从 DataLoader 的 index 队列取任务，执行：

```text
idx
  → CustomNuScenesDataset.__getitem__(idx)
  → prepare_train_data(idx)
  → 对时序 queue 中每一帧执行 get_data_info、pre_pipeline、pipeline
  → union2one(queue)
  → 一个样本 dict
```

worker 会预取后续 batch，因此日志中“某张图片被读取”的时间不一定恰好对应当前正在 GPU 上计算的 iteration。`persistent_workers=True`（只要 `num_workers>0`）意味着 epoch 结束后 worker 不销毁，下个 epoch 继续复用；好处是避免反复创建进程，代价是 worker 内对象会长期存在。

若训练入口设置了 `--seed`，worker 初始化函数使用：

```python
worker_seed = num_workers * rank + worker_id + seed
np.random.seed(worker_seed)
random.seed(worker_seed)
```

这让不同 rank、不同 worker 的 NumPy/Python 随机流不同，避免所有 worker 对不同样本做完全相同的随机增强。注意当前函数没有设置 `torch.manual_seed`；而 Dataset 的历史帧随机丢弃使用的是 Python `random.shuffle`，图像缩放用的是 NumPy，因此它们受此初始化控制。


#### 13.7 为什么 PyTorch 默认 collate 不够用

PyTorch 默认 `default_collate` 假定同一字段可规整地堆叠：

```python
torch.stack([sample_0['img'], sample_1['img']])
```

它适合固定大小的图像 Tensor，却无法表达当前项目的两类字段：

1. 每个样本的 GT 数量 `M` 不同。`gt_labels_3d` 是 `[M_i]`，强行 `torch.stack` 会失败或需要人为 padding；`gt_bboxes_3d` 还是带几何方法的 `LiDARInstance3DBoxes` Python 对象，不能直接堆叠。
2. `img_metas` 是字典，其中含文件名、相机矩阵、场景 token、CAN bus 等异构元信息。它们必须保持每个样本独立，且通常不应整体搬到 GPU。

此外，旧版 MMCV 的 `MMDataParallel` 需要知道“一个全局 batch 中哪些样本应送往哪张 GPU”。普通 list/Tensor 无法同时表达这些字段的 batch 规则和每卡分块规则。

`DataContainer` 正是给每个字段添加这份运输说明的轻量包装。它不转换数据内容，主要保存：

| 属性 | 含义 |
|-|-|
| `data` | 实际数据 |
| `stack` | 同一 GPU 上的样本是否应堆叠为 Tensor |
| `cpu_only` | 是否保持 Python/CPU 对象，不参与 GPU scatter |
| `padding_value`、`pad_dims` | 当同一 batch 尺寸不同而仍要 stack 时，如何补齐尾部维度 |


#### 13.8 MMCV `collate`：按 DataContainer 规则打包

DataLoader 的 `collate_fn` 是：

```python
partial(mmcv.parallel.collate, samples_per_gpu=samples_per_gpu)
```

它接收一个 Python `list`，其中每项是一个 Dataset 样本 dict，然后按 dict 的每个 key 分别递归处理。对于 `DataContainer`，行为可以概括为：

```text
cpu_only=True
  不 stack；按每卡切成 list，并把 list 重新包进 DataContainer

cpu_only=False, stack=True
  对同一卡的样本 Tensor 做 stack；需要时按 pad_dims 补齐；
  每张卡得到一个 Tensor，再整体包进 DataContainer

cpu_only=False, stack=False
  不 stack；每卡保留 list[原始数据]，再整体包进 DataContainer
```

这里的“每卡切分”很重要。设非分布式训练有两张卡，`samples_per_gpu=2`，全局 DataLoader batch 是 4 个样本；collate 后的 `DataContainer.data` 长度为 2：

```text
data[0]  对应 GPU 0 的两个样本
data[1]  对应 GPU 1 的两个样本
```

分布式训练是一进程一张卡，DataLoader 的 local batch 正好是 `samples_per_gpu`，所以 `DataContainer.data` 通常只有一个元素 `data[0]`。这就是本项目代码常写 `data_batch['img'].data[0]` 的原因；它还不是最终传给 `forward_train` 的裸 Tensor。

当前训练样本的四个主要字段如下：

| 字段 | 在 pipeline / `union2one` 中的包装 | collate 后每卡的内容 | 为什么这样做 |
|-|-|-|-|
| `img` | `DC(Tensor[T,N,C,H,W], stack=True)` | Tensor `[B_local,T,N,C,H,W]` | 同一配置下图像已 pad 到相同尺寸，可高效堆叠并送 GPU |
| `gt_labels_3d` | `DC(Tensor[M_i])`，默认 `stack=False` | `list[Tensor[M_i]]` | 每张图的目标数不同，无需 padding |
| `gt_bboxes_3d` | `DC(LiDARInstance3DBoxes, cpu_only=True)` | `list[LiDARInstance3DBoxes]` | 框对象含几何方法，检测头按样本逐个使用 |
| `img_metas` | `DC(metas_map, cpu_only=True)` | `list[metas_map]` | meta 是异构字典，且只被 Python 控制流/几何逻辑读取 |

以 `B_local=1, T=4, N=6` 为例，Runner 接到的中间对象更准确地写作：

```text
data_batch['img']           = DC(data=[Tensor[1, 4, 6, 3, 928, 1600]], stack=True)
data_batch['img_metas']     = DC(data=[[{0: meta0, 1: meta1, 2: meta2, 3: meta3}]], cpu_only=True)
data_batch['gt_bboxes_3d']  = DC(data=[[LiDARInstance3DBoxes]], cpu_only=True)
data_batch['gt_labels_3d']  = DC(data=[[Tensor[M]]], stack=False)
```

两个方括号层次不可省略：外层表示“GPU 分块”，内层才表示“该卡的 B 个样本”。`samples_per_gpu=1` 时它们看上去都只有一个元素，但语义不同。


#### 13.9 从 DataContainer 到 `forward_train`

普通训练配置的 runner 是 `EpochBasedRunner`。它取到 `data_batch` 后，调用并行模型包装器的 `train_step`。MMCV 的 `MMDataParallel` 或 `MMDistributedDataParallel` 会根据 `DataContainer` 做 scatter：

- 对 `stack=True, cpu_only=False` 的 `img`：取对应 GPU 的 Tensor，并搬到该 GPU；
- 对 `stack=False, cpu_only=False` 的 `gt_labels_3d`：保留该 GPU 的 Tensor list，再将其中 Tensor 放到对应 GPU；
- 对 `cpu_only=True` 的 `img_metas`、`gt_bboxes_3d`：只取对应 GPU 的 Python list，不调用 `.cuda()`；
- 并行包装器调用内部检测器的 `train_step(data, optimizer)`；检测器基类再以 `self(**data)` 解包这些关键字参数，进入 `forward_train`。

因此真正到达 `BEVFormer.forward_train` 的实参为：

```text
points        -> None（训练 collect 的 keys 中没有它）
img           -> Tensor [B_local, 4, 6, 3, 928, 1600]
img_metas     -> list，长度 B_local；每项是 {0:meta, 1:meta, 2:meta, 3:meta}
gt_bboxes_3d  -> list，长度 B_local；每项是 LiDARInstance3DBoxes
gt_labels_3d  -> list，长度 B_local；每项是 Tensor[M_i]
```

注意 `img` 的 batch 维只在 collate 处加入；Dataset 单样本的形状始终是 `[T,N,C,H,W]`。模型侧表达式：

```python
[each[len_queue - 1] for each in img_metas]
```

表示“对 batch 中每个样本，取其第 `T-1` 个（当前帧）meta”。


#### 13.10 一个 iteration 的可执行心智模型

假设 8 卡 DDP，当前 rank=2，`samples_per_gpu=1`、`workers_per_gpu=4`，训练 sampler 在这一 epoch 给 rank 2 的 index 序列为 `[217, 91, ...]`。第一个 iteration 的过程是：

```text
DistributedGroupSampler(rank=2)
  输出 217
      │
DataLoader 内部 BatchSampler(batch_size=1)
  输出 [217]
      │
一个 worker
  调用 dataset[217]
  读取并处理 4 帧 × 6 相机，返回 queue[-1]
      │
mmcv.collate(samples_per_gpu=1)
  img: [4,6,3,H,W] → [1,4,6,3,H,W]，并包为 DC(data=[...])
  meta/GT: 保持按样本组织的 list，并包为 DC
      │
MMDistributedDataParallel
  img Tensor 放到 rank 2 当前 GPU
  meta/boxes 留为 CPU Python 对象；labels Tensor 放到当前 GPU
      │
BEVFormer.forward_train
  历史 3 帧先建立 prev_bev，当前帧计算检测 loss
```

这也解释了内存和吞吐的来源：一条“样本”不是一张图，而是训练时 4 个时刻 × 每时刻 6 张相机图。`samples_per_gpu` 增加 1，近似多增加一整条时序 queue 的图像和中间特征，而不是只多一张图片。


#### 13.11 调试 DataLoader 时应看哪一层

推荐用下面的顺序定位问题：

```python
## 1. Dataset：不经过 sampler/collate，确认单样本
sample = dataset[0]
print(sample['img'].data.shape)  # [T, N, C, H, W]

## 2. Sampler：确认 index 顺序和当前 rank 的分片
print(list(iter(data_loader.sampler))[:16])

## 3. DataLoader：确认 collate 后的 DataContainer
batch = next(iter(data_loader))
print(batch['img'].data[0].shape)  # [B_local, T, N, C, H, W]
print(len(batch['img_metas'].data[0]))  # B_local

## 4. 模型：在 forward_train 入口临时打印 img.shape、len(img_metas)
```

不要在多 worker、分布式训练进程中随意 `print` 每条样本：输出会交错且严重拖慢读取。优先用单卡、`workers_per_gpu=0` 的小型调试配置确认 Dataset 与 collate，再恢复原有设置。

---


### 14. 模型怎样消费这些字段：forward_train 的数据流


#### 14.1 帧分离

`forward_train`（`projects/mmdet3d_plugin/bevformer/detectors/bevformer.py`）不把队列整体送入网络，先做切分：

```python
len_queue = img.size(1)      # 4
prev_img = img[:, :-1, ...]  # 前 3 帧，历史
img = img[:, -1, ...]        # 当前帧
```


#### 14.2 obtain_history_bev：无梯度迭代建立历史 BEV

关键行为：

- `self.eval()` + `torch.no_grad()`：BatchNorm 统计不更新，不构建反向图；
- 历史帧先 reshape 成 `[B*(T-1), N, C, H, W]`，一次性过 backbone+neck（`extract_img_feat` 按 `len_queue` 把特征重新分组回 `[B, T-1, N, C, H, W]`）；
- 逐历史帧循环：`img_metas = [each[i] for each in img_metas_list]`；
- 该帧 `prev_bev_exists` 为 False 时，先把 `prev_bev` 置 None；
- 调用 `self.pts_bbox_head(img_feats, img_metas, prev_bev, only_bev=True)`，只执行 BEV encoder（TSA+SCA）得到该帧的 BEV 特征，作为下一轮循环的 `prev_bev`；
- 结束前恢复 `self.train()`。

也就是说：参与梯度的网络前向只有当前帧一次，历史 BEV 链条是一条纯推理缓存。


#### 14.3 当前帧走完整检测流程

- 取 `img_metas = each[len_queue-1]`（当前帧的 meta），若当前帧 `prev_bev_exists` 为 False，同样丢弃历史 BEV；
- 当前帧特征送入 `forward_pts_train` → `pts_bbox_head(feats, metas, prev_bev)`：BEV encoder 中 TSA 融合 `prev_bev`，SCA 聚合图像特征，DETR decoder 输出预测；
- `gt_bboxes_3d`、`gt_labels_3d` 只在 assigner 匹配和 loss 计算中被消费。


#### 14.4 每个字段的具体去处

| 字段 | 模型侧使用位置 |
|-|-|
| `img` | `extract_img_feat` → backbone/neck → SCA 的 key/value |
| `lidar2img` | `BEVFormerEncoder.point_sampling`：BEV 网格的 3D 参考点投影到各相机，生成 `reference_points_cam` 和 `bev_mask` |
| `can_bus[:2]` 与 `can_bus[-2]` | `get_bev_features` 换算成 BEV 网格平移量 shift，供 TSA 对历史帧参考点做位移对齐 |
| `can_bus[-1]` | 有 `prev_bev` 时按 yaw 差旋转历史 BEV（`torchvision rotate`） |
| `can_bus` 完整 18 维 | 经 `can_bus_mlp` 加到 BEV queries 上，注入自车运动信息 |
| `scene_token` | 主要由 Dataset 合队列时和 `forward_test` 使用；训练前向依赖的是已生成的 `prev_bev_exists` |
| `prev_bev_exists` | `obtain_history_bev` 循环内与当前帧前向前，决定是否清空 `prev_bev` |
| `img_shape` / `ori_shape` / `pad_shape` | 投影几何计算与结果后处理使用 |

---


### 15. 验证与测试时的数据形态：单帧和 track 模式

前面几节都面向训练。val/test 的 Dataset 虽然同为 CustomNuScenesDataset 对象，路径却完全不同。


#### 15.1 配置差异

`data.val` 和 `data.test` 均使用 `test_pipeline`：

```python
test_pipeline = [
    dict(type='LoadMultiViewImageFromFiles', to_float32=True),
    dict(type='NormalizeMultiviewImage', **img_norm_cfg),
    dict(type='PadMultiViewImage', size_divisor=32),
    dict(
        type='MultiScaleFlipAug3D',
        img_scale=(1600, 900),
        pts_scale_ratio=1,
        flip=False,
        transforms=[
            dict(type='DefaultFormatBundle3D', with_label=False),
            dict(type='CustomCollect3D', keys=['img'])
        ])
]
```

对比训练 pipeline：没有光度增强，没有 LoadAnnotations3D，没有 ObjectRangeFilter/ObjectNameFilter，最终只收集 `img`（`img_metas` 仍一定会附加）。


#### 15.2 test_mode=True 改变了整个取样逻辑

- `__getitem__` 直接走 `prepare_test_data(index)`：`get_data_info` → `pre_pipeline` → `pipeline`，返回单帧结果，没有 queue，也没有 `union2one`；
- `get_data_info` 在 test_mode 下根本不构造 `ann_info`，所以全程不存在 `gt_*` 字段；
- `queue_length`、`bev_size` 在 test_mode 下不起作用；
- 训练过程中的周期性验证，是用 `default_args=dict(test_mode=True)` 另建 Dataset 对象，与 tools/test.py 行为一致。


#### 15.3 时序由模型对象跨调用维护（track 模式）

Dataset 只给单帧，时序链条靠检测器自身的缓存补齐，即 `BEVFormer.forward_test`。它在 `__init__` 中维护：

```python
self.prev_frame_info = {
    'prev_bev': None,
    'scene_token': '',
    'prev_pos': 0,
    'prev_angle': 0,
}
```

每个测试样本的处理顺序：

1. 若 `img_metas[0][0]['scene_token']` 与缓存不同，清空 `prev_bev`（场景切换重置）；
2. 先备份本帧的绝对位姿 `tmp_pos`/`tmp_angle`；
3. 若缓存中已有 `prev_bev`：`can_bus[:3] -= prev_pos`、`can_bus[-1] -= prev_angle`（就地改成帧间增量）；否则把这两段直接置 0（新场景第一帧无参考量）；
4. `simple_test` 返回 `(new_prev_bev, bbox_results)`；
5. 把 `tmp_pos`/`tmp_angle`/`new_prev_bev` 写回缓存，供下一个样本使用。

这与训练形成镜像：训练时“帧间差分”发生在 `union2one()`，测试时发生在 `forward_test` 内，语义一致——都是当前帧相对“上一条参与建立 BEV 的帧”的运动量。

另外两点：

- `video_test_mode=False` 会强制关闭时序缓存，每帧独立预测（单帧基线模式）；
- `tools/test.py` 在非分布式分支上直接 `assert False`，官方只支持 `tools/dist_test.sh` 多卡测试；配合 shuffle=False 的按时间顺序取样（13.1 节），`prev_bev` 缓存链才不会断裂。


#### 15.4 测试 batch 的数据形状

```text
img        -> list（测试增强外层），内含 Tensor [B, 1, N, C, H, W]
img_metas  -> list[list[dict]]，外层是增强维，内层是 batch 维
```

双层嵌套正是 `forward_test` 以 `img_metas[0][0]` 取值的由来；T 维为 1，因为测试样本只有一帧。

---


### 16. 评测：从预测结果到 NDS/mAP


#### 16.1 入口

训练中的周期性验证与 `tools/test.py` 最终都调用 `dataset.evaluate(results)`。继承的 `NuScenesDataset.evaluate` 做两件事：

1. `format_results`：把逐帧的 `pts_bbox` dict（`boxes_3d`/`scores_3d`/`labels_3d`）转换成 nuScenes 官方提交格式 JSON——框写回 translation、size、rotation、velocity，附 token 与 detection_name；
2. 对每个结果文件调用 `_evaluate_single`。


#### 16.2 BEVFormer 覆盖了 _evaluate_single

子类把官方 evaluator 换成项目自定义版本：

```python
self.nusc_eval = NuScenesEval_custom(
    self.nusc,
    config=self.eval_detection_configs,
    result_path=result_path,
    eval_set=eval_set_map[self.version],
    output_dir=output_dir,
    verbose=True,
    overlap_test=self.overlap_test,
    data_infos=self.data_infos
)
self.nusc_eval.main(plot_examples=0, render_curves=False)
```

注意：评测需要本地存在与 PKL 配套的 nuScenes 完整数据库（`v1.0-trainval` 元数据表），因为 GT 要通过 SDK 查询；把 `self.data_infos` 传入是为了让评测样本集与 Dataset 加载的样本对齐。


#### 16.3 NuScenesEval_custom 相对官方 DetectionEval 的差异

`projects/mmdet3d_plugin/datasets/nuscnes_eval.py` 中的改动：

- `DetectionBox_modified`：给每个框附加 index（scene 内帧序号）和 visibility 等字段；
- `add_center_dist`：预算到自车中心的距离，供距离分桶统计；
- `filter_eval_boxes_by_overlap`：当 `overlap_test=True` 时，按相机视锥体同时过滤 GT 和预测，只保留图像可见目标，用于“纯视觉可见集”消融评测；
- `update_gt`：支持按可见度等级或帧索引子集重算指标（visibility 分档评测）；
- 匹配、各类 AP、TP 误差与 NDS 的计算流程与官方协议一致。


#### 16.4 指标去向

`main()` 结束后读取 `metrics_summary.json`，展开为 detail dict：

- 逐类别逐距离 AP：如 `pts_bbox_NuScenes/car_AP_dist_0.5`；
- 逐类别 TP 误差：平移、尺度、朝向、速度等；
- 汇总指标 `pts_bbox_NuScenes/NDS` 与 `pts_bbox_NuScenes/mAP`。

detail 返回给 EvalHook，写入训练日志与 TensorBoard；`tools/test.py --eval bbox` 打印的也是同一组数值。

---


### 17. PKL 字段最终流向总表

下面以当前纯视觉 `bevformer_base.py` 为准。

| PKL 字段 | Dataset 中的处理 | 最终进入模型？ | 大致用途 |
|-|-|-|-|
| `token` | → `sample_idx` | 是，meta | 样本唯一标识、结果对应关系 |
| `lidar_path` | → `pts_filename` | 是，meta，但不读取文件 | 通用接口兼容；当前纯视觉模型不用点云 |
| `sweeps` | 放入中间 dict | 否 | 当前 pipeline 不加载历史点云 |
| `timestamp` | 微秒转秒 | 否 | 中间排序/信息；默认 collect 未收集 |
| `prev` | → `prev_idx` | 是，meta | 帧链接信息；当前训练 queue 实际按整数索引选帧 |
| `next` | → `next_idx` | 是，meta | 帧链接/调试或扩展用途 |
| `scene_token` | 原样保留 | 是，meta | 判断场景切换，重置历史 BEV |
| `frame_idx` | 放入中间 dict | 否 | 默认 collect 未收集 |
| `ego2global_translation` | 写入 `can_bus[:3]` | 间接进入 | 计算入选帧间自车平移 |
| `ego2global_rotation` | 写入 quaternion/yaw | 间接进入 | 计算自车朝向及 BEV 旋转 |
| `can_bus` | 位姿字段被整理，合队列时求差 | 是，meta | 历史 BEV 对齐、运动 embedding |
| `cams[*].data_path` | → `img_filename` → 读取像素 | 是，img | 六相机视觉输入 |
| `cams[*].sensor2lidar_rotation` | 取逆并组装矩阵 | 间接进入 | 构造 lidar2cam 和 lidar2img |
| `cams[*].sensor2lidar_translation` | 取逆并组装矩阵 | 间接进入 | 构造 lidar2cam 和 lidar2img |
| `cams[*].cam_intrinsic` | 扩展为 4×4，乘入投影矩阵 | lidar2img 进入，单独内参不进入 | 把 camera 3D 坐标投影为像素坐标 |
| `gt_boxes` | 过滤并封装框对象 | 是，仅训练当前帧 | 3D 框回归真值 |
| `gt_names` | 映射为整数类别 | 是，仅训练当前帧 | 分类真值 |
| `gt_velocity` | NaN 置零，拼到框最后 | 是，仅训练当前帧 | 速度回归真值 |
| `valid_flag` | 作为初始 GT mask | 不直接进入 | 过滤无有效传感器点的框 |
| `num_lidar_pts` | `use_valid_flag=False` 时作为 mask | 当前配置否 | 另一种 GT 有效性过滤方式 |
| `num_radar_pts` | 已用于转换阶段形成 valid flag | 不直接进入 | 有效性信息来源之一 |

“不进入模型”不等于字段无用。例如 timestamp 已经参与 `data_infos` 排序，valid_flag 已经决定哪些 GT 被保留，只是它们自身不作为 forward 参数。

---


### 18. 每个最终数据项简要做什么

##### `img`

模型真正的传感器输入。包含 4 个时刻、每个时刻 6 张相机图。ResNet/FPN 先逐相机提取多尺度特征，Spatial Cross-Attention 再利用投影矩阵把图像特征聚合到 BEV query。

##### `img_metas[*]['lidar2img']`

每帧 6 个 LiDAR/BEV 三维坐标到相机像素齐次坐标的投影矩阵。它让 BEVFormer 知道某个 BEV 三维参考点应当去哪些相机、哪些像素附近采样特征。

##### `img_metas[*]['lidar2cam']`

LiDAR 到相机坐标的外参。当前 Spatial Cross-Attention 的关键投影主要使用 `lidar2img`；单独保留它便于扩展、几何处理和调试。

##### `img_metas[*]['can_bus']`

编码自车运动。训练 queue 中主要包含相邻入选帧的平移和 yaw 差，以及其余 CAN bus 状态。模型用它移动/旋转历史 BEV并生成运动 embedding。

##### `img_metas[*]['scene_token']`

区分不同驾驶片段。场景变化时禁止沿用上一场景的历史 BEV。

##### `img_metas[*]['prev_bev_exists']`

由 Dataset 在 `union2one()` 中新增，直接告诉模型当前帧是否存在同场景的历史 BEV。

##### `img_metas[*]['img_shape/ori_shape/pad_shape/img_norm_cfg']`

描述图片读取、padding 和归一化状态，供几何计算、结果处理、调试和通用框架组件使用。

##### `gt_bboxes_3d`

只对应 queue 最后一帧。每个真值框通常含：

```text
[x, y, z, w, l, h, yaw, vx, vy]
```

位置、方向和速度均按当前关键帧 LIDAR_TOP 坐标轴表达，用于当前帧 3D 检测回归监督。

##### `gt_labels_3d`

与 `gt_bboxes_3d` 一一对应的整数类别，用于分类监督。

---


### 19. 标准流程中哪些是必须的，哪些可以定制


#### 19.1 对普通 PyTorch Dataset 真正必须的部分

若完全脱离 MMDetection3D，map-style Dataset 原则上只必须提供：

```text
__len__()
__getitem__()
```

返回 tuple 还是 dict 都可由自己决定。


#### 19.2 接入 MMDetection3D 训练框架时的标准契约

为了与 Registry、pipeline、sampler、runner 和模型接口协作，通常需要保证：

1. Dataset 类注册到 DATASETS；
2. 配置中的构造参数能被 `__init__()` 接收；
3. `load_annotations()` 能生成有稳定索引的 `data_infos`；
4. `get_data_info(index)` 能产生 pipeline 所需字段；
5. 训练时能提供模型需要的 GT；
6. `pre_pipeline()` 建立框架约定的字段列表和框类型；
7. pipeline 最终把模型参数需要的键收集出来；
8. `__len__()` 和 `__getitem__()` 行为稳定；
9. 若过滤后可能返回 None，必须有重采样策略；
10. 评测时实现对应数据集的结果格式化和 evaluation。

这些不一定全部由子类亲自实现。继承父类的意义就是复用已经满足契约的部分。


#### 19.3 最常定制的部分

##### 更换自己的 PKL 格式

通常修改：

- `load_annotations()`：解释顶层文件；
- `get_data_info()`：把每条自定义记录转换成 pipeline 统一字段；
- `get_ann_info()`：把自定义标签转换成框架 GT 对象。

如果可以控制离线转换脚本，更推荐先把自有数据转换为已有 NuScenesDataset 接近的 info 格式，从而少改运行时 Dataset。

##### 修改传感器输入

例如加入点云，需要：

- PKL 中存在正确的点云路径和标定；
- `modality.use_lidar=True`；
- pipeline 加入 LoadPointsFromFile，需要时加入 LoadPointsFromMultiSweeps；
- `CustomCollect3D.keys` 收集 points；
- 模型 forward 确实接收并使用 points。

仅修改 modality 不会自动完成其余步骤。

##### 修改数据增强

可以在 pipeline 插入或替换颜色、resize、crop、flip 等变换。但几何增强必须同步更新：

- 六张图片；
- `lidar2img` 或相关投影矩阵；
- 3D 框；
- 必要的 shape/meta。

只改图片而不改投影关系，会让 BEV reference point 投到错误像素。

##### 修改时序策略

可在 `prepare_train_data()` 定制：

- 固定连续历史帧；
- 不随机丢帧；
- 按时间间隔取帧；
- 严格限制同 scene；
- 历史帧允许空 GT；
- 增加未来帧或双向上下文。

同时要检查 `union2one()` 和模型对 T、meta 结构、`prev_bev_exists` 的假设。

##### 增加模型所需 meta

如果 `get_data_info()` 已产生字段但模型收不到，优先检查：

```python
CustomCollect3D(meta_keys=...)
```

例如需要独立相机内参时，应加入 `cam_intrinsic`；需要时间间隔时，应加入 `timestamp`。

##### 修改类别或检测范围

修改 classes 时，还要同步检查：

- 模型 head 的 `num_classes`；
- 标签映射；
- evaluation 类别；
- ObjectNameFilter。

修改 `point_cloud_range` 时，还要同步模型 encoder、bbox coder、assigner 等配置。Dataset 中的 ObjectRangeFilter 只是其中一处。

---


### 20. 当前 BEVFormer 相对标准 NuScenesDataset 到底改了什么

只看 Dataset 取样部分，可以概括为：

| 标准 NuScenesDataset | BEVFormer CustomNuScenesDataset |
|-|-|
| 训练一次返回一帧 | 训练一次返回一个 4 帧 queue |
| 基础 sample_idx/pts/sweeps/timestamp | 增加 scene、prev/next、ego pose、CAN bus、frame index |
| 相机侧主要产生 img_filename/lidar2img | 额外保留 lidar2cam/cam_intrinsic 中间字段 |
| 不组织帧间运动 | 把绝对位姿整理为相邻入选帧运动差 |
| 无 `prev_bev_exists` | 根据 `scene_token` 创建该标志 |
| 使用标准单帧训练准备 | 覆盖 `prepare_train_data()` 和 `union2one()` |
| 标准 nuScenes evaluator | 使用支持 overlap test 的自定义 evaluator |

没有改的核心部分包括：

- PKL 顶层加载和按时间排序；
- 类别管理；
- GT 框、标签、速度的基本解析；
- `LiDARInstance3DBoxes`；
- `pre_pipeline()`；
- `__len__()` 和 `_rand_another()` 的基础逻辑；
- 大部分 MMDetection3D pipeline 组件。

---


### 21. 常见误解

##### 误解一：PKL 已经是模型输入

PKL 是结构化索引和元数据。图片仍在 JPEG 文件中，Dataset pipeline 才会读取、归一化并转 Tensor。

##### 误解二：纯视觉模型不应该出现 LiDAR 字段

LIDAR_TOP 在这里还承担统一三维参考坐标系的作用。纯视觉表示“不把 LiDAR 点云作为输入”，不表示不能用 LiDAR 坐标系定义 3D 框和 BEV 空间。

##### 误解三：lidar2img 是为了把 GT 框投到图片上监督

BEVFormer 的核心用途是把 BEV query 对应的三维参考点投到六张图片，从相应位置聚合图像特征。3D GT 直接在 LiDAR 坐标系监督检测 head，不需要先产生六份 2D/3D 图片框标注。

##### 误解四：queue_length=4 就是当前帧之前连续三帧

当前实现从前四个索引随机丢一个，再加当前帧，因此存在随机时间间隔。

##### 误解五：Dataset 返回的就是 [B,T,N,C,H,W]

Dataset 单样本没有 batch 维，返回 `[T,N,C,H,W]`。B 是 DataLoader collate 后增加的。

##### 误解六：字段在 `get_data_info()` 里出现就会进入模型

不一定。最终只有 `CustomCollect3D.keys` 和 `meta_keys` 收集的内容才保留下来。pipeline 中间字典包含许多临时字段。

##### 误解七：训练和测试都由 Dataset 一次返回四帧

训练时一次返回 queue；测试时 Dataset 返回单帧，模型对象跨测试调用维护上一帧 BEV。

---


### 22. 最简调用伪代码

把所有框架细节压缩后，当前 Dataset 可以理解为：

```python
class CustomNuScenesDataset:
    def __init__(self, ann_file, pipeline, queue_length=4, ...):
        raw = mmcv.load(ann_file)
        self.data_infos = sorted(raw['infos'], key=timestamp)
        self.pipeline = Compose(pipeline)

    def __len__(self):
        return len(self.data_infos)

    def __getitem__(self, index):
        if test_mode:
            return pipeline(get_data_info(index))

        while True:
            history = 从 index 前四条中随机选三条
            selected = history + [index]
            frames = []

            for i in selected:
                results = get_data_info(max(0, i))
                results = pre_pipeline(results)
                results = pipeline(results)

                if results 无有效 GT:
                    break并随机换一个index重来

                frames.append(results)
            else:
                return union2one(frames)
```

最关键的一条主线是：

```text
PKL 的一条 info
→ get_data_info 解释语义和构造投影矩阵
→ pipeline 读取并加工一帧
→ union2one 合并时序
→ DataLoader 组成 batch
→ 模型用前三帧建历史 BEV，用最后一帧 GT 训练
```

理解这条主线后，再阅读 Dataset 源码时，就可以判断每段代码属于“索引管理、单帧解析、数据变换、时序合并、批处理”中的哪一层，而不会把所有字典字段看成一团。


## 详细实现手册：C. Python 配置、plugin 与 Registry 构建

以下单元是从 `config_and_registry.md` 整理进 notebook 的完整细节快照。前面的教程单元负责建立主线；本节保留推导、边界条件、张量形状与源码定位，便于离线顺序阅读。


## BEVFormer 的配置与注册机制：从 Python 配置到模型对象

本文解释当前仓库中 **配置（configuration）** 和 **注册表（Registry）** 的完整机制。本文以当前经典基线 `projects/configs/bevformer/bevformer_base.py` 为准，目标是回答下面几个问题：

- `projects/configs/bevformer/bevformer_base.py` 为什么能描述一个完整模型与训练任务？
- `type='BEVFormer'`、`type='BEVFormerHead'` 这样的字符串怎样找到真正的 Python 类？
- `projects/mmdet3d_plugin/` 中的自定义代码为什么会生效？
- 配置、插件导入、模型构建、Dataset 构建和训练 API 的实际调用顺序是什么？
- 新增一个模块、替换一个模块或遇到 “not in the registry” 报错时，应当怎样做？

本文针对仓库当前使用的 OpenMMLab 旧版体系：MMCV 1.x、MMDetection 2.x、MMDetection3D v0.17.1。新版 MMEngine/MMDetection3D 1.x 的 `Registry`、`default_scope` 和配置写法有明显差异，不能直接套用。

相关源码：

```text
tools/train.py                                      # 本仓库训练入口
projects/configs/bevformer/bevformer_base.py        # 当前文档采用的基线配置
projects/mmdet3d_plugin/__init__.py                 # 项目插件总入口
projects/mmdet3d_plugin/bevformer/__init__.py       # BEVFormer 子包入口
mmdetection3d-v0.17.1/mmdet3d/models/builder.py     # 模型构建器
mmdetection3d-v0.17.1/mmdet3d/datasets/builder.py   # 数据集构建器
```

三篇相关文档的分工如下：

| 文档 | 主要回答的问题 |
|---|---|
| 本文 | 配置如何加载、`type` 如何经 Registry 找到类、对象如何递归构建 |
| `docs/framework_training_pipeline.md` | DataLoader、Runner、Hook 与训练循环如何组织 |
| `docs/training_overview.md` | batch 的模型输入输出、Hungarian 匹配和 loss |

---


### 1. 全局图：配置怎样变成一次训练

从命令行启动训练时，真正发生的过程如下：

```text
python tools/train.py projects/configs/bevformer/bevformer_base.py
                         │
                         ▼
                 Config.fromfile(...)
                         │
                         ├── 执行 Python 配置文件
                         └── 合并 _base_ 中的基础配置
                         │
                         ▼
             cfg：包含 model、data、optimizer、runner ...
                         │
                         ├── 读取 custom_imports（若配置了）
                         └── 读取 plugin=True，导入 projects.mmdet3d_plugin
                                                    │
                                                    ▼
                              import 触发 @xxx.register_module()
                              自定义类进入各个共享 Registry
                         │
          ┌──────────────┼─────────────────────────┐
          ▼              ▼                         ▼
 build_model(cfg.model) build_dataset(cfg.data.train) build optimizer/runner/hooks
          │              │
          ▼              ▼
  BEVFormer 实例       CustomNuScenesDataset 实例
          │              │
          └──── custom_train_model(...) ────────────┘
                         │
                         ▼
                       训练循环
```

先记住两条原则：

1. **配置只保存描述，不保存对象。** `dict(type='BEVFormer', ...)` 只是一个普通字典；在 `build_model` 之前没有 `BEVFormer` 实例。
2. **注册发生在导入时，构建发生在之后。** 注册表必须已经包含 `type` 对应的类，构建器才能创建对象。

---


### 2. 配置机制


#### 2.1 配置文件是 Python 文件

当前训练配置使用 `.py` 文件。例如 `bevformer_base.py` 中：

```python
_dim_ = 256
bev_h_ = 200
bev_w_ = 200

model = dict(
    type='BEVFormer',
    img_neck=dict(out_channels=_dim_),
    pts_bbox_head=dict(bev_h=bev_h_, bev_w=bev_w_, ...))
```

因此配置可以定义变量、复用变量、拼接路径和使用 Python 表达式。这也是 `_dim_`、`point_cloud_range`、`class_names` 能在同一文件多次复用的原因。

配置中的顶层变量会被 MMCV 收集成 `Config`；通常以下划线开头的临时变量不会出现在最终配置中。例如 `_dim_` 用于组织配置，不是训练器要读取的运行时字段。


#### 2.2 `_base_`：继承与合并

`bevformer_base.py` 开头定义：

```python
_base_ = [
    '../datasets/custom_nus-3d.py',
    '../_base_/default_runtime.py'
]
```

加载时，`Config.fromfile()` 先读取基础文件，再将当前文件的同名字段合并或覆盖。该基础配置中：

- `custom_nus-3d.py` 提供 nuScenes 相关默认字段；
- `default_runtime.py` 提供日志、分布式后端、工作目录、checkpoint 等默认字段；
- 当前文件重新定义 `model`、`data`、`optimizer`、`runner` 等，因此这些顶层字段以当前文件为准。

合并不是所有字段都简单替换。对于两个同名 `dict`，MMCV 默认递归合并：子配置只写发生变化的键即可。若希望子配置完全替换基类中同名字典，可使用 `_delete_=True`：

```python
model = dict(
    _delete_=True,
    type='MyDetector',
    backbone=dict(...))
```

`_delete_` 是配置合并的控制标记，并不会作为构造参数传给 `MyDetector`。


#### 2.3 `Config` 的常用 API

训练入口的实际代码位于 `tools/train.py`：

```python
cfg = Config.fromfile(args.config)
if args.cfg_options is not None:
    cfg.merge_from_dict(args.cfg_options)
```

常用操作如下：

```python
from mmcv import Config

## 读取配置并完成 _base_ 合并
cfg = Config.fromfile('projects/configs/bevformer/bevformer_base.py')

## ConfigDict 支持属性和字典两种访问方式
print(cfg.model.type)
print(cfg['model']['type'])

## 在代码中覆盖字段
cfg.merge_from_dict({'optimizer.lr': 1e-4, 'data.samples_per_gpu': 2})

## 输出合并后的完整配置；tools/train.py 也会写入 work_dir
print(cfg.pretty_text)
cfg.dump('/tmp/resolved_config.py')
```

命令行中等价的覆盖方法：

```bash
python tools/train.py projects/configs/bevformer/bevformer_base.py \
  --cfg-options optimizer.lr=0.0001 data.samples_per_gpu=2
```

配置覆盖发生在插件导入之前。因此可以通过 `--cfg-options` 修改普通超参数；但不要在运行中随意改 `plugin`、`plugin_dir` 或 `type`，除非相应模块的导入与注册关系已经确认。


#### 2.4 配置中的四类字段

可按“由谁消费”理解一个训练配置：

| 字段 | 主要消费者 | `bevformer_base.py` 中的例子 |
|---|---|---|
| `model` | `build_model`，及其递归构建过程 | backbone、neck、BEVFormerHead、Transformer |
| `data` | `build_dataset`、`build_dataloader` | PKL、pipeline、queue_length、sampler |
| `optimizer`、`lr_config` | `build_optimizer`、runner hooks | AdamW、学习率策略、梯度裁剪 |
| `runner`、`evaluation`、`log_config` | `custom_train_model` 和 MMCV runner | epoch 数、评测、日志 |

不要把 `train_cfg` 和顶层 `data.train` 混淆：前者是模型训练策略的一部分，例如 Hungarian matching；后者是训练数据集描述。

---


### 3. Registry 的原理


#### 3.1 它解决了什么问题

若没有 Registry，配置必须直接写 Python 类：

```python
## 不适合作为可保存、可复现实验配置
model = BEVFormer(...)
```

Registry 建立一张从名称到类（或函数）的映射表，使配置能写成：

```python
model = dict(type='BEVFormer', use_grid_mask=True, ...)
```

可将一个 Registry 简化理解为：

```python
REGISTRY = {
    'BEVFormer': BEVFormer,
    'BEVFormerHead': BEVFormerHead,
    ...
}
```

实际的 `mmcv.utils.Registry` 比普通字典多了重复名称检查、父 Registry 查找、模块路径记录和 `build()` 等能力，但基本思想就是上面的映射。


#### 3.2 注册：装饰器在 import 时执行

项目中的 detector 定义是：

```python
from mmdet.models import DETECTORS

@DETECTORS.register_module()
class BEVFormer(MVXTwoStageDetector):
    ...
```

当 Python 首次导入 `bevformer.py` 时，类定义完成，装饰器立刻执行；效果近似：

```python
DETECTORS.register_module(module=BEVFormer)
```

于是 `DETECTORS.get('BEVFormer')` 能取得这个类。默认注册名是类的 `__name__`；也可显式起别名：

```python
@DETECTORS.register_module(name='MyBEVFormer')
class ExperimentalBEVFormer(...):
    ...
```

此后配置必须写 `type='MyBEVFormer'`。同一 Registry 中重复注册同名模块通常会报错；只有明确使用 `force=True` 才会覆盖已有项，这在实验代码中应谨慎使用。


#### 3.3 构建：`build_from_cfg` 的近似逻辑

MMCV 的 `build_from_cfg(cfg, registry, default_args=None)` 可抽象为：

```python
def simplified_build_from_cfg(cfg, registry, default_args=None):
    args = cfg.copy()
    obj_type = args.pop('type')

    if isinstance(obj_type, str):
        obj_cls = registry.get(obj_type)
        if obj_cls is None:
            raise KeyError(f'{obj_type} is not in {registry.name}')
    elif isinstance(obj_type, type):
        obj_cls = obj_type
    else:
        raise TypeError(...)

    args = {**(default_args or {}), **args}
    return obj_cls(**args)
```

重点是：`build_from_cfg` 只处理**当前层**。嵌套配置由父类构造函数显式调用其他构建函数。例如 `BEVFormerHead.__init__()` 调用 `build_bbox_coder(bbox_coder)`，其父类 `DETRHead` 进一步构建 Transformer、loss 等对象。


#### 3.4 多个 Registry 和父 Registry

不同类型的组件登记在不同表中，避免 `BEVFormer`、`BEVFormerHead`、`NMSFreeCoder` 混在一起：

| Registry | 配置中常见位置 | 本项目自定义项 |
|---|---|---|
| `DETECTORS` | `model.type` | `BEVFormer`、`BEVFormerV2` |
| `HEADS` | `pts_bbox_head.type` | `BEVFormerHead` |
| `ATTENTION` | `attn_cfgs[*].type` | `TemporalSelfAttention`、`SpatialCrossAttention` |
| `TRANSFORMER` | `transformer.type` | `PerceptionTransformer` |
| `TRANSFORMER_LAYER_SEQUENCE` | `encoder.type`、`decoder.type` | `BEVFormerEncoder`、`DetectionTransformerDecoder` |
| `TRANSFORMER_LAYER` | `transformerlayers.type` | `BEVFormerLayer` |
| `DATASETS` | `data.train/val/test.type` | `CustomNuScenesDataset` |
| `PIPELINES` | `pipeline[*].type` | 多视图加载、图像增强、收集变换 |
| `BBOX_CODERS` | `bbox_coder.type` | `NMSFreeCoder` |
| `BBOX_ASSIGNERS` | `train_cfg.pts.assigner.type` | `HungarianAssigner3D` |
| `MATCH_COST` | Hungarian assigner 的 cost 配置 | `BBox3DL1Cost`、`SmoothL1Cost` |
| `HOOKS`、`OPTIMIZERS` | 运行时配置 | 自定义 hook、`AdamW2` |

`mmdet3d.models.builder` 还定义了一个以 MMCV `MODELS` 为父表的 `MODELS`。在这个旧版生态中，一部分组件来自 MMCV，一部分来自 MMDetection，一部分来自 MMDetection3D；通过各自 Registry 的构建函数和父表查找，它们能共同出现在一份配置内。

---


### 4. 本项目的插件注册链


#### 4.1 为什么需要 `plugin=True`

`BEVFormer` 不在原始 MMDetection3D v0.17.1 中。即使底层框架已安装，它也不会自动扫描 `projects/`。因此 BEVFormer 配置写了：

```python
plugin = True
plugin_dir = 'projects/mmdet3d_plugin/'
```

`tools/train.py` 在读取配置后检测这两个字段，将目录转换成模块名并执行：

```python
importlib.import_module('projects.mmdet3d_plugin')
```

路径最后的 `/` 不参与模块名；这里实际导入的是包目录中的 `projects/mmdet3d_plugin/__init__.py`。


#### 4.2 `__init__.py` 是注册的开关

插件根目录的 `__init__.py` 会导入：

```python
from .core.bbox.assigners.hungarian_assigner_3d import HungarianAssigner3D
from .core.bbox.coders.nms_free_coder import NMSFreeCoder
from .datasets.pipelines import ...
from .models.opt.adamw import AdamW2
from .bevformer import *
```

而 `bevformer/__init__.py` 再导入：

```python
from .dense_heads import *
from .detectors import *
from .modules import *
from .runner import *
from .hooks import *
```

每个子包的 `__init__.py` 又会导入具体 `.py` 文件。链条的目的不是把符号暴露给用户，而是保证定义中的 `@register_module()` 全部执行。

这解释了一个常见现象：新写了 `my_attention.py` 并加了装饰器，但配置仍报找不到类。通常不是装饰器错误，而是没有把该文件加入某一级 `__init__.py`，导致它从未被 import。


#### 4.3 `custom_imports`：另一种通用导入方式

训练入口也支持 MMCV 的标准字段：

```python
custom_imports = dict(
    imports=['projects.mmdet3d_plugin'],
    allow_failed_imports=False)
```

`tools/train.py` 会调用：

```python
from mmcv.utils import import_modules_from_strings
import_modules_from_strings(**cfg['custom_imports'])
```

它同样是“先 import，再利用装饰器注册”。当前经典 BEVFormer 配置使用的是 `plugin=True/plugin_dir`，没有使用 `custom_imports`。不要同时为同一插件维护两套不一致的导入约定；若项目迁移到标准 MMCV 配置风格，`custom_imports` 更通用。

---


### 5. `bevformer_base.py` 的实际递归构建过程


#### 5.1 第一层：模型

配置片段：

```python
model = dict(
    type='BEVFormer',
    img_backbone=dict(type='ResNet', ...),
    img_neck=dict(type='FPN', ...),
    pts_bbox_head=dict(type='BEVFormerHead', ...))

## train_cfg 与 model 并列；它不是 model 的子字段。
train_cfg = dict(
    pts=dict(assigner=dict(type='HungarianAssigner3D', ...)))
```

训练入口执行：

```python
model = build_model(cfg.model,
                    train_cfg=cfg.get('train_cfg'),
                    test_cfg=cfg.get('test_cfg'))
```

其中 `build_model()` 根据模型类型选择 `build_detector()`，最终以 `DETECTORS.build(cfg)` 找到并调用：

```python
BEVFormer(
    img_backbone={...},
    img_neck={...},
    pts_bbox_head={...},
    train_cfg={...},
    ...)
```

`BEVFormer.__init__()` 将这些字典交给父类 `MVXTwoStageDetector`。父类负责分别构建 backbone、neck、head 等；因此配置里的 `ResNet`、`FPN` 来自框架已有注册表，`BEVFormerHead` 来自本项目插件。


#### 5.2 第二层：检测头、coder、Transformer

`BEVFormerHead` 自身登记到 `HEADS`。在其构造函数中：

```python
self.bbox_coder = build_bbox_coder(bbox_coder)
```

于是 `type='NMSFreeCoder'` 从 `BBOX_CODERS` 找到本项目的 `NMSFreeCoder`。

检测头继承的 `DETRHead` 会构建 `transformer`。配置中：

```python
transformer=dict(
    type='PerceptionTransformer',
    encoder=dict(type='BEVFormerEncoder', ...),
    decoder=dict(type='DetectionTransformerDecoder', ...))
```

`PerceptionTransformer.__init__()` 中明确调用：

```python
self.encoder = build_transformer_layer_sequence(encoder)
self.decoder = build_transformer_layer_sequence(decoder)
```

所以 encoder、decoder 继续由 `TRANSFORMER_LAYER_SEQUENCE` 构建。`BEVFormerEncoder` 构建其多层 `BEVFormerLayer`，而每个 layer 的 `attn_cfgs` 又通过 `ATTENTION` 构建：

```text
BEVFormer
  └── BEVFormerHead
        ├── NMSFreeCoder
        └── PerceptionTransformer
              ├── BEVFormerEncoder
              │     └── BEVFormerLayer × 6
              │           ├── TemporalSelfAttention
              │           └── SpatialCrossAttention
              └── DetectionTransformerDecoder
                    └── DetrTransformerDecoderLayer × 6
```

这正是配置可以描述深层网络结构的原因：每个对象只负责构建自己直接拥有的子对象。


#### 5.3 第三层：训练匹配与损失

`train_cfg.pts` 会由 `MVXTwoStageDetector.__init__()` 写入 `pts_bbox_head` 的构造参数。`DETRHead` 初始化时据此构建 `self.assigner`；训练时 `BEVFormerHead._get_target_single()` 调用 `self.assigner.assign(...)`。该 assigner 的构造函数还会调用三次 `build_match_cost`：

```python
self.cls_cost = build_match_cost(cls_cost)
self.reg_cost = build_match_cost(reg_cost)
self.iou_cost = build_match_cost(iou_cost)
```

因此 `BBox3DL1Cost` 是项目注册的 cost，`FocalLossCost` 与 `IoUCost` 是 MMDetection 已注册的 cost。Registry 可以跨层使用：顶层模型、检测头、匹配器及 cost 都遵循相同模式。


#### 5.4 Dataset 与 pipeline 的构建

训练入口执行：

```python
datasets = [build_dataset(cfg.data.train)]
```

`build_dataset` 先识别 `RepeatDataset`、`ConcatDataset` 等包装器；普通数据集则等价于：

```python
dataset = build_from_cfg(cfg.data.train, DATASETS)
```

于是 `type='CustomNuScenesDataset'` 构建出项目的数据集类。它的父类会把 `pipeline` 列表交给 `Compose`；`Compose` 逐一从 `PIPELINES` 取得 `LoadMultiViewImageFromFiles`、`PhotoMetricDistortionMultiViewImage`、`NormalizeMultiviewImage` 等变换，并在取样时顺序调用。

这部分输入字段与实际样本流动，请继续结合 `docs/dataset.md` 阅读。

---


### 6. 训练 API 的实际使用方法


#### 6.1 标准训练命令

在仓库根目录运行：

```bash
python tools/train.py projects/configs/bevformer/bevformer_base.py
```

通常使用分布式启动脚本：

```bash
./tools/dist_train.sh ./projects/configs/bevformer/bevformer_base.py 8
```

该脚本不写死配置路径：第一个位置参数 `$1` 保存为 `CONFIG`，随后原样放到
`tools/train.py $CONFIG` 中，因此 `argparse` 将它解析为 `args.config`。
`torch.distributed.launch` 会创建 8 个进程，并为每个进程注入不同的
`--local_rank`；所有进程读取同一份配置，但 sampler 按 rank 分配不同数据。

指定输出目录、恢复 checkpoint 和覆盖少量配置：

```bash
python tools/train.py projects/configs/bevformer/bevformer_base.py \
  --work-dir work_dirs/base_debug \
  --resume-from work_dirs/base_debug/latest.pth \
  --cfg-options data.workers_per_gpu=2 optimizer.lr=0.0001
```

多卡启动时，仍由同一个入口完成配置加载和插件注册；随后以 `--launcher pytorch` 初始化分布式环境。具体启动脚本应以项目 README 和当前机器环境为准。


#### 6.2 本项目为什么调用 `custom_train_model`

插件导入完成后，`tools/train.py` 导入：

```python
from projects.mmdet3d_plugin.bevformer.apis.train import custom_train_model
```

最后调用：

```python
custom_train_model(model, datasets, cfg,
                   distributed=distributed,
                   validate=not args.no_validate,
                   timestamp=timestamp,
                   meta=meta)
```

这是本项目对训练流程的入口封装。它进入 `mmdet_train.py`，在那里按配置构建 dataloader、optimizer、runner，注册训练、checkpoint、日志和评测 hooks，然后执行 `runner.run(...)`。因此，`plugin=True` 不仅让模型类可见，也确保自定义训练 API、runner 和 hooks 可被导入。


#### 6.3 构建 API 的最小示例

下面的代码用于理解或调试构建，不等同于完整训练。关键前提是先导入插件：

```python
from mmcv import Config
import projects.mmdet3d_plugin  # 触发项目模块注册
from mmdet3d.models import build_model
from mmdet3d.datasets import build_dataset

cfg = Config.fromfile('projects/configs/bevformer/bevformer_base.py')

model = build_model(
    cfg.model,
    train_cfg=cfg.get('train_cfg'),
    test_cfg=cfg.get('test_cfg'))
dataset = build_dataset(cfg.data.train)

print(type(model))
print(type(dataset))
```

若省略 `import projects.mmdet3d_plugin`，自定义类型尚未注册，构建会失败。这段示例也说明：`Config.fromfile()` 只负责加载配置，不会因看到 `plugin=True` 自动导入插件；这个额外步骤是 `tools/train.py` 的项目逻辑。

---


### 7. 怎样新增或替换一个组件

以新增 attention 为例，完整步骤是：

1. 选择正确的 Registry。当前项目的 attention 使用
   `mmcv.cnn.bricks.registry.ATTENTION`，不是 `HEADS` 或 `DETECTORS`。
2. 创建实现，例如 `projects/mmdet3d_plugin/bevformer/modules/my_attention.py`：

```python
import torch.nn as nn
from mmcv.cnn.bricks.registry import ATTENTION

@ATTENTION.register_module()
class MyAttention(nn.Module):
    def __init__(self, embed_dims=256, **kwargs):
        super().__init__()
        self.proj = nn.Linear(embed_dims, embed_dims)

    def forward(self, query, **kwargs):
        return self.proj(query)
```

3. 确保它会在插件导入时被导入。在 `projects/mmdet3d_plugin/bevformer/modules/__init__.py` 加入：

```python
from .my_attention import MyAttention
```

4. 修改模型配置中实际消费 attention 的位置：

```python
attn_cfgs=[dict(type='MyAttention', embed_dims=_dim_), ...]
```

5. 核对 `__init__` 接受的参数、`forward` 签名和返回 tensor 的 shape 是否与调用位置一致。Registry 只解决“找到并实例化谁”；它不保证模块接口兼容。

替换 Dataset、detector、bbox coder 的过程相同：选择相应 Registry、实现并注册、保证导入、在配置中改 `type`、验证构造参数与运行接口。

---


### 8. 调试与常见错误


#### 8.1 `KeyError: xxx is not in the ... registry`

按下面顺序排查：

1. 配置 `type='xxx'` 是否与类名或显式 `name=` 完全一致，注意大小写；
2. 类是否使用了正确 Registry 的 `@register_module()`；
3. 定义文件是否被某一级 `__init__.py` 导入；
4. 训练是否从 `tools/train.py` 启动，且 `plugin=True`、`plugin_dir` 路径正确；
5. 若是独立脚本，是否在 `build_model` 前执行了 `import projects.mmdet3d_plugin`；
6. 是否误把新版 MMEngine 风格代码放入旧版 MMCV 体系。

可在交互环境中确认注册结果：

```python
import projects.mmdet3d_plugin
from mmdet.models import DETECTORS, HEADS
from mmdet.models.utils.builder import ATTENTION
from mmdet3d.datasets import DATASETS

print(DETECTORS.get('BEVFormer'))
print(HEADS.get('BEVFormerHead'))
print(ATTENTION.get('TemporalSelfAttention'))
print(DATASETS.get('CustomNuScenesDataset'))
```


#### 8.2 `TypeError: __init__() got an unexpected keyword argument ...`

这说明已经成功找到类，错误发生在实例化阶段。检查 `type` 同层的其他配置键能否被该类的 `__init__` 接收。不要用大量 `**kwargs` 无条件吞掉拼写错误；在研究代码中，明确构造参数通常更容易发现配置失配。


#### 8.3 Registry 正确，但 forward 才报错

这说明注册和构造都已通过，但模块接口不匹配。重点核对：

- 输入 tensor shape；
- `forward` 参数名和调用处传入的 keyword；
- 返回值数量、顺序与含义；
- 是否需要读取 `img_metas`、`reference_points`、`prev_bev` 等 BEVFormer 特有输入。


#### 8.4 配置看似修改了，却没有生效

首先打印 `cfg.pretty_text` 或查看 `work_dir` 中训练入口自动保存的同名配置。它反映的是 `_base_` 合并并应用 `--cfg-options` 后的最终值。重点检查：

- 修改的是否是实际启动的那份配置；
- 子配置是否被当前文件同名顶层字段覆盖；
- 修改的是 `data.train.pipeline` 还是仅修改了未被当前配置采用的基类 pipeline；
- 是否需要 `_delete_=True` 来替换复杂字典。

---


### 9. 阅读顺序与下一步

配置和注册机制掌握后，建议沿一次真实构建的对象树向下阅读：

```text
bevformer_base.py
  → BEVFormer.__init__ / forward_train
  → BEVFormerHead.forward / loss
  → PerceptionTransformer.forward
  → BEVFormerEncoder
  → TemporalSelfAttention
  → SpatialCrossAttention / MSDeformableAttention3D
  → DetectionTransformerDecoder
  → HungarianAssigner3D 与 NMSFreeCoder
```

这条路线连接了你已完成的“nuScenes 到 Dataset”部分和下一阶段的“图像如何形成时序 BEV、BEV 如何生成 3D 检测框”。阅读时始终把配置中的某个 `type` 与其注册装饰器、构造函数和 `forward` 放在一起看；这是最快建立全局掌控感的方法。


## 详细实现手册：D. MMCV / MMDetection3D 训练装配、Runner 与 Hook

以下单元是从 `framework_training_pipeline.md` 整理进 notebook 的完整细节快照。前面的教程单元负责建立主线；本节保留推导、边界条件、张量形状与源码定位，便于离线顺序阅读。


## BEVFormer：从配置到训练循环的框架全局总览

本文解释当前仓库如何把一份配置文件变成一次完整训练。目标是建立**全局调用图**：谁负责创建对象、谁负责调度、数据何时变成 batch、模型在哪一层被调用、loss 又如何回到优化器。

本文以 `projects/configs/bevformer/bevformer_base.py` 和 `tools/train.py` 为准。模型内部的图像编码、时序注意力、空间交叉注意力、检测 decoder 和匹配/loss 公式，见 `docs/model_architecture.md` 与 `docs/training_overview.md`；这里把它们看作一个能接收 batch、返回 loss 的模型模块。

本文只解释框架如何组织训练。配置文件如何加载、注册和递归构建，见 `docs/config_and_registry.md`；预测与 GT 如何形成 loss，见 `docs/training_overview.md`。

---


### 1. 先建立四层框架的边界

当前项目不是由 BEVFormer 单独提供全部训练代码，而是逐层复用下层框架：

```text
PyTorch
│  Tensor / autograd / nn.Module / DataLoader / Optimizer / Distributed
│
MMCV 1.x
│  Config / Registry / build_from_cfg / Runner / Hook / 并行封装 / checkpoint
│
MMDetection + MMDetection3D v0.17.1
│  检测器基类 / build_model / build_dataset / bbox 与评测接口 / 标准 train_step
│
BEVFormer 项目（projects/mmdet3d_plugin）
   时序 nuScenes Dataset / 自定义 sampler / BEVFormer detector / BEV 编码器
   自定义训练入口 / 可选视频 Runner / 分布式评测 Hook
```

| 层级 | 主要问题 | 当前项目中对应内容 |
|---|---|---|
| PyTorch | 张量如何计算、怎样求梯度、怎样加载数据 | `torch.Tensor`、`nn.Module`、`DataLoader`、AdamW |
| MMCV | 配置如何变成对象、训练何时调用什么 | `Config`、Registry、`EpochBasedRunner`、Hook |
| MMDetection3D | 一个 3D 检测模型应满足什么接口 | `build_model`、`build_dataset`、Detector 基类、`train_step` |
| BEVFormer | 任务特有的数据和网络如何实现 | `CustomNuScenesDataset`、`BEVFormer`、`BEVFormerHead`、TSA、SCA |

最重要的职责划分是：

```text
Runner：管理“何时取 batch、何时训练、何时反传、何时保存/评测”
模型：管理“batch 如何变成预测和 loss”
Dataset / DataLoader：管理“磁盘样本如何变成 batch”
Registry：管理“配置中的 type 字符串如何找到 Python 类”
```

因此 `EpochBasedRunner` 不认识 BEV、相机标定、TSA 或 SCA；它只要求模型遵循通用的 `train_step` 协议。

---


### 2. 一条训练命令的全局调用图

典型启动命令：

```bash
python tools/train.py projects/configs/bevformer/bevformer_base.py
```

完整调用链如下：

```text
tools/train.py: main()
│
├─ Config.fromfile(config)
├─ 导入 projects.mmdet3d_plugin，使自定义类注册到 MMCV/MMDet Registry
├─ build_model(cfg.model)                 → BEVFormer nn.Module
├─ build_dataset(cfg.data.train)          → CustomNuScenesDataset
└─ custom_train_model(...)
   └─ custom_train_detector(...)
      ├─ build_dataloader(dataset, ...)   → PyTorch DataLoader
      ├─ MMDataParallel / MMDistributedDataParallel 包装模型
      ├─ build_optimizer(model, ...)      → AdamW
      ├─ build_runner(cfg.runner, ...)    → EpochBasedRunner
      ├─ register_training_hooks(...)     → LR / Optimizer / Checkpoint / Logger Hook
      ├─ register_hook(EvalHook)          → 可选验证 Hook
      ├─ resume() / load_checkpoint()     → 可选恢复或加载权重
      └─ runner.run(data_loaders, workflow)
         └─ 反复执行 DataLoader → model.train_step → Hook → optimizer 更新
```

可以把前半段理解为“装配训练系统”，把最后一行理解为“真正开跑”。

---


### 3. 配置：声明训练系统，而不是按顺序执行的脚本


#### 3.1 `_base_` 继承和最终配置

`bevformer_base.py` 的开头：

```python
_base_ = [
    '../datasets/custom_nus-3d.py',
    '../_base_/default_runtime.py'
]
```

`Config.fromfile()` 会先加载父配置，再以当前文件的同名字段覆盖父字段，最终得到一个 `cfg` 对象。它不是“逐行运行模型”的程序，而是一棵描述对象和运行策略的嵌套字典。

当前配置中最关键的顶层字段：

| 配置段 | 作用 | 后续消费者 |
|---|---|---|
| `plugin`、`plugin_dir` | 允许导入项目扩展包 | `tools/train.py` |
| `model` | 网络的递归结构与超参数 | `build_model` |
| `data` | Dataset、pipeline、sampler、batch/worker 数 | `build_dataset`、`build_dataloader` |
| `optimizer` | AdamW 和参数组策略 | `build_optimizer` |
| `runner` | Runner 类型及 epoch 上限 | `build_runner` |
| `lr_config` | 学习率策略 | `LrUpdaterHook` |
| `optimizer_config` | 梯度裁剪、混合精度相关选项 | `OptimizerHook` / `Fp16OptimizerHook` |
| `checkpoint_config` | checkpoint 间隔 | `CheckpointHook` |
| `log_config` | 日志间隔和输出后端 | `LoggerHook` |
| `evaluation` | 验证间隔和结果配置 | `EvalHook` |
| `workflow` | train/val 阶段的调度顺序 | `runner.run` |

命令行 `--cfg-options key=value` 在 `Config.fromfile()` 后通过 `cfg.merge_from_dict()` 覆盖最终配置，因此可临时改变学习率、batch size 或工作目录，无须改原文件。


#### 3.2 为什么配置的 `type` 能创建 Python 对象

配置中的：

```python
model = dict(type='BEVFormer', ...)
```

不是 Python 类本身，而是一个名字。`build_model(cfg.model)` 的核心逻辑可以抽象为：

```python
cls = DETECTORS.get(cfg['type'])   # 根据 'BEVFormer' 查 Registry
obj = cls(**其余配置字段)          # 调用 BEVFormer 的 __init__
```

`BEVFormer` 类在 `projects/mmdet3d_plugin/bevformer/detectors/bevformer.py` 中用：

```python
@DETECTORS.register_module()
class BEVFormer(...):
    ...
```

注册是 Python import 的副作用：模块被导入时，装饰器执行，类对象进入 `DETECTORS` Registry。随后递归构建过程中，`BEVFormer` 的构造函数继续让 MMDetection3D 构建 ResNet、FPN、`BEVFormerHead`；检测头继续构建 Transformer；Transformer 再构建 encoder、decoder 和 attention。


#### 3.3 为什么必须有 `plugin=True`

基础框架默认并不知道 `BEVFormer`、`TemporalSelfAttention` 或 `CustomNuScenesDataset`。本项目配置设置：

```python
plugin = True
plugin_dir = 'projects/mmdet3d_plugin/'
```

`tools/train.py` 将路径转换为 `projects.mmdet3d_plugin` 并 `importlib.import_module(...)`。包内 `__init__.py` 连锁导入 Dataset、模型、Runner 等子包，完成相应 Registry 的注册。

如果跳过这一步，配置字符串 `type='BEVFormer'` 即使拼写正确，也会因 Registry 中找不到同名类而构建失败。


#### 3.4 Registry 是分区的，不是一个全局字典

不同类别注册到不同 Registry：

| `type` 出现的位置 | 典型 Registry | 例子 |
|---|---|---|
| `model.type` | `DETECTORS` | `BEVFormer` |
| `pts_bbox_head.type` | `HEADS` | `BEVFormerHead` |
| `transformer.type` | `TRANSFORMER` | `PerceptionTransformer` |
| `attn_cfgs[*].type` | `ATTENTION` | `TemporalSelfAttention`、`SpatialCrossAttention` |
| `encoder.type`、`decoder.type` | Transformer layer/sequence Registry | `BEVFormerEncoder`、`DetectionTransformerDecoder` |
| `data.train.type` | `DATASETS` | `CustomNuScenesDataset` |
| `pipeline[*].type` | `PIPELINES` | `LoadMultiViewImageFromFiles` |
| `runner.type` | `RUNNERS` | `EpochBasedRunner`、`EpochBasedRunner_video` |
| `custom_hooks[*].type` | `HOOKS` | 项目自定义 Hook |

这使配置可以描述一套可替换的组合。例如把 `runner.type` 改成自定义 Runner，只会影响调度实现；模型部分仍由 `model` 段决定。

---


### 4. 训练开始前：对象怎样被装配


#### 4.1 模型对象

`tools/train.py`：

```python
model = build_model(
    cfg.model,
    train_cfg=cfg.get('train_cfg'),
    test_cfg=cfg.get('test_cfg'))
model.init_weights()
```

此时得到的是 CPU 上的 `BEVFormer` `nn.Module`。它已经拥有完整网络，但还没有拿到数据、没有创建优化器、也没有开始 forward。随后 `custom_train_detector()` 才把它移动到 GPU 并包装为：

```text
单进程：MMDataParallel(BEVFormer)
分布式：MMDistributedDataParallel(BEVFormer)
```

在 Hook 中如需访问实际 BEVFormer 成员，通常使用：

```python
model = runner.model.module
head = model.pts_bbox_head
```

因为 `runner.model` 可能是并行包装器，`.module` 才是原模型。


#### 4.2 Dataset、Sampler 与 DataLoader

训练入口先调用：

```python
datasets = [build_dataset(cfg.data.train)]
```

`cfg.data.train.type='CustomNuScenesDataset'` 经过 `DATASETS` Registry 创建 Dataset。一个训练样本是一段时序队列，而非一张图片。对默认配置，单样本的图像字段在 formatting 后可概念化为：

```text
img: [T, N_cam, 3, H, W]
     T=queue_length=4，N_cam=6
```

接着 `custom_train_detector()` 调用项目的 `build_dataloader()`：

```python
build_dataloader(
    dataset,
    samples_per_gpu=cfg.data.samples_per_gpu,
    workers_per_gpu=cfg.data.workers_per_gpu,
    dist=distributed,
    shuffler_sampler=cfg.data.shuffler_sampler,
    nonshuffler_sampler=cfg.data.nonshuffler_sampler,
)
```

它组装了四个不同职责的组件：

```text
Sampler
  决定“下一个取哪个样本索引”；分布式时每个 rank 取不同子集。
  训练使用 DistributedGroupSampler，验证使用 DistributedSampler。

隐式 BatchSampler（PyTorch DataLoader 内部）
  将 sampler 连续给出的索引每 samples_per_gpu 个打成一批。

Dataset.__getitem__(index)
  按索引读取一个队列样本，运行 pipeline，返回 dict 和 DataContainer。

MMCV collate_fn
  合并 B 个样本；规则张量沿 batch 维堆叠，变长或 CPU-only 信息保留容器结构。
```

训练配置默认 `samples_per_gpu=1`，所以单卡常见 batch 是：

```text
img: [B=1, T=4, N_cam=6, 3, H, W]
```

若把每卡 batch 调为 `B>1`，`img` 的第一维增加；每个样本自己的时间长度、相机数和图像尺寸保持不变。


#### 4.3 DataContainer 为什么存在

batch 内字段不是都能 `torch.stack`：

| 字段 | batch 后表示 | 原因 |
|---|---|---|
| `img` | 规则 Tensor，`[B,T,N_cam,3,H,W]` | 所有样本经 padding 后尺寸一致 |
| `img_metas` | `list[B][T]` 的字典 | 每帧标定矩阵、文件名、位姿等结构复杂且不需要直接组成 Tensor |
| `gt_bboxes_3d` | `list[B]` | 每个场景中目标数不同 |
| `gt_labels_3d` | `list[B]` | 长度跟随每个样本的目标数 |

`DataContainer` 告诉 MMCV `collate` 和并行 scatter：该字段是否 stack、是否只留在 CPU、怎样按 GPU 切分。它是“数据搬运规则的描述”，不是模型中的特征张量。


#### 4.4 优化器和 Runner

训练入口按配置创建：

```python
optimizer = build_optimizer(model, cfg.optimizer)
runner = build_runner(
    cfg.runner,
    default_args=dict(
        model=model,
        optimizer=optimizer,
        work_dir=cfg.work_dir,
        logger=logger,
        meta=meta,
    ))
```

这两段配置分别产生：

```python
optimizer = dict(type='AdamW', lr=2e-4, ...)
runner = dict(type='EpochBasedRunner', max_epochs=24)
```

配置没有整体塞给 Runner。`cfg.runner` 仅负责 Runner 自身的构造参数；已创建的模型、优化器、日志器等运行时对象经 `default_args` 传入。`Runner` 因而持有 `runner.model`、`runner.optimizer`、`runner.work_dir`、`runner.logger`、epoch/iter 计数器和当前输出。

---


### 5. Hook：把训练附属行为从主循环中拆开


#### 5.1 Hook 的注册方式

`custom_train_detector()` 中：

```python
runner.register_training_hooks(
    cfg.lr_config,
    optimizer_config,
    cfg.checkpoint_config,
    cfg.log_config,
    cfg.get('momentum_config', None))
```

这是便利方法：根据配置一次注册多个标准 Hook，包括学习率、优化器、checkpoint 和日志 Hook。之后还可多次调用：

```python
runner.register_hook(DistSamplerSeedHook())
runner.register_hook(eval_hook(val_dataloader, **eval_cfg))
runner.register_hook(custom_hook, priority='NORMAL')
```

Hook 没有“只能注册一次”的限制。Runner 保存的是按优先级排序的 Hook 列表。重复注册同一类 Hook 在技术上可行，但一般会造成重复保存、重复评测或重复更新，不应无意重复。


#### 5.2 配置决定实例，类定义决定触发点

若配置中有：

```python
custom_hooks = [
    dict(type='MyHook', interval=2, priority='LOW')
]
```

入口代码用 `build_from_cfg(hook_cfg, HOOKS)` 创建 `MyHook(interval=2)`，并调用 `runner.register_hook(...)`。配置没有写“在哪个时刻触发”；触发点写在类的方法名中：

```python
@HOOKS.register_module()
class MyHook(Hook):
    def after_train_iter(self, runner):
        pass

    def after_train_epoch(self, runner):
        pass
```

一个 Hook 可以实现多个生命周期方法。常见时机：

```text
before_run / after_run
before_train_epoch / after_train_epoch
before_train_iter / after_train_iter
```

例如：

| Hook | 常见触发点 | 默认配置中的来源 |
|---|---|---|
| 学习率 Hook | epoch 或 iter 前 | `lr_config` |
| Optimizer Hook | `after_train_iter` | `optimizer_config` |
| Checkpoint Hook | `after_train_epoch` | `checkpoint_config.interval=1` |
| Text/TensorBoard Logger | 定期的 iter 后 | `log_config.interval=50` |
| Eval Hook | 指定 epoch 后 | `evaluation.interval=1` |
| DistSamplerSeedHook | epoch 前 | 分布式训练自动追加 |


#### 5.3 Hook 如何访问模型和训练状态

Hook 的回调只接收 `runner`，但 Runner 创建时已保存模型、优化器和状态：

```python
def after_train_iter(self, runner):
    lr = runner.optimizer.param_groups[0]['lr']
    loss = runner.outputs['loss']
    epoch = runner.epoch
    iteration = runner.iter
```

常用可访问成员：

```text
runner.model          当前模型（可能是 DP/DDP 包装器）
runner.optimizer      优化器
runner.epoch          当前 epoch，从 0 开始
runner.iter           全局 iteration，从 0 开始
runner.inner_iter     当前 epoch 内的 iteration
runner.outputs        本次 train_step 返回的 loss/log_vars/num_samples
runner.log_buffer     用于日志输出的聚合指标
runner.data_loader    当前阶段正在迭代的 DataLoader
```

标准 Hook 通常不能直接取得当前 `data_batch`，因为它作为 `run_iter()` 的局部参数传给 `model.train_step()`。若必须观察原始 `img` 或 `img_metas`，应在模型中记录统计量、扩展 Runner 的 `run_iter()`，或让模型将所需数值放入 `log_vars`。

---


### 6. `runner.run()`：标准训练调度到底做了什么

当前基础配置：

```python
runner = dict(type='EpochBasedRunner', max_epochs=24)
workflow = [('train', 1)]
```

`runner.run(data_loaders, cfg.workflow)` 是进入训练循环的入口。`EpochBasedRunner` 的抽象运行顺序为：

```text
before_run Hooks
│
├─ epoch = 0 ... max_epochs - 1
│  │
│  └─ workflow 中的每一个阶段
│     │
│     └─ ('train', 1)
│        │
│        ├─ model.train()
│        ├─ before_train_epoch Hooks
│        │
│        └─ for data_batch in train_dataloader
│           │
│           ├─ before_train_iter Hooks
│           ├─ run_iter(data_batch, train_mode=True)
│           └─ after_train_iter Hooks
│
│        └─ after_train_epoch Hooks
│           ├─ CheckpointHook：按 interval 保存状态
│           └─ EvalHook：按 interval 运行验证（若已注册）
│
└─ after_run Hooks
```

其中 `run_iter()` 的标准核心是：

```python
outputs = model.train_step(data_batch, optimizer)
runner.outputs = outputs
runner.log_buffer.update(outputs['log_vars'], outputs['num_samples'])
```

训练时最关键的一次迭代可进一步展开为：

```text
DataLoader 取 B 个 Dataset 样本
  → MMCV collate 组织为 data_batch / DataContainer
  → 并行包装器 scatter 到当前 GPU
  → model.train_step(data_batch, optimizer)
  → model(..., return_loss=True)
  → BEVFormer.forward_train(...)
  → 返回各项 loss
  → Detector 基类汇总为 outputs['loss']、outputs['log_vars']
  → OptimizerHook.after_train_iter
       optimizer.zero_grad()
       outputs['loss'].backward()
       可选 grad clip
       optimizer.step()
```

因此反向传播和参数更新不直接写在 `BEVFormer.forward_train()` 中，而在 `OptimizerHook` 中。模型只负责返回可微的 loss 字典。


#### 6.1 当前项目的时序处理属于模型，不属于标准 Runner

对 `bevformer_base.py`，Runner 类型是标准 `EpochBasedRunner`。它把一个完整 batch 交给模型；时序切分发生在：

```text
BEVFormer.forward_train(img=[B,T,N_cam,3,H,W])
  ├─ prev_img = img[:, :-1]     → [B,T-1,N_cam,3,H,W]
  ├─ img = img[:, -1]           → [B,N_cam,3,H,W]
  ├─ 历史帧在 no_grad 下递推形成 prev_bev
  └─ 当前帧计算预测和监督 loss
```

项目另有 `EpochBasedRunner_video`，定义在 `projects/mmdet3d_plugin/bevformer/runner/epoch_based_runner.py`。它覆写 `run_iter()`，把一个队列 batch 拆成每帧 DataContainer，先通过 `eval_model.val_step()` 递推历史 BEV，再让 `model.train_step()` 训练最后一帧。它只有在配置写：

```python
runner = dict(type='EpochBasedRunner_video', ...)
```

时才会启用。不要把这条可选路径和当前基础配置的标准 Runner 路径混为一谈。

---


### 7. 一次 batch 在框架边界处的形状与所有权

这里仅追踪框架层面的边界，不展开模型内部网络：

| 位置 | `img` 形状/结构 | 负责者 | 说明 |
|---|---|---|---|
| Dataset 单样本 | `[T,N_cam,3,H,W]` | `CustomNuScenesDataset` + pipeline | 一个当前帧及其历史队列 |
| DataLoader collate 后 | `img` 的逻辑 payload 为 `[B,T,N_cam,3,H,W]`，外层仍由 `DataContainer` 包装 | MMCV `collate` | 规则图片堆叠；meta/标注保持 list |
| GPU scatter 后 | 同一逻辑形状，位于目标 GPU | MMDataParallel / DDP | DP 时按 B 维分卡；DDP 每进程已由 sampler 分到子 batch |
| `BEVFormer.forward_train` | 历史 `[B,T-1,N_cam,3,H,W]`；当前 `[B,N_cam,3,H,W]` | BEVFormer | 时序处理开始 |
| `model.train_step` 输出 | `loss` 标量、`log_vars` 字典、`num_samples` | MMDet Detector 基类 | 返回给 Runner/Hook |

`B` 的语义因运行模式略有不同：

```text
单卡：B = samples_per_gpu
MMDataParallel 多卡：DataLoader 先产生 num_gpus * samples_per_gpu，再按 B 维切到各卡
DDP 多卡：每个进程的 DataLoader 只产生本 rank 的 B = samples_per_gpu
```

默认训练配置 `samples_per_gpu=1`，但该数值是“每 GPU 的时序样本数”，不是“每 GPU 的单图数量”。每一个样本含 `T=4` 个时间点和 `N_cam=6` 个相机视角。

---


### 8. 验证、checkpoint 与恢复训练


#### 8.1 验证

`tools/train.py` 默认把 `validate=True` 传给 `custom_train_model`；除非启动参数给出 `--no-validate`。训练入口构建：

```python
val_dataset = custom_build_dataset(cfg.data.val, dict(test_mode=True))
val_dataloader = build_dataloader(..., shuffle=False)
eval_hook = EvalHook 或 CustomDistEvalHook
runner.register_hook(eval_hook(...))
```

默认 `evaluation.interval=1`，所以 Eval Hook 每个 epoch 后运行验证。测试/验证时，`BEVFormer.forward_test()` 在 detector 实例的 `prev_frame_info` 中缓存上一帧 `prev_bev`，并在 scene 改变时清空；这与训练时 Dataset 队列提供历史帧的方式不同。


#### 8.2 checkpoint

`checkpoint_config=dict(interval=1)` 创建 checkpoint Hook。该 Hook 在 epoch 结束后保存：

```text
模型参数 state_dict
优化器状态（动量、AdamW 的一阶/二阶矩等）
epoch / iter 计数
meta：配置文本、类别顺序、框架版本、环境信息等
```

两种加载语义：

| 配置/命令 | 方法 | 恢复范围 | 典型用途 |
|---|---|---|---|
| `resume_from` / `--resume-from` | `runner.resume()` | 模型、优化器、epoch/iter 等完整状态 | 中断后继续同一实验 |
| `load_from` | `runner.load_checkpoint()` | 主要加载模型参数 | 从预训练权重开始新实验 |

基础配置中的 `load_from='ckpts/r101_dcn_fcos3d_pretrain.pth'` 用于初始化图像 backbone 等可匹配权重，不代表恢复一次已经进行到中途的 BEVFormer 训练。

---


### 9. 阅读和调试的推荐顺序

若目标是掌握整个项目，建议沿真实调用边界逐层阅读：

1. `tools/train.py`：配置加载、插件导入、模型/Dataset 创建、训练入口。
2. `projects/mmdet3d_plugin/bevformer/apis/train.py`：为什么 BEVFormer 进入自定义训练函数。
3. `projects/mmdet3d_plugin/bevformer/apis/mmdet_train.py`：DataLoader、并行封装、优化器、Runner、Hook、验证。
4. `projects/mmdet3d_plugin/datasets/builder.py`：sampler、batch、collate、worker 的实际组装。
5. 当前配置 `projects/configs/bevformer/bevformer_base.py`：将每段配置映射回其消费者。
6. `BEVFormer.forward_train()`：Runner 调用 `train_step` 后，模型如何处理一个完整时序 batch。
7. 需要理解另一种时序策略时，再阅读 `EpochBasedRunner_video.run_iter()`，并先确认配置是否真的选择它。

阅读时始终回答三个问题：

```text
这个配置段由谁消费？
这个对象由哪个 Registry 构建？
当前数据是在 Dataset、DataLoader、Runner，还是模型内部？
```

能清楚回答这三问，就能把配置、注册和训练调度从看似分散的文件还原成一条完整系统链路。


## 详细实现手册：E. 模型结构：图像、BEV encoder、TSA、SCA 与检测头

以下单元是从 `model_architecture.md` 整理进 notebook 的完整细节快照。前面的教程单元负责建立主线；本节保留推导、边界条件、张量形状与源码定位，便于离线顺序阅读。


## BEVFormer 模型结构详解：从一个 batch 到 3D 检测结果

本文按一次真实前向传播的数据流解释当前仓库的经典 BEVFormer。基线是 projects/configs/bevformer/bevformer_base.py：ResNet-101、四层 FPN、200×200 BEV、6 层 BEV encoder、900 个 object query 和 6 层 detection decoder。

本文只讨论模型内部的数据流。PKL、Dataset、DataLoader、训练 loss 和配置注册请分别参阅 docs/nuscence_to_pkl.md、docs/dataset.md、docs/training_overview.md、docs/config_and_registry.md。

主要源码：

~~~
projects/mmdet3d_plugin/bevformer/detectors/bevformer.py
projects/mmdet3d_plugin/bevformer/dense_heads/bevformer_head.py
projects/mmdet3d_plugin/bevformer/modules/transformer.py
projects/mmdet3d_plugin/bevformer/modules/encoder.py
projects/mmdet3d_plugin/bevformer/modules/temporal_self_attention.py
projects/mmdet3d_plugin/bevformer/modules/spatial_cross_attention.py
projects/mmdet3d_plugin/bevformer/modules/decoder.py
~~~

---


### 阅读路线

本文按“先走通主链路，再下钻关键模块”的顺序组织。第一次阅读建议只看每一部分的概览、形状和结论；遇到坐标、时序或注意力实现细节时，再回到对应的小节。

| 阅读目标 | 建议阅读 | 能回答的问题 |
|-|-|-|
| 先建立全局感 | 第 1、2、3、6、10、11、12、15、16、18 节 | 一个 batch 如何变成 3D 框？关键张量是什么形状？ |
| 弄清时序 BEV | 第 7 节和 10.1 节 | 历史 BEV 如何旋转、平移、采样并融合？ |
| 弄清图像如何写入 BEV | 第 8、9、10.2 节 | 一个 BEV cell 怎样投影到六相机并取图像特征？ |
| 弄清检测输出 | 第 12～16 节 | 900 个 query 怎样从 BEV 得到类别和 3D 框？ |
| 对照源码调试 | 第 19 节 | 应该在哪些函数打断点、打印哪些 shape？ |

> **先记住主线：**历史帧先递推为 `prev_bev`；当前帧图像经 CNN/FPN 得到六相机多尺度特征；BEV encoder 每层先用 TSA 读取历史 BEV、再用 SCA 读取当前图像；检测 decoder 用 900 个 object query 从最终 BEV 读取局部证据并输出 3D 框。

---


### 第一部分：输入、时序队列与图像特征


#### 1. 全局图和记号

| 符号 | base 配置 | 含义 |
|-|-:|-|
| B | 通常 1 | 每张 GPU 的 batch size |
| T | 4 | 训练队列：3 历史帧和 1 当前帧 |
| N | 6 | 相机数 |
| C | 256 | FPN/Transformer embedding 维度 |
| H_bev, W_bev | 200, 200 | BEV 网格高、宽 |
| L_bev | 40,000 | BEV token 数，200×200 |
| L_enc | 6 | BEV encoder layer 数 |
| Q | 900 | object query 数 |
| L_dec | 6 | detection decoder layer 数 |
| D | 4 | 每个 BEV 柱的高度 reference point 数 |

~~~
batch
  img: [B,T,6,3,H,W]
  img_metas: B 个时序 meta 字典
  gt_bboxes_3d / gt_labels_3d: 当前帧 GT
      │
      ├─ 历史 3 帧：图像 → 特征 → encoder（无梯度）→ prev_bev
      │
      └─ 当前帧：图像 → ResNet/FPN → 四尺度、六相机 feature
                         │
                         ▼
                   6 层 BEV encoder
                     ├─ TSA：当前 BEV 与 prev_bev 融合
                     └─ SCA：按几何投影从六相机 feature 取证
                         │
                         ▼
                 BEV memory [B,40000,256]
                         │
                         ▼
                 900 个 object query + 6 层 decoder
                         │
                         ▼
          分类 logits [6,B,900,10]，框编码 [6,B,900,10]
~~~

当前模型是纯视觉模型。虽然框架属性名是 pts_bbox_head，它不读取 points，也没有 LiDAR voxel 分支；pts 的含义是输出 3D point-cloud 坐标系中的检测框。

---


#### 2. 模型入口：batch 的每个字段怎样使用

DataLoader 解包后，训练调用可抽象成：

~~~
model(
  return_loss=True,
  img=img,
  img_metas=img_metas,
  gt_bboxes_3d=gt_bboxes_3d,
  gt_labels_3d=gt_labels_3d)
~~~

| 字段 | 入口形态 | 主要消费者 | 实际用途 |
|-|-|-|-|
| img | Tensor [B,T,N,3,H,W] | extract_feat | 全部相机图像 |
| img_metas | list[B]，每项含 T 个 frame meta | Transformer/encoder | 相机投影、图像尺寸、CAN bus、场景时序 |
| gt_bboxes_3d | list[B] | head.loss | 当前帧框的匹配和回归监督 |
| gt_labels_3d | list[B] | head.loss | 当前帧类别监督 |
| points、gt_bboxes、gt_labels | None | 无 | 纯视觉配置不使用 |

img_metas 中进入核心计算的字段：

| 字段 | 使用处 | 作用 |
|-|-|-|
| lidar2img | encoder.point_sampling | 把 LiDAR/BEV 坐标的 3D 点投到每个相机 |
| img_shape | point_sampling | 像素坐标归一化、可见性判断 |
| can_bus[:3] | transformer.get_bev_features | 历史 BEV 的平移 shift |
| can_bus[-2] | 同上 | 绝对 yaw，换算 shift 的坐标方向 |
| can_bus[-1] | 同上 | 相邻帧 yaw 差，旋转历史 BEV |
| 完整 18 维 can_bus | 同上 | 经 MLP 加给每一个 BEV query |
| prev_bev_exists | detector | scene 切换时丢弃历史 BEV |
| box_type_3d | get_bboxes | 推理框封装类型 |

GT 不会进入 CNN、TSA、SCA 或 detection decoder。模型先预测；GT 只在预测完成后参与 Hungarian matching 和 loss。

---


#### 3. 先从时序队列分离当前帧和历史帧

BEVFormer.forward_train 的实际顺序是：

~~~
prev_img = img[:, :-1, ...]  # [B,3,6,3,H,W]
cur_img  = img[:, -1, ...]   # [B,6,3,H,W]

prev_bev = obtain_history_bev(prev_img, img_metas)
cur_meta = [each[T-1] for each in img_metas]
cur_feats = extract_feat(cur_img, cur_meta)
losses = forward_pts_train(cur_feats, current GT, cur_meta, prev_bev)
~~~

历史三帧不是和当前帧平行做四次检测。它们按时间顺序迭代：

~~~
历史帧 0 → BEV_0
历史帧 1 + BEV_0 → BEV_1
历史帧 2 + BEV_1 → prev_bev
当前帧 + prev_bev → 当前预测 → loss
~~~

obtain_history_bev 使用 self.eval() 和 torch.no_grad()。历史帧能更新 prev_bev，但不保存反向图、不产生 loss、不对历史图像反传梯度。这样训练时的时序上下文来自 4 帧，显存却主要由当前帧的可训练前向占用。

若 meta 中 prev_bev_exists 为 False，代表新 scene 或队列边界；此时 prev_bev 清空。历史帧仍可在新 scene 内继续建立新的状态。


#### 4. 相机维怎样经过 CNN

当前帧的图像形状为：

~~~
[B,N,3,H,W] = [B,6,3,H,W]
~~~

共享的 ResNet 只接受普通 4D 图像 batch，因此 extract_img_feat 将相机维合并进 batch：

~~~
[B,6,3,H,W] → [B×6,3,H,W]
~~~

B=1 时，代码通过 squeeze 得到等价的 [6,3,H,W]。六张图共用完全相同的 ResNet 和 FPN 参数，模型不会给 CAM_FRONT、CAM_BACK 各建一套 CNN。相机身份稍后由 camera embedding 和 lidar2img 几何区分。

历史帧也采用同样方式，只是先把时间维并入 batch：

~~~
[B,3,6,3,H,W] → [B×3,6,3,H,W] → [B×3×6,3,H,W]
~~~

因此历史三帧的 CNN/FPN 特征可以一次批量提取，再按时间重新切开。


#### 5. GridMask、ResNet、FPN 的输出

##### 5.1 GridMask

配置 use_grid_mask=True 时，进入 backbone 前，当前训练帧经过 GridMask：周期性网格中的部分像素置零。它不改变形状，不修改相机内外参；仅作为图像级正则化。历史帧在 obtain_history_bev 内临时切到 eval 模式，因而不经过 GridMask；推理时同样不生效。

##### 5.2 ResNet 与 FPN

base 使用 ResNet-101 的三个 stage 输出，FPN 将它们统一为 C=256 通道并产生 4 个尺度：

~~~
FPN 输出：
[B×N,256,h0,w0]
[B×N,256,h1,w1]
[B×N,256,h2,w2]
[B×N,256,h3,w3]
~~~

若 pipeline 输出图像是典型的 928×1600，四层通常为：

~~~
[116,200]， [58,100]， [29,50]， [15,25]
~~~

随后每层恢复相机维：

~~~
[B×N,256,h,w] → [B,N,256,h,w]
~~~

故第一部分最终输出是长度为 4 的 list：

~~~
img_feats[l] = [B,6,256,h_l,w_l]
~~~

到这里所有特征依旧位于各自二维图像平面中：没有相机融合、没有鸟瞰表示、没有时序融合。

---


### 第二部分：BEV 构建、时序对齐与多视角融合


#### 6. 初始 BEV query 与图像 token 整理

BEVFormerHead 在进入 Transformer 前准备：

| 名称 | 形状 | 含义 |
|-|-|-|
| bev_embedding.weight | [40000,256] | 每个 BEV cell 一个可学习内容 token |
| bev_pos | [B,256,200,200] | LearnedPositionalEncoding 产生的二维位置编码 |
| query_embedding.weight | [900,512] | detection decoder 的 object query，暂不进入 encoder |

BEV query 不是由图片直接生成的。它起初是一张可学习的 200×200 token 表；SCA 在后续步骤把视觉信息写入这些 token，TSA 把时间信息写入这些 token。

四层图像特征被整理为 token。对 level l：

~~~
[B,6,256,h_l,w_l]
  → flatten h_l×w_l
[6,B,h_l×w_l,256]
  + camera embedding[6,256]
  + level embedding[256]
~~~

将 4 层沿 token 维拼接：

~~~
feat_flatten = [N,S,B,256]
spatial_shapes = [[h0,w0],[h1,w1],[h2,w2],[h3,w3]]
level_start_index = 各尺度 token 的起始偏移
~~~

928×1600 的典型例子：

~~~
S = 116×200 + 58×100 + 29×50 + 15×25 = 30,825
feat_flatten = [6,30825,B,256]
level_start_index = [0,23200,29000,30450]
~~~

camera embedding 标记“此 token 来自哪一个相机”，level embedding 标记“此 token 来自哪一个 FPN 尺度”。两者均是加法，不改变 feature shape。


#### 7. CAN bus 如何对齐和增强 BEV

这一节完成的不是图像坐标投影，也不是直接平移 `prev_bev` Tensor；它做的是：将**前一时刻 BEV 的采样坐标**对齐到**当前时刻 ego/LiDAR BEV 网格**。平移作用于 TSA 的历史 `reference point`，旋转则作用于历史 BEV feature map。

这一节较长，建议按下面顺序读：

1. **先固定坐标事实**：物理 LiDAR 是 `x` 前、`y` 左；模型数组数值是 `W→x`、`H→y`。数组 `H` 向下显示只影响视觉直觉，不能改变这一数值映射。
2. **再看源码实际做了什么**：7.1 说明 `can_bus` 在队列中保存的量，7.2 将其逐式展开为 `shift`。
3. **最后看它怎样被消费**：7.3 概览三条运动通路，7.4 说明 `rotate(prev_bev, Δψ)` 和 `ref_2d + shift` 怎样共同决定历史采样位置。

> `shift` 的数值约定是本实现最容易误读的部分。不要先假设它等于“当前局部物理 x/y 位移”；先看 7.2.4 的标准公式与源码公式对照，再回来看 TSA 的采样位置。

##### 7.1 训练队列中的 CAN bus 到底保存什么

单帧 `get_data_info()` 刚读取 PKL 时，`can_bus` 的位姿字段是当前帧的 global 绝对量：

~~~
can_bus[:3]  = p_t^G = [x_t^G, y_t^G, z_t^G]      # global 平移，单位 m
can_bus[-2]  = ψ_t                                 # 当前绝对 yaw，弧度
can_bus[-1]  = ψ_t × 180/π                         # 当前绝对 yaw，角度
~~~

训练集随后在 `CustomNuScenesDataset.union2one()` 把一个样本的 T 帧合并。对同一 scene 中第 t 条**实际入选帧**，它改写为：

$$
\Delta p_t^G = p_t^G - p_{t-1}^G,
\qquad
\Delta \psi_t^{\circ} = \psi_t^{\circ} - \psi_{t-1}^{\circ}
$$

~~~
can_bus[:3]  = Δp_t^G                 # 相对前一入选帧的 global 平移
can_bus[-2]  = ψ_t                     # 仍是当前帧绝对 yaw（弧度）
can_bus[-1]  = Δψ_t°                   # 相对前一入选帧的 yaw 差（角度）
~~~

队列会随机丢弃一条历史帧，因此“前一入选帧”不保证是原数据集严格相邻的关键帧。例如训练队列为 `[t0,t2,t3,t4]` 时，`t2` 存相对 `t0` 的运动，`t3` 存相对 `t2` 的运动，`t4` 存相对 `t3` 的运动。scene 首帧设 `prev_bev_exists=False`，平移和 `can_bus[-1]` 置零；此时不会使用跨 scene 的历史 BEV。

`obtain_history_bev()` 逐帧调用本函数。因此一次 `get_bev_features()` 只读取**当前正在编码帧**的 meta，只计算：

~~~
前一条入选帧的 prev_bev  →  当前帧 BEV
~~~

它不会一次把 T 帧的全部运动量相加。训练中的递推是：

~~~
t0  → BEV_0
t2 + BEV_0 → BEV_2
t3 + BEV_2 → BEV_3
t4 + BEV_3 → 当前 BEV 与检测结果
~~~

##### 7.2 从 global 相对平移变为当前 BEV 网格位移

`get_bev_features()` 读取：

~~~
delta_x = can_bus[0]     # Δx_G，单位 m
delta_y = can_bus[1]     # Δy_G，单位 m
ego_angle = can_bus[-2]  # ψ_t，当前绝对 yaw，代码转为 degree
~~~

这里的 `(delta_x, delta_y)` 仍在 global 坐标轴中，而 TSA 的二维 reference point 位于当前帧的 BEV 平面。代码先将 global 平移写成极坐标：

$$
r = \sqrt{\Delta x_G^2 + \Delta y_G^2}
$$

$$
\phi = \operatorname{atan2}(\Delta y_G, \Delta x_G)
$$

再用当前 ego 的 global yaw `ψ_t` 得到 BEVFormer 代码约定下的局部方向：

$$
\alpha = \psi_t - \phi
$$

其中 `atan2` 能正确区分四个象限。`ψ_t-φ` 的符号、以及后面 x 用 `sin`、y 用 `cos`，来自该项目的 ego/BEV 网格轴约定；不要在不了解坐标轴方向时任意交换它们。

###### 7.2.1 一个必须算清的例子：`shift_x/shift_y` 不是 global 的 `(Δx,Δy)`

代码真正计算的是：

$$
\begin{aligned}
\mathrm{shift}_{W}&=\frac{r\sin(\psi_t-\phi)}{g_W W_{bev}},\\
\mathrm{shift}_{H}&=\frac{r\cos(\psi_t-\phi)}{g_H H_{bev}}.
\end{aligned}
$$

这里下标 W/H 比 x/y 更不容易误导：`shift_x` 在源码中加到 reference point 的第 0 维（BEV 图的 W/column 方向），`shift_y` 加到第 1 维（H/row 方向）。它们是 **TSA 采样栅格的坐标偏移**，不能直接理解为 global 坐标中的 `(Δx_G,Δy_G)`。

例如用户常会设想：

~~~text
ψ_t = 0°
Δp_G = (Δx_G,Δy_G) = (10m,10m)
r = 10√2
φ = 45°
~~~

直接按源码代入：

~~~text
bev_angle = ψ_t - φ = -45°
shift_x（W 方向）= r·sin(-45°) = -10m
shift_y（H 方向）= r·cos(-45°) = +10m
~~~

所以代码输出的粗平移是 `(-10m,+10m)`（再除以 102.4m 变成归一化 reference point 偏移），**不是** `(10m,10m)`。这个结论由源码确定，不能因为变量名叫 `shift_x/shift_y` 而改写。

这里必须避免把 **`shift` 的源码公式** 误当成 BEV 网格本身的物理轴定义。`get_reference_points()` 已经明确规定：BEV 的 `W/column` 对应 LiDAR `x`（前方），`H/row` 对应 LiDAR `y`（左方）。因此，若先把 global 位移按通常的右手系变换到当前 ego/LiDAR 平面，标准局部物理位移应为：

$$
\begin{bmatrix}\Delta x_E\\\Delta y_E\end{bmatrix}
=R(-\psi_t)
\begin{bmatrix}\Delta x_G\\\Delta y_G\end{bmatrix}
=
\begin{bmatrix}
\cos\psi_t&\sin\psi_t\\
-\sin\psi_t&\cos\psi_t
\end{bmatrix}
\begin{bmatrix}\Delta x_G\\\Delta y_G\end{bmatrix}.
$$

它若直接写入 BEV 网格，本应是 `(ΔW,ΔH)=(Δx_E,Δy_E)`。但这份 BEVFormer 复现代码实际计算的是：

$$
(\Delta W,\Delta H)=(-\Delta y_E,\ \Delta x_E).
$$

也就是相对“LiDAR `(x,y)` → BEV `(W,H)`”的直接映射再转了 $90^\circ$。所以对 `ψ_t=0°`、global 位移 `(10,10)m`，源码才会给出 `(-10,+10)m`。这是该 checkpoint 对应实现中 `can_bus`/`shift` 的既有约定；文档不能因此倒推出“BEV 的 H 是前方、W 是右方”。分析或修改这一段时应以源码的精确矩阵结果为准，并通过固定场景或单点测试验证目标数据的 pose 定义。

因此判断平移或旋转“是否取反”时，必须先写清正在谈的是哪一个坐标：

| 坐标 | 含义 | 能否直接与 `shift` 比较 |
|-|-|-|
| global `(Δx_G,Δy_G)` | 数据集世界坐标差 | 否 |
| ego/LiDAR `(forward,left)` | 车辆局部物理坐标 | 仍需转换到栅格 W/H |
| TSA reference `(W,H)` | `[0,1]` 归一化 BEV 图 column/row 坐标 | 是，`shift` 就在此坐标系 |

###### 7.2.2 把源码公式展开为矩阵：任意 yaw 下到底怎样换轴

将 $r\sin(\psi_t-\phi)$、$r\cos(\psi_t-\phi)$ 展开，并用
$\Delta x_G=r\cos\phi$、$\Delta y_G=r\sin\phi$ 代换，可得到源码在除网格尺度前的精确形式：

$$
\begin{bmatrix}
\Delta W\\
\Delta H
\end{bmatrix}
=
\begin{bmatrix}
\sin\psi_t & -\cos\psi_t\\
\cos\psi_t & \ \sin\psi_t
\end{bmatrix}
\begin{bmatrix}
\Delta x_G\\
\Delta y_G
\end{bmatrix}.
$$

然后才转成 reference point 的归一化偏移：

$$
\mathrm{shift}=
\left[
\frac{\Delta W}{g_W W_{bev}},
\frac{\Delta H}{g_H H_{bev}}
\right].
$$

这比极坐标写法更方便做 sanity check：

| 当前绝对 yaw $\psi_t$ | 源码得到的未归一化 `(ΔW,ΔH)` |
|-|-|
| $0°$ | $(-\Delta y_G,\ \Delta x_G)$ |
| $90°$ | $(\Delta x_G,\ \Delta y_G)$ |
| $180°$ | $(\Delta y_G,-\Delta x_G)$ |
| $-90°$ | $(-\Delta x_G,-\Delta y_G)$ |

因此在 `yaw=0°` 时，global 向北/全局 y 正向的移动不是写入 W 正向，而是写入 W 负向；global x 正向移动写入 H 正向。这个表描述的是**这段代码计算的结果**，不是我们主观选择的更直觉坐标定义。

###### 7.2.3 平移的符号为什么是“当前中心在历史图里”的方向

训练队列中保存的是：

$$
\Delta\mathbf{p}^G=\mathbf{p}_{t}^{G}-\mathbf{p}_{t-1}^{G}.
$$

即“当前 ego 原点相对上一 ego 原点去了哪里”。TSA 的目标恰好是：对于当前 cell，去上一帧 memory 中读取同一世界位置。因此它使用这个 current-minus-previous 位移来构造历史采样位置，而不是保存相反的 $\mathbf{p}_{t-1}^{G}-\mathbf{p}_{t}^{G}$。

不过源码不会把物理位移直接加到一个三维 LiDAR 坐标上；它先按上表旋转/换轴为 `(ΔW,ΔH)`，再加到归一化 `ref_2d`。所以判断“应不应该取负”必须分两层：

~~~text
物理层：当前中心在上一帧坐标中，使用 current - previous 的相对位移。
栅格层：该位移必须按源码的 (global x,y) → (BEV W,H) 规则换轴，符号由上表决定。
~~~

两层混在一起，就会出现“物理上应该是 +10,+10，代码为何得到 -10,+10”的表面矛盾。

###### 7.2.4 `sin/cos` 的常见误读：它不只是数组行列方向造成的等价写法

这一段常被解释成：“`bev_angle=ψ_t-φ` 取了常见局部角度的相反数，因此把 `sin`、`cos` 交换后，仍等价于前方用 cos、左方用 sin。”在本项目已经确认的 BEV 映射下，这个结论**不成立**。

令位移在当前车体右手平面中的标准方向角为：

$$
\beta=\phi-\psi_t.
$$

那么标准的当前 ego/LiDAR 平移分量是：

$$
\begin{bmatrix}\Delta x_E\\\Delta y_E\end{bmatrix}
=
\begin{bmatrix}r\cos\beta\\r\sin\beta\end{bmatrix},
$$

其中 `x_E` 为前方、`y_E` 为左方。因为 `W→x_E`、`H→y_E`，若把物理平移直接写入 `ref_2d[..., (W,H)]`，应当是：

~~~text
direct_shift_W = r * cos(φ - ψ_t)     # 前方 x_E
direct_shift_H = r * sin(φ - ψ_t)     # 左方 y_E
~~~

而源码实际是：

$$
\begin{aligned}
\Delta W_{code}
 &=r\sin(\psi_t-\phi)=-r\sin(\phi-\psi_t)=-\Delta y_E,\\
\Delta H_{code}
 &=r\cos(\psi_t-\phi)= r\cos(\phi-\psi_t)= \Delta x_E.
\end{aligned}
$$

即：

$$
\begin{bmatrix}\Delta W_{code}\\\Delta H_{code}\end{bmatrix}
=
\begin{bmatrix}0&-1\\1&0\end{bmatrix}
\begin{bmatrix}\Delta x_E\\\Delta y_E\end{bmatrix}.
$$

它相对直接的 `(x_E,y_E) → (W,H)` 映射是一个 $+90^\circ$ 变换，而不是单纯因为二维数组 `h` 向下增长产生的上下镜像。数组显示方向会影响人眼判断“顺/逆时针”，但不会改变 `ref_2d` 数值第 0 维就是 W、第 1 维就是 H 的事实。

因此应把下列三件事分开：

| 层次 | 结论 |
|-|-|
| BEV 存储 | `ref_2d[...,0]` 是 W，对应 LiDAR x；`ref_2d[...,1]` 是 H，对应 LiDAR y。 |
| 人眼显示 | H 行号向下增长，导致物理正 yaw 在数组画面中看起来顺时针。 |
| 本仓库 `shift` 源码 | 计算出 `(-Δy_E, Δx_E)` 后加到 `(W,H)`；这是必须保留和单独验证的实现约定。 |

![标准局部位移与源码 shift 的 90 度关系](../docs/bev_shift_comparison.svg)

所以，不能仅凭“它考虑了 BEV 数组存储方向”就断言该 `shift` 与标准物理局部位移完全等价。若未来重写时序对齐，必须用已知 pose、单个 BEV 热点和预期 source cell 做单元测试；直接把 `sin` 改成 `cos` 或交换 `shift_x/shift_y` 会改变已有 checkpoint 的行为。

当前配置的 BEV 覆盖范围为：

~~~
x ∈ [-51.2, 51.2] m，W_bev = 200
y ∈ [-51.2, 51.2] m，H_bev = 200
~~~

所以每格物理大小为：

$$
g_x = \frac{102.4}{200} = 0.512\ \text{m/cell},
\qquad
g_y = \frac{102.4}{200} = 0.512\ \text{m/cell}
$$

代码将米单位位移先换成 cell 数，再换成 `[0,1]` 归一化 reference point 位移：

$$
\mathrm{shift}_x =
\frac{r\sin(\alpha)}{g_x W_{bev}},
\qquad
\mathrm{shift}_y =
\frac{r\cos(\alpha)}{g_y H_{bev}}
$$

对应实现：

~~~
shift_y = r * cos(α) / grid_length_y / H_bev
shift_x = r * sin(α) / grid_length_x / W_bev
shift = [shift_x, shift_y]             # [B,2]
~~~

由于 `g_x W_bev=g_y H_bev=102.4m`，若车辆沿某一 BEV 轴移动 `1.024m`，最终归一化偏移约为 `0.01`，即 `0.01×200=2` 个 BEV cell。`bev_queries.new_tensor(...)` 使 `shift` 自动与 BEV token 同 device、同 dtype，形状为：

~~~
shift: [B,2]
shift[b] = [shift_x[b], shift_y[b]]
~~~

最终坐标系不是相机像素坐标，也不是 global 坐标，而是**当前帧 ego/LiDAR 平面上的归一化 BEV 网格坐标**：

~~~
x_norm ∈ [0,1]，沿 W_bev=200 个格子
y_norm ∈ [0,1]，沿 H_bev=200 个格子
~~~

该 shift 在 encoder 中只加给历史二维 reference point：

~~~
ref_2d:           [B,L_bev,1,2]
shift_ref_2d = ref_2d + shift[:,None,None,:]
                [B,L_bev,1,2]
~~~

因此 TSA 会从“粗平移对齐后”的历史 BEV 局部区域采样，而不是直接把整个 `prev_bev` 做平移。

##### 7.3 平移、旋转和 CAN embedding 是三条互补通路

运动信息以三种不同方式进入模型：

1. **平移对齐**：本节的 `shift=[B,2]` 加到历史 `ref_2d`，改变 TSA 的历史采样位置；`use_shift=False` 时直接乘零关闭。
2. **旋转对齐**：若 `prev_bev` 存在，代码用 `can_bus[-1]=Δψ_t°` 将历史 BEV 从前一车体朝向旋转到当前朝向。旋转输入 feature map，平移输入 reference point，两者职责不同。
3. **全局运动条件**：完整 18 维 `can_bus` 经 `18→128→256→LayerNorm` 的 MLP 得到 `[1,B,256]`，broadcast 加到每个当前 BEV query `[L_bev,B,256]`。

所以时序对齐不是一个单独的刚体矩阵乘法，而是当前实现拆开的“旋转 feature + 平移采样坐标 + 运动条件 embedding”。随后 TSA 的可变形 offset 还会在这套粗对齐基础上学习局部残差对齐。

##### 7.4 历史 BEV 的旋转与平移：当前 cell 如何去上一帧图中取值

这一部分最容易混淆的原因是：我们习惯说“把上一帧 BEV 变到当前坐标系”，但代码并没有为每个当前 cell 显式计算一个完整的 `current → previous` 二维坐标矩阵。它把同一个刚体对齐问题拆为两次操作：

~~~text
1. 旋转：先把整张历史 feature map 转向当前车体朝向。
2. 平移：TSA 采样历史图时，将当前 cell 的 reference point 平移到它应从历史图读取的位置。
~~~

因此，TSA 的问题可以准确表述为：**对当前帧的一个 BEV cell，先找到它在“旋转后的历史 BEV 图”中的粗采样位置，再在其附近学习少量残差采样点。**

###### 7.4.1 理想的坐标关系：当前点应映射到上一帧哪个位置

记 global 坐标中的两帧 ego 原点为 $\mathbf{p}_{t-1}^G,\mathbf{p}_{t}^G$，yaw 为 $\psi_{t-1},\psi_t$。一个物理点在当前 ego/BEV 平面中的坐标为 $\mathbf{x}_t$。同一物理点满足：

$$
\mathbf{p}_{t}^G+R(\psi_t)\mathbf{x}_t
=\mathbf{p}_{t-1}^G+R(\psi_{t-1})\mathbf{x}_{t-1}.
$$

所以，要从上一帧 BEV 中读取当前位置 $\mathbf{x}_t$ 的证据，理论上应使用：

$$
\boxed{
\mathbf{x}_{t-1}
=R(-\psi_{t-1})(\mathbf{p}_{t}^G-\mathbf{p}_{t-1}^G)
+R(\psi_t-\psi_{t-1})\mathbf{x}_{t}}
$$

右侧有两部分：

| 理想变换项 | 物理意义 | 当前代码的近似实现 |
|-|-|-|
| $R(\psi_t-\psi_{t-1})\mathbf{x}_{t}$ | 两帧车体朝向不同，当前网格方向需换到上一帧方向 | 对整张历史 feature map 执行 `rotate(..., Δψ)` |
| $R(-\psi_{t-1})(\mathbf{p}_{t}^G-\mathbf{p}_{t-1}^G)$ | 两帧 ego 原点不同，当前 cell 在历史网格中有平移 | `shift_ref_2d = ref_2d + shift` |

这里的数学式用于说明“当前 cell 应去历史图哪里读”。实际代码使用的是图像 row/column 坐标、归一化 reference point 和 `torchvision.rotate` 的角度约定，因此不应把公式中的 x/y 轴或正负号逐字替换为代码；代码的 `sin/cos` 顺序与旋转符号已经适配该仓库的 BEV 网格约定。

###### 7.4.2 旋转究竟操作了什么

当 `prev_bev` 存在且 `rotate_prev_bev=True`（base 配置为 True）时，代码对 batch 内每个样本独立执行：

~~~text
prev_bev[:, i]                         [L,C]，L=200×200
  → reshape(200,200,C)                 [H_bev,W_bev,C]
  → permute(2,0,1)                     [C,H_bev,W_bev]
  → rotate(angle=can_bus[-1],
           center=[100,100])           [C,H_bev,W_bev]
  → permute(1,2,0).reshape(L,1,C)      [L,1,C]
  → 写回 prev_bev[:, i]                [L,C]
~~~

`can_bus[-1]` 是当前入选帧相对上一入选帧的 yaw 增量：

$$
\Delta\psi=\psi_t-\psi_{t-1}.
$$

**7.4.2.1 先分清物理右手系与数组的视觉方向**

当前项目的 BEV 物理平面仍是当前 `LIDAR_TOP` 的右手系：`x` 向前、`y` 向左、`z` 向上。它不是一张以“车辆前方朝屏幕上方”保存的图片。根据 `get_reference_points()`，第 `h` 行、第 `w` 列 cell 的中心是：

$$
x=x_{min}+\frac{w+0.5}{W}(x_{max}-x_{min}),\qquad
y=y_{min}+\frac{h+0.5}{H}(y_{max}-y_{min}).
$$

因此原始数组从上到下、从左到右打印时是：

~~~text
                 w 增大：LiDAR x 增大（后方 → 前方）
h=0    y 小：右侧       (-x,-y) ───────── (+x,-y)
  ↓
h 增大 y 大：左侧       (-x,+y) ───────── (+x,+y)
~~~

也就是说，`w` 向右确实对应物理 `+x`，`h` 向下也确实对应物理 `+y`；但从车辆上方沿 `+z` 向下看时，物理 `+y` 本应显示在纸面上方，而数组把它显示在下方。用屏幕的“右、上”作为二维视觉轴，物理点 $(x,y)$ 显示为 $(w,-h)$，可写成反射矩阵：

$$
S=\begin{bmatrix}1&0\\0&-1\end{bmatrix}.
$$

故物理右手平面中的正 yaw（从 `+x` 逆时针转向 `+y`）在数组画面里表现为**顺时针**。这不是 `h/y` 或 `w/x` 对应错了，而是二维数组的行号向下增大；数组画面相对从 `+z` 俯视的物理平面翻转了一次。

![BEV 网格索引和 LiDAR 物理坐标的关系](../docs/bev_coordinate_grid.svg)

**7.4.2.2 `rotate(+angle)` 的视觉效果与实际取样方向**

`torchvision.transforms.functional.rotate` 的正角度从人眼看会让**图像内容逆时针**旋转。它却不是把每个输出格子的值向前搬运，而是对输出格子做逆向重采样。忽略离散插值时，若视觉坐标中的正逆时针矩阵记作 $R_{vis}$，则：

$$
\operatorname{rotate}(F,+\alpha)(\mathbf q)
=F\bigl(R_{vis}(-\alpha)\mathbf q\bigr).
$$

上一小节的镜像关系意味着：

$$
S R_{phys}(\alpha) S^{-1}=R_{vis}(-\alpha).
$$

所以把 `rotation_angle=Δψ` 传给 `rotate` 后，虽然看到的历史 BEV 内容是视觉逆时针转动，**当前输出格子实际去原历史图读取的是物理正 $\Delta\psi$ 的位置**：

$$
\widetilde F_{t-1}(\mathbf x_t)
\approx F_{t-1}\bigl(R_{phys}(\Delta\psi)\mathbf x_t\bigr).
$$

这正好符合无平移时的理想关系 $\mathbf x_{t-1}=R_{phys}(\Delta\psi)\mathbf x_t$。例如 `ψ_t=0°`、`ψ_{t-1}=-30°`，则 `Δψ=+30°`；当前的正前方 cell 应到历史 BEV 的“前左方 $30°$”读值。该位置在原数组中是右下方；`rotate(prev_bev,+30°)` 的视觉内容虽然逆时针转动，但其逆采样恰好使当前正前方输出格从这个右下方源位置取值。

所以判断代码方向时不能只说“图看起来顺时针还是逆时针”，必须同时说清：

1. 比较的是物理右手系，还是 `w` 向右、`h` 向下的数组画面；
2. 说的是被旋转的**图像内容**，还是某个**输出位置在源图的取样位置**。

本实现以第二种语义完成 current cell 到历史 memory 的查询；这也是 `rotate` 的正角度看似和物理直觉相反、实际却能满足采样关系的原因。

![历史 BEV 的视觉旋转和逆采样关系](../docs/bev_rotate_inverse_sampling.svg)

旋转中心 `[100,100]` 是 200×200 BEV feature map 的中心附近。当前 BEV 范围关于 ego 原点对称（`[-51.2,51.2]m`），所以它对应自车附近的网格中心。该操作对 **C=256 个通道使用同一张二维几何变换**；它不混合通道、不重新编码图像，也不重新运行历史帧的 CNN。

需要特别区分“旋转图像”和“查询图像”的语义。`rotate(F_prev, Δψ)` 产生一张新图 $\widetilde F_{prev}$；若下式中的 $R_{vis}$ 明确指 **数组画面** 的旋转矩阵，则输出坐标的值来自输入图的反向采样位置：

$$
\widetilde F_{prev}(\mathbf{q})
\approx F_{prev}\bigl(R_{vis}(-\Delta\psi)(\mathbf{q}-\mathbf{c})+\mathbf{c}\bigr).
$$

其中 $\mathbf{c}$ 是旋转中心，$\mathbf{q}$ 是旋转后图上的数组位置。通过上一小节的 $S$ 变换，`R_vis(-Δψ)` 对应物理坐标中的 `R_phys(+Δψ)`；因此它与理想的 current-to-previous 采样关系一致。代码把“方向变化”吸收进了 memory 图本身，TSA 不必为每个 query 再显式乘一次旋转矩阵。

`rotate` 调用没有显式传入 interpolation、fill 或 expand 参数，实际插值与边界填充遵循当前环境安装的 torchvision 默认行为。因此它会带来栅格化/边界误差；不能把这一步理解成无误差的连续几何变换。它只在每次 `get_bev_features()` 开头执行一次，六层 encoder 复用同一份旋转后的历史 memory。

###### 7.4.3 平移究竟操作了什么

旋转完成后，代码**不**执行类似 `torch.roll(prev_bev, ...)` 的整图平移。它保留旋转后的 `prev_bev`，只移动 TSA 的历史 reference point：

~~~text
当前帧规则网格中心： ref_2d          [B,L,1,2]
当前相对前帧位移：   shift           [B,2]
历史槽粗采样中心：   ref_2d + shift  [B,L,1,2]
当前槽粗采样中心：   ref_2d          [B,L,1,2]
~~~

对 batch 样本 b、当前 cell q，令当前归一化网格中心为 $\mathbf{r}_{bq}$，由 7.2 得到的归一化平移为 $\mathbf{s}_b$。历史槽传给 TSA 的基准位置就是：

$$
\mathbf{r}_{bq}^{hist}=\mathbf{r}_{bq}+\mathbf{s}_{b}.
$$

TSA 再为每个 head h 和采样点 p 预测一个以 BEV cell 为单位的偏移 $\Delta\mathbf{o}_{bqhp}$，最终在旋转后的历史 memory 中读取：

$$
\mathbf{u}_{bqhp}^{hist}
=\mathbf{r}_{bq}+\mathbf{s}_{b}
+\frac{\Delta\mathbf{o}_{bqhp}}{(W_{bev},H_{bev})}.
$$

对应的当前槽没有 shift：

$$
\mathbf{u}_{bqhp}^{curr}
=\mathbf{r}_{bq}
+\frac{\Delta\mathbf{o}_{bqhp}}{(W_{bev},H_{bev})}.
$$

这正是“得到当前帧 reference point，在上一帧 BEV 图中找到位置”的实现：`ref_2d + shift` 给出粗位置，learned offset 校正 CAN bus、离散栅格、动态物体和建模误差。

###### 7.4.4 两步合在一起时，TSA 实际从原始历史图哪里读

令 $F_{t-1}$ 是旋转前的历史 feature map，$\widetilde F_{t-1}$ 是旋转后的历史 memory。历史槽的单个采样可以按数据流写成：

~~~text
先构造旋转 memory：
    F̃_{t-1} = Rotate(F_{t-1}, Δψ)

再在 F̃_{t-1} 上按当前 cell 的历史位置采样：
    feature = BilinearSample(F̃_{t-1}, ref_2d + shift + learned_offset)
~~~

结合上一小节的图像反向采样语义，它等价于从未旋转的历史图的大致位置读取：

$$
F_{t-1}\left(
R(-\Delta\psi)
\left[\mathbf{r}_{bq}+\mathbf{s}_b+
\frac{\Delta\mathbf{o}_{bqhp}}{(W_{bev},H_{bev})}-\mathbf{c}\right]
+\mathbf{c}
\right).
$$

这不是代码逐点显式计算的公式，而是“先 rotate memory、再 sample”的等效解释。它回答了两个常见误解：

1. **不是**直接把当前 query 的 feature 与未对齐的 `prev_bev[q]` 相加；当前 q 会去历史图的另一个位置读值。
2. **不是**只做平移；yaw 变化先由整图旋转处理，再由 reference point 的平移和可学习 offset 处理位置变化。

###### 7.4.5 单帧递推例子

设当前训练队列连续处理 `t-1 → t`，且当前帧车辆相对前帧有平移和左转：

~~~text
历史 encoder 输出 F_{t-1}
    │ rotate(F_{t-1}, Δψ) around [100,100]
    ▼
旋转后的历史 memory F̃_{t-1}
    │ 对当前 cell q，使用 ref_2d[q] + shift_t
    │ 并叠加 TSA 学习到的局部 offset
    ▼
从 F̃_{t-1} 的局部位置稀疏采样
    │ 与当前槽的采样结果做时间槽平均
    ▼
当前层 TSA 输出 → SCA 写入当前相机证据 → FFN
~~~

下一入选帧再把本帧 encoder 输出作为新的 `prev_bev`，重复同样过程。因此历史 3 帧并不是各自分别变换到当前帧；它们在 `obtain_history_bev()` 的递推中已逐步对齐并压缩为一张状态图，当前帧只对最近的这张状态图执行一次旋转和平移对齐。


#### 8. 两套 BEV reference point

BEVFormerEncoder 为同一 200×200 网格生成两种 reference point。

##### 8.1 TSA 使用的二维 ref_2d

每个 BEV cell 中心对应一个 [0,1] 内二维坐标：

~~~
ref_2d = [B,L_bev,1,2]
       = [B,40000,1,2]
~~~

有历史 BEV 时，历史 reference point 加上 shift，当前 reference point 保持原位置：

~~~
hybrid_ref_2d = [2B,40000,1,2]
             = [shifted history refs, current refs]
~~~

这里有一个容易误读、但会影响理解每层数据流的实现细节。`BEVFormerEncoder.forward()` 在进入 `for layer in self.layers` 前只构造一次 `prev_bev`：

~~~python
prev_bev = stack([history_bev, initial_bev_query])  # [2B,L,256]
~~~

因此第二个时间槽是加入 CAN embedding 后、**尚未经过任一 encoder layer 的初始当前 BEV query**。在第 k 层，传入 TSA 的 `query` 是第 k−1 层更新后的 `[B,L,256]`，它用于预测本层的 sampling offset 和 attention weight；但传入 TSA 的 `value` 仍是循环外固定的 `[history_bev, initial_bev_query]`。代码不会在每一层后把新的 query 再写回这条 2-queue。

这并不意味着各层没有逐层更新：每层 TSA/SCA/FFN 的输出都会成为下一层的 `query`。区别只在于，TSA 的可采样 value memory 是固定的两槽，而查询和采样参数逐层更新。

##### 8.2 SCA 使用的三维 ref_3d

每个 BEV cell 沿竖直柱均匀取 D=4 个高度点：

~~~
ref_3d = [B,D,L_bev,3]
       = [B,4,40000,3]
~~~

它们初始是归一化 x、y、z。point_sampling 按 point_cloud_range 变为真实 LiDAR 坐标：

~~~
x = x_norm × (x_max-x_min) + x_min
y = y_norm × (y_max-y_min) + y_min
z = z_norm × (z_max-z_min) + z_min
~~~

多高度锚点使一个 BEV cell 可以在图像中检查竖直柱不同高度的位置；这对车、人、锥桶等具有实际高度的目标是必要的。


#### 9. 3D reference point 投影到六相机

每个 meta 提供 lidar2img [N,4,4]。point_sampling 对每个 BEV cell、每个高度锚点、每个相机执行：

~~~
P_img_h = lidar2img × [x,y,z,1]^T
u = P_img_h[0] / P_img_h[2]
v = P_img_h[1] / P_img_h[2]
~~~

u,v 除以图像宽高后成为归一化像素坐标。仅保留：

~~~
depth > 0
0 < u < 1
0 < v < 1
~~~

输出为：

~~~
reference_points_cam: [N,B,L_bev,D,2]
bev_mask:             [N,B,L_bev,D]
~~~

`point_sampling` 内部的精确 shape 变换如下，其中 `D=4`、`N=6`：

~~~text
归一化柱点:  [B,D,L,3]
反归一化并补齐次项: [B,D,L,4]
交换高度和 batch 维: [D,B,L,4]
复制相机维并加列向量维: [D,B,N,L,4,1]
lidar2img 广播:            [D,B,N,L,4,4]
矩阵投影后:                [D,B,N,L,4]
取前两项并除深度:          [D,B,N,L,2]
相机维提前:                [N,B,L,D,2]
~~~

代码用 `img_metas[0]['img_shape'][0]` 的宽高把全部投影坐标归一化；该实现默认一个 batch 中各相机、各样本已经经过相同的图像变换，具有一致的处理后尺寸。`bev_mask` 是几何可见性掩码，并非数据增强 mask。某 BEV cell 位于某相机后方或其全部高度锚点都在画面外时，该相机不会向它提供图像 feature。


#### 10. 每层 BEVFormerLayer 的顺序

6 个 BEVFormerLayer 都执行：

~~~
TemporalSelfAttention
  → LayerNorm
  → SpatialCrossAttention
  → LayerNorm
  → FFN
  → LayerNorm
~~~

每一个 attention 和 FFN 都带残差连接。encoder 刚接收的 token-first 输入为：

~~~
[L_bev,B,256] = [40000,B,256]
~~~

进入 `BEVFormerLayer` 前 encoder 已把主 query 与 BEV 位置编码从 token-first 转为 batch-first：

~~~text
bev_query: [L,B,256] → [B,L,256]
bev_pos:   [L,B,256] → [B,L,256]
~~~

故 TSA 和 SCA 的实际 `query` 都是 `[B,L,256]`。`BEVFormerLayer` 结束后 encoder 不会立即转回 token-first；它直接返回 `[B,L,256]`。完整 `PerceptionTransformer.forward()` 随后执行 `bev_embed.permute(1,0,2)`，把它转为 detection decoder 使用的 token-first memory `[L,B,256]`。阅读其他调用点时，应以函数边界的实际约定为准，不能把 encoder 内部最初的 `[L,B,C]` 输入约定套用到其输出。

##### 10.1 TSA：Temporal Self-Attention

TSA 不直接读取 3 张历史原始图像。当前帧 TSA 只融合两个 BEV 状态：

~~~
历史状态：prev_bev（前三帧已经迭代压缩后的结果）
当前 query：当前 encoder layer 已更新到的 token
当前 value 槽：进入 encoder 前的初始当前 BEV query
~~~

###### TSA 接口中三个容易混淆的变量

`BEVFormerLayer` 调用 TSA 时写成 `attention(query, prev_bev, prev_bev, ...)`，因此形式参数看起来是：

~~~text
query: [B,L,C]     # 当前 encoder layer 的 BEV token
key:   [2B,L,C]    # 传入 prev_bev
value: [2B,L,C]    # 传入 prev_bev
~~~

但 `TemporalSelfAttention.forward()` **没有读取 `key`**；它只是为了保持 MMCV attention 的通用函数签名而存在。实际参与计算的是：

| 名称 | 实际内容 | 是否每层更新 |
|-|-|-|
| `query` | 当前层的 BEV token；加上 `bev_pos` 后用于预测 offset 和 weight | 是 |
| `value` | 由 `[B,2,L,C]` 展平而成的 `[2B,L,C]`，每个样本的历史、初始当前槽交替排列 | 否 |
| `key` | 与 value 同样传入，但 TSA 实现不使用 | 不适用 |

所以“历史 + 当前拼接成 2C query”更精确地写作：

~~~python
predictor_query = cat([value[:B], query_with_bev_pos], dim=-1)  # [B,L,2C]
~~~

它只是 `sampling_offsets` 和 `attention_weights` 两个预测器的输入；它不是 CUDA deformable-attention 算子的 query/value memory。后者的可采样 memory 始终是 `value=[2B,L,C]`。

###### 10.1.1 `2B → B`：两次聚合与实际内存顺序

encoder 用以下方式构造 TSA 的 value queue：

~~~python
queue = torch.stack([history_bev, initial_bev_query], dim=1)  # [B,2,L,C]
value = queue.reshape(2 * B, L, C)                            # [2B,L,C]
~~~

这里的 `2B` 表示“B 个样本 × 2 个时间槽”，不是数据加载 batch 真正扩大了一倍。按 PyTorch 的连续行优先布局，展平后的索引严格为：

~~~text
value[0] = (sample 0, history)
value[1] = (sample 0, current)
value[2] = (sample 1, history)
value[3] = (sample 1, current)
...
~~~

`hybird_ref_2d` 使用相同的 `stack(..., dim=1).reshape(...)`，所以第 i 条 value、采样位置和采样权重始终属于同一个 `(sample, time-slot)`。CUDA deformable-attention 在 batch 维不混合样本也不混合时间槽；它分别完成第一个聚合：

$$
\mathbf{o}_{b,t,q}=\sum_{h,l,p} a_{b,t,q,h,l,p}
\operatorname{BilinearSample}(\mathbf{V}_{b,t,h,l},\mathbf{s}_{b,t,q,h,l,p}),
$$

输出仍为 `[2B,L,C]`，只是每项已在各自的 feature level 和 P 个局部采样点上完成加权空间聚合。随后：

~~~text
[2B,L,C] → permute(1,2,0) → [L,C,2B]
         → view(L,C,B,2)  → [L,C,B,2]
~~~

由于这里的 `view(...,B,2)` 与最初 `(B,2)` 的展平顺序完全对应，最后一维 `0` 是历史槽、`1` 是当前槽。`mean(-1)` 才是第二个聚合：

$$
\mathbf{o}_{b,q}=\frac{\mathbf{o}_{b,history,q}+\mathbf{o}_{b,current,q}}{2}.
$$

它把独立采样后的两个时间槽结果合成为 `[B,L,C]`。

需要把实现限制与概念分开看。虽然上式正确描述了 `2B` 的采样和平均，当前 TSA 随后又用 `value[:bs]` 与当前 query 拼接来预测 offset/weight。对于本配置常用的 `samples_per_gpu=1`，`value[:bs]` 就是历史槽，语义正确；当 `B>1` 时，交替布局使 `value[:bs]` 不是“每个样本的历史槽集合”。因此这份原始实现隐含了单样本每 GPU 的假设；若要安全支持 `B>1`，应在拼接预测器输入前先恢复 `[B,2,L,C]`，显式取 `queue[:,0]` 作为历史槽。

###### TSA 的“先分、再合”总结

在当前配置 `B=1` 下，预测器 query 的构造是：

~~~text
history BEV [B,L,C] + current-layer query [B,L,C]
    └── 沿 C 维拼接 ──→ predictor_query [B,L,2C]
~~~

它同时预测两个时间槽各自的参数：

~~~text
sampling_offsets: [B,L,H,2,L_f,P,2]
attention_weights:[B,L,H,2,L_f,P]
                    ↓ 将时间槽维移入 batch
sampling_offsets: [2B,L,H,L_f,P,2]
attention_weights:[2B,L,H,L_f,P]
value:            [2B,L,H,C/H]
~~~

三者的第 0 维都指向同一个 `(sample, time-slot)`，因此 CUDA 算子的第一次聚合完全在槽内进行：历史槽只从历史 BEV 中读取，当前槽只从当前 BEV 中读取；二者不会在这一步相互采样或相互加权。每个槽的输出是 `[L,C]`，合起来仍是 `[2B,L,C]`。

第二次才跨时间槽融合：

~~~text
[2B,L,C] → [L,C,B,2] → mean(time_slot) → [B,L,C]
~~~

因此 `mean(-1)` 的语义就是：对同一 BEV cell、同一通道，将“历史槽经过局部可变形采样后的结果”和“当前槽经过局部可变形采样后的结果”取等权平均。随后才经过 `output_proj → dropout → residual addition`。

num_bev_queue 固定为 2。若有 prev_bev，encoder 组织：

~~~
value = [2B,L_bev,256] = flatten([B,2,L_bev,256])
      = [hist_0,curr_0,hist_1,curr_1,...]
~~~

若没有历史，TSA 内部把当前 query 复制为两份 value；接口相同，但这时不包含真实历史信息。

在当前配置 `B=1` 下，TSA 把“历史 value”和当前 query 拼为 512 维输入，预测每个 query 的可变形采样 offset 和权重；`B>1` 的布局限制见上面的 10.1.1。

~~~
sampling_offsets: [B,L_bev,heads,2,levels,points,2]
attention_weights:[B,L_bev,heads,2,levels,points]
~~~

本配置为 8 heads、1 个 BEV level、每 head 4 个点、2 个 BEV 状态。历史 reference point 已经被 shift，历史 BEV feature 已经被 yaw_delta 旋转，因此采样在粗对齐后的局部范围继续学习残差对齐。

注意 `attention_weights` 的归一化维度。它先被 reshape 成：

~~~text
[B,L,H,2,L_f×P]
~~~

随后执行 `softmax(-1)`。因此每个 head 的**历史槽**和**当前槽**各自在 `L_f×P` 个采样点上归一化；两个时间槽不在同一个 softmax 分母中竞争。之后才把 queue 维移到 batch 维：

~~~text
offset: [B,L,H,2,L_f,P,2] → [2B,L,H,L_f,P,2]
weight: [B,L,H,2,L_f,P]   → [2B,L,H,L_f,P]
value:  [2B,L,C]          → [2B,L,H,C/H]
~~~

对第 q 个 BEV token、头 h、时间槽 t、采样点 p，位置为：

$$
\mathbf{s}_{qhtp}=
\mathbf{r}_{qt}+\frac{\Delta\mathbf{p}_{qhtp}}{(W_{bev},H_{bev})}.
$$

其中历史槽的 $\mathbf{r}_{qt}$ 是 `ref_2d + shift`，当前槽是原始 `ref_2d`。CUDA deformable-attention 算子分别从两个槽的 BEV grid 做双线性采样，产生 `[2B,L,C]`；代码恢复为 `[L,C,B,2]` 后沿最后一维直接取算术平均。也就是说，两个槽各自的采样点权重会先归一化，**槽间融合本身固定为 1/2 与 1/2**。

最后实际执行顺序为：

~~~text
[2B,L,C] → permute → [L,C,2B]
         → view     → [L,C,B,2]
         → mean     → [L,C,B]
         → permute  → [B,L,C]
         → output_proj → dropout → + identity
~~~

`identity` 是加位置编码之前的当前层输入 `[B,L,C]`，因此残差表达的是“当前层原 token + TSA 产生的时序增量”。

可变形注意力输出两个队列的结果 [2B,L_bev,256]。实现沿队列维求平均，接 output projection、dropout 和残差，得到：

~~~
TSA output: [B,L_bev,256]
~~~

TSA 的作用是让过去看见的证据和当前 query 对齐融合；它不会显式追踪目标 ID，但通过对齐后的 BEV state 累积时序信息。

##### 10.2 SCA：Spatial Cross-Attention

普通全局 cross-attention 要让 40,000 个 BEV query 与约 6×30,825 图像 token 两两计算，代价过高。SCA 的约束是：每个 BEV cell 只在它投影到可见相机的像素附近取少量特征。

第一步，按每个相机的 bev_mask 筛出可见 query。每台相机可见 query 数不同，故 padding 到 max_len：

~~~
queries_rebatch:          [B,N,max_len,256]
reference_points_rebatch: [B,N,max_len,D,2]
~~~

当前实现还有一个隐藏的 batch 假设：`SpatialCrossAttention` 用每个相机的 `bev_mask[i, 0]`（batch 第 0 项）生成可见 query 下标，再把同一套下标复用于 batch 内所有样本。NuScenes 的固定相机布置和统一图像处理通常使这个假设成立；但若改造为 batch 内相机内参、外参或裁剪方式都可能不同的数据，不能直接沿用这段重组代码，应按每个 `j` 单独生成下标和 padding。

第二步，将相机维并入 batch：

~~~
图像 key/value: [N,S,B,256] → [B×N,S,256]
BEV query:      [B,N,max_len,256] → [B×N,max_len,256]
~~~

每个相机独立调用 MSDeformableAttention3D。参数是 8 heads、4 FPN levels、8 个总采样点、4 个高度锚点。8 个点会按高度锚点重组，因此每个高度锚点在每尺度获得 2 个可学习偏移采样位置。

每个采样位置为：

~~~
projected reference point
  + learned offset / feature-map width-height
~~~

底层 CUDA deformable-attention 算子从四层图像 feature 做双线性采样并按权重加和，得到：

~~~
per-camera feature: [B,N,max_len,256]
~~~

第三步，将每个相机结果 scatter 回完整 40,000 个 BEV token。一个 BEV cell 可能同时被多个相机看到，SCA 对可见相机的输出求和，再除以可见相机数：

~~~
slots[q] = sum(camera_output_i[q]) / visible_camera_count(q)
~~~

最后 output projection、dropout、残差连接：

~~~
SCA output: [B,L_bev,256]
~~~

SCA 是多视角图像真正写入 BEV 的核心位置：几何投影决定从哪里看，deformable offsets 决定在附近精确取哪里，attention weight 决定各尺度、各高度和各采样点的贡献。

###### SCA 内层 `MSDeformableAttention3D` 如何使用高度锚点

SCA 已经将每个 BEV 柱的 `D=4` 个 3D 锚点投影为一个相机中的二维坐标：

~~~text
reference_points: [B×N,max_len,D,2]
~~~

内层对每个 `(batch, camera)` 独立运行。它先将该相机的四层 FPN token 投影并分头：

~~~text
value: [B×N,S,256] → [B×N,S,8,32]
offset: [B×N,max_len,8,4,8,2]
weight: [B×N,max_len,8,4,8]
~~~

这里第二个 `8` 是每个 head、每个 FPN level 的总采样点数。它并不表示“一个高度锚点有 8 个点”。代码将其拆成：

~~~text
8 total points = 2 points per height anchor × 4 height anchors

offset: [B×N,max_len,H,L_f,8,2]
     → [B×N,max_len,H,L_f,2,D=4,2]
reference point:
       [B×N,max_len,D,2]
     → [B×N,max_len,1,1,1,D,2]
~~~

二者广播相加，得到 `[B×N,max_len,H,L_f,2,D,2]`，再展平回 `[B×N,max_len,H,L_f,8,2]` 交给 CUDA 算子。对图像 level $l$，采样位置公式是：

$$
\mathbf{s}_{qhlpd}=
\pi_l(\mathbf{x}_{qd}^{bev})+
\frac{\Delta\mathbf{p}_{qhlpd}}{(W_l,H_l)}.
$$

$\pi_l(\mathbf{x}_{qd}^{bev})$ 是第 d 个 BEV 柱高度锚点投影到当前相机后的归一化像素位置；偏移预测以 feature-map 像素为单位，除以 `(W_l,H_l)` 后与该归一化位置相加。每个 head 的 attention weight 在 `4 FPN levels × 8 points` 上执行 softmax。

内层只返回单相机结果 `[B×N,max_len,256]`，没有自己的 output projection。SCA 将其 reshape 成 `[B,N,max_len,256]`，scatter 回 `[B,L,256]` 的 `slots`，对相机贡献求和并按可见相机数平均，最后才执行外层的 `output_proj`、dropout 和残差连接。


#### 11. 最终 BEV 表征

每层依次融合历史 BEV、当前多相机证据，再用 FFN 更新通道。六层完成后：

~~~
BEVFormerEncoder 输出 bev_embed = [B,L_bev,256] = [B,40000,256]
~~~

它可 reshape 为 `[B,200,200,256]`：200×200 个鸟瞰 cell，每个 cell 是一个融合当前视觉和历史时序的 256 维向量。完整 Transformer 将它从 encoder 的 batch-first `[B,L,256]` 转置为 detection decoder 的 token-first memory/value `[L,B,256]`。

在时序路径中，下一次 `get_bev_features()` 若接到 encoder 的 batch-first `prev_bev=[B,L,256]`，会先识别 `shape[1] == L_bev`，再转成内部构造 2-queue 前需要的 token-first `[L,B,256]`。所以同一个 BEV state 会在不同函数边界采用两种布局；判断时应看具体调用点，而不要只凭变量名推断 shape。

---


### 第三部分：检测头与 3D 检测结果


#### 12. 900 个 object query 和初始 3D reference point

检测不直接在 40,000 个 BEV cell 上输出 anchor，而使用 Q=900 个可学习 object query。

query_embedding 的形状为 [900,512]，被一分为二：

~~~
query_pos: [900,256]  # query 身份和位置 embedding
query:     [900,256]  # query 内容 embedding
~~~

复制到 batch 后，Linear(query_pos) 接 sigmoid 生成：

~~~
initial reference_points: [B,900,3]
~~~

每个 query 都有一个可学习的归一化 x,y,z 初始位置。它们不是 GT anchor、RPN proposal 或 LiDAR 先验。

格式转为 token-first：

~~~
query: [900,B,256]
query_pos: [900,B,256]
BEV memory: [40000,B,256]
~~~


#### 13. 一层 detection decoder

base 的 6 层 DetrTransformerDecoderLayer 按顺序执行：

~~~
object-query self-attention
  → LayerNorm
  → query-to-BEV cross-attention
  → LayerNorm
  → FFN
  → LayerNorm
~~~

第一步 MultiheadAttention 只让 900 个 object query 相互交流。它不读取图像、不直接读取 BEV，有助于 query 之间协调和去重。

第二步 CustomMSDeformableAttention 以 BEV memory 为 value：

~~~
query:            [900,B,256]
BEV value:        [40000,B,256]
reference_points: [B,900,1,2]
spatial_shapes:   [[200,200]]
~~~

每个 object query 的 x,y reference point 指向 BEV 平面位置。cross-attention 预测局部 4 个采样 offset 和权重，从 reference point 附近读取少量 BEV token；它不对 900×40,000 对全部计算注意力。

**这里的三维 reference point 要和 encoder 的 `ref_3d` 区分。**检测 decoder 虽保存并逐层细化 `[B,900,3]` 的 `(x,y,z)`，但传给 cross-attention 前，代码在 `DetectionTransformerDecoder.forward()` 中明确执行：

~~~python
reference_points_input = reference_points[..., :2].unsqueeze(2)
## [B,Q,3] → [B,Q,1,2]
~~~

所以 decoder 的可变形 cross-attention 只在单层 `200×200` **二维 BEV memory** 上按 `(x,y)` 稀疏取样；`z` 不会改变它从哪一个 BEV cell 读取 feature。`z` 仍然有两项用途：它与 `x,y` 一起作为 3D box 的中心参考点，被当前层回归分支更新；并在最后将归一化预测还原到 `pc_range` 的真实高度范围。BEV 的 cell feature 已经通过前面的 SCA 汇聚了不同高度锚点的图像证据，因此检测 decoder 不再保存显式的 z 轴 feature map。

相反，encoder 的 SCA 使用的 `ref_3d=[B,D,L_bev,3]` 的 z 是实际几何量：同一 BEV cell 的 `D=4` 个不同高度点都会经 `lidar2img` 投影为不同的图像 `(u,v)`，随后分别参与图像特征采样。两者的区别如下：

| 位置 | reference point | z 是否影响采样位置 | 采样对象 |
|-|-|-|-|
| encoder SCA | `[B,4,L_bev,3]`，BEV 柱的 4 个高度锚点 | 是；z 改变投影后的图像像素 `(u,v)` | 4 层、6 相机的二维图像 feature |
| detection decoder | `[B,900,3]`，object query 的 3D 中心参考 | 否；cross-attention 只传 `[..., :2]` | 单层 `200×200` 二维 BEV memory |


#### 14. 层间 box refinement

with_box_refine=True，因此每个 decoder layer 有独立 regression branch。第 l 层从 query feature 产生 tmp，并更新 x,y,z reference point：

~~~
new_ref_xy = sigmoid(tmp_xy + inverse_sigmoid(old_ref_xy))
new_ref_z  = sigmoid(tmp_z  + inverse_sigmoid(old_ref_z))
~~~

更新后的 reference point 会 detach 后交给下一层。下一层以新的位置采样 BEV memory；detach 使后续层的采样位置梯度不穿过上一层 refinement 链，但每层仍通过自己的辅助 loss 学习。

decoder 返回：

~~~
hs:               [6,900,B,256]
inter_references: [6,B,900,3]
init_reference:   [B,900,3]
~~~


#### 15. 分类和 3D box 分支

head 将 hs 改成 [6,B,900,256]。第 l 层对应独立 cls_branch 和 reg_branch：

~~~
classification: [B,900,256] → [B,900,10]
regression:     [B,900,256] → [B,900,10]
~~~

分类是 10 类 logits，尚未 sigmoid。回归的中心相关分量利用当前 reference point：

~~~
cx = sigmoid(tmp[0] + inverse_sigmoid(ref_x)) → [-51.2,51.2]
cy = sigmoid(tmp[1] + inverse_sigmoid(ref_y)) → [-51.2,51.2]
cz = sigmoid(tmp[4] + inverse_sigmoid(ref_z)) → [-5.0,3.0]
~~~

完整 10 维训练编码为：

~~~
(cx, cy, log w, log l, cz, log h, sin yaw, cos yaw, vx, vy)
~~~

所有 decoder 层堆叠为：

~~~
all_cls_scores: [6,B,900,10]
all_bbox_preds: [6,B,900,10]
~~~

as_two_stage=False，所以 enc_cls_scores 和 enc_bbox_preds 为 None；整个检测过程由 learnable 900 queries 驱动。


#### 16. 训练输出与推理输出

训练时，当前帧 GT 与 6 层预测计算 Hungarian matching、Focal classification loss 与加权 L1 box loss；head 返回 loss dict。详细公式见 docs/training_overview.md。

推理时只使用最后一层：

~~~
all_cls_scores[-1]: [B,900,10]
all_bbox_preds[-1]: [B,900,10]
~~~

NMSFreeCoder 对 900×10=9000 个 query-class 分数做 sigmoid，选全局 top-300。随后：

~~~
log w/log l/log h → exp
sin yaw/cos yaw → atan2
按 post_center_range 过滤
z 重心形式 → box 底面中心形式
~~~

最终每个样本返回：

~~~
boxes_3d:  LiDARInstance3DBoxes
scores_3d: Tensor[K]
labels_3d: Tensor[K]
~~~

该 coder 不执行 NMS；NMS-free 的含义是直接以 query 分数排序和空间范围过滤。

---


### 第四部分：回顾、形状总表与调试入口


#### 17. 训练和推理的时序差异

| 模式 | prev_bev 从哪里来 | 何时更新/重置 |
|-|-|-|
| 训练 | 同一 Dataset 样本中的 3 个历史图像 | obtain_history_bev 逐帧构造；prev_bev_exists=False 时清空 |
| 推理/验证 | 检测器 self.prev_frame_info 缓存 | 每次 forward_test 后写回；scene_token 变化或 video_test_mode=False 时清空 |

二者共享“当前帧图像 → 特征 → BEV encoder → detection decoder”的主干。不同点只是训练由 Dataset 提供有限历史帧，推理由模型对象跨调用维护状态。


#### 18. 当前帧一次前向的形状总表

典型输入为 928×1600、B=1：

| 阶段 | 张量 | 形状 |
|-|-|-|
| DataLoader | img | [1,4,6,3,928,1600] |
| 当前帧 | cur_img | [1,6,3,928,1600] |
| CNN 输入 | 相机维并入 batch | [6,3,928,1600] |
| FPN | 四层 feature | [1,6,256,116,200]、[1,6,256,58,100]、[1,6,256,29,50]、[1,6,256,15,25] |
| 图像 token | feat_flatten | [6,30825,1,256] |
| 初始 BEV | bev_queries | [40000,1,256] |
| 几何投影 | reference_points_cam | [6,1,40000,4,2] |
| BEV encoder 输出 | bev_embed | [1,40000,256] |
| object query | query | [900,1,256] |
| decoder 输出 | hs | [6,900,1,256] |
| 分类 | all_cls_scores | [6,1,900,10] |
| 回归 | all_bbox_preds | [6,1,900,10] |
| 推理解码 | boxes/scores/labels | 最多 300 个框 |


#### 19. 推荐的源码断点顺序

~~~text
BEVFormer.forward_train
  → obtain_history_bev
  → extract_img_feat
  → BEVFormerHead.forward
  → PerceptionTransformer.get_bev_features
  → BEVFormerEncoder.forward
  → BEVFormerLayer.forward
      → TemporalSelfAttention.forward
      → SpatialCrossAttention.forward
          → MSDeformableAttention3D.forward
  → PerceptionTransformer.forward 的 detection decoder 部分
  → DetectionTransformerDecoder.forward
  → cls/reg branches
  → BEVFormerHead.loss 或 NMSFreeCoder.decode
~~~

最有价值的调试输出是形状与时序开关：

~~~python
print(img.shape)
print([x.shape for x in img_feats])
print(prev_bev is None, img_metas[0]['prev_bev_exists'])
print(bev_embed.shape)
print(outs['all_cls_scores'].shape, outs['all_bbox_preds'].shape)
~~~

只要这几处符合本文的语义，便可确认完整模型链路正确执行：六相机二维特征提取 → 几何投影和时序融合形成 BEV → 900 个 query 从 BEV 读取局部证据 → 输出 3D 检测候选。


## 详细实现手册：F. 当前 batch、Hungarian matching、Loss、优化与验证

以下单元是从 `training_overview.md` 整理进 notebook 的完整细节快照。前面的教程单元负责建立主线；本节保留推导、边界条件、张量形状与源码定位，便于离线顺序阅读。


## BEVFormer 训练全流程总览：batch、预测、损失与 Runner

本文从一次训练任务的整体视角解释当前仓库的经典 BEVFormer 配置（以 `projects/configs/bevformer/bevformer_base.py` 为准）。只回答四件事：

1. 模型每次实际拿到的一个 batch 是什么；
2. 模型的输入与输出是什么；
3. 预测怎样与 GT 匹配，最终 loss 怎样计算；
4. 一次训练、一个 epoch、验证和 checkpoint 怎样被组织。

本文聚焦“模型收到 batch 后怎样得到监督 loss”。不解释 nuScenes 数据怎样转换成 PKL、字段怎样从原始数据产生，也不展开 backbone、BEV encoder、attention 或 Transformer 的内部网络结构。相关内容请分别阅读：

- `docs/nuscence_to_pkl.md`：原始 nuScenes 到 PKL；
- `docs/dataset.md`：PKL 到 Dataset/DataLoader batch；
- `docs/config_and_registry.md`：配置加载、插件和 Registry；
- `docs/framework_training_pipeline.md`：DataLoader、Runner、Hook 与优化器调度；
- 后续模型文档：网络内部的计算结构。

主要源码位置：

```text
tools/train.py
projects/mmdet3d_plugin/bevformer/apis/train.py
projects/mmdet3d_plugin/bevformer/apis/mmdet_train.py
projects/mmdet3d_plugin/bevformer/detectors/bevformer.py
projects/mmdet3d_plugin/bevformer/dense_heads/bevformer_head.py
projects/mmdet3d_plugin/core/bbox/assigners/hungarian_assigner_3d.py
projects/configs/bevformer/bevformer_base.py
```

---


### 1. 一张总图

```text
配置文件
  │  定义模型、数据、优化器、runner、评测频率
  ▼
tools/train.py
  │  加载配置、导入 plugin、构建 model 和 train Dataset
  ▼
custom_train_model → custom_train_detector
  │  构建 DataLoader、DDP、optimizer、EpochBasedRunner、hooks
  ▼
每一个 iteration
  │
  ├── DataLoader 给出一个 batch
  ├── model.train_step
  │     └── BEVFormer.forward_train
  │           ├── 历史帧：无梯度建立 prev_bev
  │           └── 当前帧：预测 Q=900 个 3D 检测候选
  ├── BEVFormerHead.loss
  │     ├── Hungarian matching：预测 query ↔ 当前帧 GT
  │     └── 所有 decoder 层的分类 loss + 框回归 loss
  ├── OptimizerHook：zero_grad → backward → grad clip → optimizer.step
  └── 日志 hook：每 50 iteration 记录 loss、lr、时间
  ▼
每个 epoch 结束
  ├── 保存 checkpoint（配置为每 1 epoch）
  └── 验证并计算 nuScenes mAP/NDS（配置为每 1 epoch）
```

最重要的边界是：**Dataset 的一个训练样本是一个时序队列；loss 的监督对象只有该队列最后一帧。** 历史帧提供时序上下文，不各自产生一份检测 loss。

---


### 2. 训练启动：配置如何变成运行对象

典型命令为：

```bash
python tools/train.py projects/configs/bevformer/bevformer_base.py
```

`tools/train.py` 按以下顺序完成准备：

1. `Config.fromfile()` 加载配置并合并 `_base_`；
2. 读取 `plugin=True`，导入 `projects.mmdet3d_plugin`，使 `BEVFormer`、`CustomNuScenesDataset`、`HungarianAssigner3D` 等自定义类注册；
3. 设置 work directory、随机种子、分布式环境；
4. `build_model(cfg.model, ...)` 创建 `BEVFormer`，并执行 `model.init_weights()`；
5. `build_dataset(cfg.data.train)` 创建训练 Dataset；
6. 调用 `custom_train_model(...)`，进入项目自定义训练入口。

`custom_train_model` 对检测模型调用 `custom_train_detector`。后者负责：

```text
训练 Dataset
  → 自定义 build_dataloader
  → train DataLoader

BEVFormer
  → MMDistributedDataParallel（多卡）或 MMDataParallel（单进程）
  → build_optimizer
  → build_runner
  → register hooks
  → runner.run(data_loaders, cfg.workflow)
```

经典配置的关键运行参数如下：

| 配置 | `bevformer_base.py` 的值 | 实际作用 |
|-|-|-|
| `samples_per_gpu` | 1 | 每张 GPU 每个 iteration 一个时序样本 |
| `workers_per_gpu` | 4 | 每个 DDP rank 的 DataLoader worker 数 |
| `queue_length` | 4 | 一个训练样本含 3 个历史帧和 1 个当前帧 |
| `runner` | `EpochBasedRunner`, 24 epochs | 以 epoch 为训练组织单位 |
| `optimizer` | AdamW, lr `2e-4` | 参数更新器 |
| `img_backbone.lr_mult` | 0.1 | backbone 使用 `2e-5` 学习率，其余参数 `2e-4` |
| `checkpoint_config.interval` | 1 | 每 epoch 保存一次 |
| `evaluation.interval` | 1 | 每 epoch 验证一次 |

---


### 3. 一个训练 batch 到底是什么


#### 3.1 单个 Dataset 样本

训练 Dataset 返回的是一个“当前帧容器”，其中 `img` 和 `img_metas` 已合并了时序队列，GT 则只保留当前帧。

设：

```text
T = queue_length = 4
N = 相机数 = 6
B = 当前 GPU 的 batch size = samples_per_gpu = 1
M_i = 第 i 个当前帧的 GT 目标数，因样本而异
```

一个样本的语义是：

```text
第 0、1、2 帧：历史图像 + 历史 meta，仅用于建立历史 BEV
第 3 帧：当前图像 + 当前 meta + 当前帧 GT，用于最终预测和 loss
```

Dataset 单样本的主要字段：

| 字段 | 单样本形态 | 模型中的用途 |
|-|-|-|
| `img` | Tensor `[T,N,C,H,W]` | 六相机图像的时序输入 |
| `img_metas` | `{0:meta_0, ..., T-1:meta_current}` | 相机投影、场景信息、CAN bus、帧间运动等辅助信息 |
| `gt_bboxes_3d` | 一个 `LiDARInstance3DBoxes`，含 `M_i` 个框 | 当前帧匹配和框回归目标 |
| `gt_labels_3d` | Tensor `[M_i]` | 当前帧分类目标，类别范围 0～9 |

在经典配置中，图像经过 resize/pad 后通常是 `[T,6,3,928,1600]`；分辨率取决于配置和 pipeline，不是模型接口的固定常数。


#### 3.2 DataLoader collate 和并行封装之后

MMCV 的 `DataContainer` 和并行包装器会把 batch 解包后送入模型。`BEVFormer.forward_train` 实际看到：

```text
img           Tensor [B, T, N, C, H, W]
img_metas     list，长度 B；每项是 {0:meta, ..., T-1:meta}
gt_bboxes_3d  list，长度 B；每项为 LiDARInstance3DBoxes
gt_labels_3d  list，长度 B；每项为 Tensor[M_i]
```

对于 base 配置的一张 GPU：

```text
img           [1, 4, 6, 3, 928, 1600]
img_metas     [{0: ..., 1: ..., 2: ..., 3: ...}]
gt_bboxes_3d  [LiDARInstance3DBoxes(M_0, box_dim=9)]
gt_labels_3d  [Tensor[M_0]]
```

`points`、`gt_bboxes`、`gt_labels` 等通用检测接口参数在该纯视觉训练 pipeline 中没有收集，因此为 `None` 或不参与计算。


#### 3.3 当前帧与历史帧的分离

`BEVFormer.forward_train` 首先做：

```python
len_queue = img.size(1)
prev_img = img[:, :-1, ...]  # [B, T-1, N, C, H, W]
img = img[:, -1, ...]        # [B, N, C, H, W]

prev_img_metas = copy.deepcopy(img_metas)
prev_bev = self.obtain_history_bev(prev_img, prev_img_metas)
img_metas = [each[len_queue - 1] for each in img_metas]
```

这段代码决定了整个训练的监督边界：

- 历史 `T-1` 帧依次得到 `prev_bev`，运行在 `torch.no_grad()` 中；它们不保留反向传播图；
- 当前帧单独前向，接收 `prev_bev`；
- `gt_bboxes_3d` 和 `gt_labels_3d` 从头到尾都是当前帧 GT，最终只监督当前帧。

如果当前帧的 `img_metas[0]['prev_bev_exists']` 为 False（例如跨 scene），当前帧会丢弃 `prev_bev`，变成没有历史上下文的预测。

---


### 4. 模型输入和输出：只看接口，不看内部架构


#### 4.1 `forward_train` 的输入和直接输出

训练前向的最终调用关系是：

```text
BEVFormer.forward_train(...)
  → forward_pts_train(pts_feats, gt_bboxes_3d, gt_labels_3d, img_metas, prev_bev)
  → pts_bbox_head(pts_feats, img_metas, prev_bev)
  → pts_bbox_head.loss(gt_bboxes_3d, gt_labels_3d, outs, img_metas)
  → losses dict
```

图像经过模型特征提取后，检测头得到 `outs`：

```python
outs = {
    'bev_embed':       Tensor,             # 供后续时序或调试使用
    'all_cls_scores':  Tensor [L,B,Q,C],
    'all_bbox_preds':  Tensor [L,B,Q,10],
    'enc_cls_scores':  None,
    'enc_bbox_preds':  None,
}
```

各维含义为：

| 符号 | base 配置值 | 含义 |
|-|-|-|
| `L` | 6 | decoder layer 数；每层都输出一组辅助监督预测 |
| `B` | 每 GPU 的 `samples_per_gpu`，通常 1 | 当前帧 batch 大小 |
| `Q` | 900 | 固定数目的 object queries / 候选预测 |
| `C` | 10 | nuScenes 检测类别数；是 sigmoid 分类 logits，没有单独 background logit |
| `10` | 10 | 回归编码 `(cx, cy, log w, log l, cz, log h, sin yaw, cos yaw, vx, vy)` |

`all_cls_scores` 是未 sigmoid 的分类 logits。`all_bbox_preds` 的中心坐标已被映射回 `point_cloud_range` 的物理坐标范围；宽、长、高仍以 log 尺寸编码，yaw 以正弦/余弦编码。


#### 4.2 训练输出和推理输出不同

训练的返回值不是最终 3D 框，而是 loss 字典：

```python
{
    'loss_cls': Tensor, 'loss_bbox': Tensor,
    'd0.loss_cls': Tensor, 'd0.loss_bbox': Tensor,
    ...,
    'd4.loss_cls': Tensor, 'd4.loss_bbox': Tensor,
}
```

推理时才调用 `get_bboxes` 与 `NMSFreeCoder.decode`：

1. 只取最后一层 `all_cls_scores[-1]`、`all_bbox_preds[-1]`；
2. 对每个 query、每个类别执行 sigmoid；
3. 从 `Q × C` 个分数中取全局 top-300；
4. 反变换 box 编码，并按 `post_center_range` 过滤；
5. 输出每个样本的 `boxes_3d`、`scores_3d`、`labels_3d`。

因此，训练阶段的 6 层预测都受监督；评测/提交阶段只使用最后一层，并且是 NMS-free top-k 解码。

---


### 5. GT 如何变成监督目标


#### 5.1 当前帧 GT 的统一表示

`gt_bboxes_3d` 是 `LiDARInstance3DBoxes`。loss 函数将每个框变成：

```text
(cx, cy, cz, w, l, h, yaw, vx, vy)
```

其中中心通过 `gt_bboxes.gravity_center` 取得，再接上原始 tensor 的第 3 列及之后内容。然后 `normalize_bbox` 改写为与预测一致的回归空间：

```text
(cx, cy, log w, log l, cz, log h, sin(yaw), cos(yaw), vx, vy)
```

这里的“normalize”不是把中心坐标缩放到 0～1；当前实现保留物理 `cx,cy,cz`，只将尺寸改为 log、角度改为 sin/cos。这样避免了 yaw 在 `-π/π` 边界的不连续问题。


#### 5.2 为什么需要 Hungarian matching

每张当前帧只有 `M` 个 GT，但模型总是输出 `Q=900` 个 query。没有 anchor 或 proposal 的预定义一对一对应关系，必须决定：

```text
900 个预测中，哪些分别负责哪一个 GT？
其余预测应当视为什么？
```

`HungarianAssigner3D` 对每一个 decoder layer、每一张当前图独立建立一个代价矩阵：

```text
cost 的形状：[Q, M]
第 q 行、第 m 列 = “预测 q 与 GT m 配对”的总代价
```

当前 base 配置的代价为：

```text
cost(q, m)
  = 2.0 × FocalLossCost(class_logit_q, gt_class_m)
  + 0.25 × L1(pred_box_q[:8], encoded_gt_box_m[:8])
```

其中前 8 个框编码分量是：

```text
cx, cy, log w, log l, cz, log h, sin yaw, cos yaw
```

调用 SciPy 的 `linear_sum_assignment` 求总代价最小的一对一匹配。若有 `M` 个 GT，最多选出 `M` 个正 query；其余 `Q-M` 个 query 都是 background。

匹配结果在实现中的约定：

```text
assigned_gt_inds[q] = 0       → background / negative
assigned_gt_inds[q] = m + 1   → positive，匹配 GT m
```


#### 5.3 当前代码中 IoU 配置的实际状态

配置虽然包含：

```python
iou_cost=dict(type='IoUCost', weight=0.0)
loss_iou=dict(type='GIoULoss', loss_weight=0.0)
```

但在当前项目代码中：

- `HungarianAssigner3D.__init__()` 的确构建了 `self.iou_cost`；
- `HungarianAssigner3D.assign()` 实际只相加 `cls_cost + reg_cost`，没有使用 `self.iou_cost`；
- `BEVFormerHead.loss_single()` 实际只计算 `loss_cls` 和 `loss_bbox`，没有调用 `self.loss_iou`。

所以对当前经典配置而言，IoU 既不影响匹配，也不贡献最终优化目标。它保留在配置中主要是 DETR 风格接口兼容/实验遗留，不能误认为训练中存在第三项 IoU loss。

---


### 6. 最终 Loss 怎样计算


#### 6.1 单 decoder layer、单个 batch 的目标

对某个 decoder layer，匹配完成后，构造：

- `labels`：长度 `B×Q`。正 query 为对应 0～9 类；未匹配 query 使用 background sentinel `num_classes=10`；
- `label_weights`：当前实现每个 query 都是 1；
- `bbox_targets`：只有正 query 写入已匹配 GT 的 box；
- `bbox_weights`：只有正 query 为 1，负 query 为 0。

因此分类 supervision 覆盖全部 `B×Q` 个 query，回归 supervision 只覆盖 Hungarian 匹配出的正 query。


#### 6.2 分类损失：sigmoid Focal Loss

配置是：

```python
loss_cls=dict(
    type='FocalLoss', use_sigmoid=True,
    gamma=2.0, alpha=0.25, loss_weight=2.0)
```

对某个类别二元目标 `y∈{0,1}` 和预测概率 `p=sigmoid(z)`，Focal Loss 的基本形式为：

```text
FL(p, y) = - α_t × (1 - p_t)^γ × log(p_t)

p_t = p       (y=1)
p_t = 1 - p   (y=0)
γ = 2.0
```

它降低大量“容易分类的背景 query”的影响，使少量困难正例和困难负例有更大梯度。框架实现再乘 `loss_weight=2.0`，并按 `cls_avg_factor` 归一化：

```text
cls_avg_factor = #positive + bg_cls_weight × #negative
```

本配置启用 `sync_cls_avg_factor=True`，所以这个分母会在所有 GPU 间 `all-reduce` 求平均，避免不同 rank 中 GT 数不同导致每卡 loss 标度不一致。


#### 6.3 框回归损失：加权 L1

配置是：

```python
loss_bbox=dict(type='L1Loss', loss_weight=0.25)
```

对所有正 query，先过滤含非有限值的 GT，再计算：

```text
L_bbox
  = 0.25 / max(mean_over_gpus(#positive), 1)
    × Σ_positive Σ_k code_weight[k] × |pred[k] - target[k]|
```

当前 `code_weight` 是：

```text
[1, 1, 1, 1, 1, 1, 1, 1, 0.2, 0.2]
```

即位置、尺寸和朝向的 8 个分量权重为 1，速度 `vx,vy` 的权重为 0.2。`loss_bbox` 自身再乘配置的 `loss_weight=0.25`。负 query 的 `bbox_weights=0`，不会贡献 L1 回归损失。


#### 6.4 多层辅助监督与总 Loss

base 配置的 decoder 有 `L=6` 层。`BEVFormerHead.loss()` 对每一层分别调用 `loss_single()`；每层都重新做 Hungarian matching，并得到一组 `loss_cls`、`loss_bbox`。

loss dict 的命名是：

```text
最后一层： loss_cls, loss_bbox
第 0～4 层：d0.loss_cls, d0.loss_bbox, ..., d4.loss_cls, d4.loss_bbox
```

MMDetection 的 `parse_losses` 将所有键名包含 `"loss"` 的项相加，所以真正反向传播的标量是：

```text
L_total = Σ_(l=0..5) [L_cls^(l) + L_bbox^(l)]
```

最后一层没有额外的更大权重；所有 decoder 层按各自已经配置好的 loss 权重参与总和。`loss_iou` 不在这个和中，原因见 5.3 节。


#### 6.5 一个 batch 的监督计数例子

设 `B=1`、当前帧有 `M=23` 个 GT：

```text
每个 decoder layer：
  900 个 query
  23 个 Hungarian 正匹配
  877 个 background query

分类 loss：900 个 query 都参与
框回归 loss：只由 23 个正 query 参与

6 个 decoder layers：
  上述过程重复 6 次
  最终累加 6 份分类 loss 与 6 份回归 loss
```

这说明 `num_query=900` 不代表模型每帧预测 900 个最终目标，也不代表有 900 个回归正样本；它是固定的候选槽位数。

---


### 7. 一个 iteration 如何更新参数


#### 7.1 Runner 与 `train_step`

base 配置使用：

```python
runner = dict(type='EpochBasedRunner', max_epochs=24)
workflow = [('train', 1)]
```

每个 epoch 中，runner 顺序迭代 train DataLoader。对每个 `data_batch`：

1. 并行包装器按 `DataContainer` 把 Tensor/列表送到本 rank 的设备；
2. 调用模型的 `train_step(data_batch, optimizer)`；
3. 检测器基类以 `return_loss=True` 调用 `BEVFormer.forward_train`；
4. 得到上节所列 loss dict；
5. 基类 `parse_losses` 计算 `L_total`、做跨 GPU 日志归约，并返回：

```python
{
    'loss': L_total,
    'log_vars': {'loss': ..., 'loss_cls': ..., ...},
    'num_samples': B
}
```

`EpochBasedRunner` 不理解 3D 检测、时序或 Hungarian matching；它只理解“模型给出一个可反传 loss”。BEVFormer 的时序逻辑完全在 `forward_train` 内，loss 逻辑完全在检测头内。


#### 7.2 OptimizerHook：反向传播和更新

默认 `OptimizerHook` 在每个 train iteration 执行等价流程：

```python
optimizer.zero_grad()
outputs['loss'].backward()
clip_grad_norm_(model.parameters(), max_norm=35, norm_type=2)
optimizer.step()
```

这里的梯度只沿当前帧前向图传播。历史帧处于 `torch.no_grad()`，故不会为历史帧特征保留计算图或得到梯度；不过当前帧会通过共享参数更新同一个网络。

配置的优化器为 AdamW：

```text
base lr       = 2e-4
weight decay  = 0.01
backbone lr   = 0.1 × base lr = 2e-5
gradient clip = L2 norm 上限 35
```

`tools/train.py` 还包含一个版本兼容分支：仅当 PyTorch 正好为 1.8.1 且 optimizer type 为 `AdamW` 时，配置会临时改成项目注册的 `AdamW2`。其他 PyTorch 版本正常使用框架的 AdamW。


#### 7.3 学习率调度

`lr_config` 定义：

```text
policy       = CosineAnnealing
warmup       = linear
warmup_iters = 500
warmup_ratio = 1/3
min_lr_ratio = 1e-3
```

含义是：前 500 次 iteration 从约 `1/3 × base_lr` 线性 warmup；之后以余弦曲线衰减到约 `0.001 × base_lr`。调度由 runner 注册的 learning-rate hook 在训练中更新；backbone 的 `lr_mult=0.1` 始终保留相对倍率。


#### 7.4 一个 epoch 完成后发生什么

每个 epoch 结束，runner 的 hooks 按配置工作：

- `TextLoggerHook`、`TensorboardLoggerHook`：每 50 iteration 记录 loss、学习率、时间等；
- `CheckpointHook`：每个 epoch 写 checkpoint，保存模型权重、optimizer 状态、runner epoch/iter 和元数据；
- 分布式时 `DistSamplerSeedHook`：为下一 epoch 设置 sampler epoch，使训练 index 顺序改变；
- `CustomDistEvalHook`：按 `evaluation.interval=1` 执行验证。

`resume_from` 与 `load_from` 的语义不同：

| 参数 | 恢复内容 | 典型用途 |
|-|-|-|
| `load_from` | 主要加载模型权重 | 从预训练 checkpoint 初始化 |
| `resume_from` | 加载模型、optimizer、runner 的 epoch/iter 等状态 | 中断后继续同一次训练 |

代码优先 `resume_from`；只有它为空时才处理 `load_from`。

---


### 8. 验证怎样插入训练流程

训练入口的 `validate=(not args.no_validate)` 默认开启验证。`custom_train_detector` 额外构建：

```text
val Dataset（test_mode=True）
  → shuffle=False 的 val DataLoader
  → CustomDistEvalHook
```

验证不计算 `loss_cls`、`loss_bbox`，而是：

1. `model.eval()`，关闭梯度；
2. 对验证 batch 调用 `model(return_loss=False, rescale=True, **data)`；
3. 获得当前帧预测框、分数、类别；
4. 多个 rank 收集预测结果；
5. Dataset 按 nuScenes 协议计算 mAP、NDS 和各类 TP error；
6. EvalHook 将指标写入日志/TensorBoard。

验证样本使用单帧 test pipeline；模型在 `forward_test` 内维护 `prev_frame_info`，以跨连续测试调用保存 `prev_bev`。训练时的“一个 Dataset 样本里主动打包 4 帧”与验证时的“模型跨调用缓存历史 BEV”是两种不同的时序组织方式。

---


### 9. 阅读和排查时的最小检查表

如果目标是从全局掌握训练，而非深入网络结构，建议在日志或断点中确认这六件事：

1. 最终配置的 `samples_per_gpu`、`queue_length`、`num_query`、`max_epochs`；
2. `forward_train` 入口的 `img.shape` 是否为 `[B,T,N,C,H,W]`；
3. `len(gt_bboxes_3d)`、`len(gt_labels_3d)` 是否等于 B，且只表示当前帧；
4. `outs['all_cls_scores'].shape` 是否为 `[L,B,Q,C]`，`all_bbox_preds` 是否为 `[L,B,Q,10]`；
5. 日志中是否同时出现 `loss_cls`、`loss_bbox` 以及 `d0`～`d4` 的辅助 loss；
6. checkpoint 和验证指标是否按 epoch 写出，学习率是否经历 warmup 后余弦下降。

至此可以将整个训练任务概括为：每次从一个多帧多相机 batch 中取当前帧预测 900 个候选框；用 Hungarian algorithm 将少数 GT 一对一分配给候选框；对所有候选算 Focal 分类 loss、对匹配候选算加权 L1 框 loss；将 6 个 decoder 层的两类 loss 相加，反向传播并由 AdamW 更新参数；每个 epoch 保存和评测一次。
